<a href="https://colab.research.google.com/github/AntikGhalt/predicting_bankruptcy_AIDA/blob/main/phase1_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Mount Drive (don't work on VSCode)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title System Status Monitor
import psutil, os, torch, time

# --- CONFIGURAZIONE CONSUMI STIMATI (Unità/Ora) ---
rates = {
    "CPU_STANDARD": 0.1,   # Spesso gratuito o ~0.1
    "CPU_HIGH_RAM": 0.46,  # La tua configurazione attuale
    "GPU_T4": 1.5,
    "GPU_L4": 5.0,
    "GPU_A100": 13.0
}

# 1. Identificazione Hardware e Tariffe
gpu_active = torch.cuda.is_available()
if gpu_active:
    gpu_name = torch.cuda.get_device_name(0)
    # Mapping rapido per la tariffa
    rate_key = "GPU_A100" if "A100" in gpu_name else ("GPU_L4" if "L4" in gpu_name else "GPU_T4")
    current_rate = rates.get(rate_key, 2.0)
else:
    gpu_name = "None (CPU Mode)"
    # Controllo se è High RAM (>20GB tot)
    is_high_ram = psutil.virtual_memory().total > 20e9
    current_rate = rates["CPU_HIGH_RAM"] if is_high_ram else rates["CPU_STANDARD"]

# 2. Raccolta Dati Sistema
ram = psutil.virtual_memory()
disk = psutil.disk_usage('/')
uptime_hr = (time.time() - os.path.getmtime('/proc/1')) / 3600

# 3. Output Minimalista
print(f"--- SYSTEM STATUS ---")
print(f"HARDWARE: {gpu_name} {'[HIGH-RAM]' if not gpu_active and is_high_ram else ''}")
print(f"CONSUMO EST.: ~{current_rate} unità/ora")
print(f"RAM:  {ram.used/1e9:.1f}GB / {ram.total/1e9:.1f}GB ({ram.percent}%)")
print(f"DISK: {disk.used/1e9:.1f}GB / {disk.total/1e9:.1f}GB ({disk.percent}%)")
print(f"UPTIME: {uptime_hr:.2f} ore")
print(f"---------------------")

--- SYSTEM STATUS ---
HARDWARE: None (CPU Mode) [HIGH-RAM]
CONSUMO EST.: ~0.46 unità/ora
RAM:  1.7GB / 54.8GB (4.2%)
DISK: 22.7GB / 242.5GB (9.4%)
UPTIME: 0.38 ore
---------------------


In [ ]:
#@title Unmount Drive
from google.colab import runtime
runtime.unassign()

### COLAB DRIVE

---



In [3]:
#@title RESET AND FILES NAME
%reset -f
#path data export


# list your files in the exact order you want their rows to appear
file_names = [
    "Aida_ALL_CODICI_000001_050000.xlsx",
    "Aida_ALL_CODICI_050001_100000.xlsx",
    "Aida_ALL_CODICI_100001_150000.xlsx",
    "Aida_ALL_CODICI_150001_190607.xlsx"
]
file_CCIAA = [
    "Aida_ALL_CCIAA_000001_050000.xlsx",
    "Aida_ALL_CCIAA_050001_100000.xlsx",
    "Aida_ALL_CCIAA_100001_150000.xlsx",
    "Aida_ALL_CCIAA_150001_190607.xlsx"
]

In [4]:
model_folder_input_dd = "model_ALL_dd/"
model_folder_input_base= "model_ALL/"
model_folder_input_finance= "Data_finance/"
model_folder_output = "output/model_ALL/"

In [5]:
path_base = "/content/drive/MyDrive/Machine Learning/AIDA/DATA/"
path_data = path_base + model_folder_input_base + model_folder_input_dd
path_data2 = path_base + model_folder_input_base + model_folder_input_finance
path_MODEL = path_base + model_folder_output

In [6]:
# --- Installation ---
# Installs all required libraries
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm tensorflow optuna plotly
!pip uninstall -y kaleido
!pip install kaleido==0.2.1
!pip install xgboost[gpu]
!pip install contextily geopandas -q
# --- Standard Library ---
import os
import re
import gc
import time
import json
import pickle
import textwrap
import warnings
from datetime import datetime
import sys
import glob
import pickle
import shutil
import threading
import queue
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Data Handling ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
# --- Scientific Computing / Stats ---
from scipy.interpolate import make_interp_spline

# --- Scikit-Learn ---
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score
)

# --- Gradient Boosting ---
import xgboost as xgb
import lightgbm as lgb

# --- Deep Learning ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- Hyperparameter Optimization ---
import optuna
from optuna.pruners import MedianPruner
from optuna.visualization import plot_optimization_history
import optuna.visualization as optuna_vis

# --- Jupyter / Colab Specific ---
from IPython.display import clear_output, display
from google.colab import data_table
from google.colab import drive

# --- ENVIRONMENT CONFIGURATION ---
# Suppress all warnings for cleaner output
warnings.filterwarnings('ignore')
# Set Optuna logging verbosity to show only warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
"""
def safe_literal_eval(x):
    # If x is a list or numpy array and empty, return an empty list
    if isinstance(x, (list, np.ndarray)) and len(x) == 0:
        return []
    # For scalar values, check if x is NaN
    if not isinstance(x, (list, np.ndarray)) and pd.isna(x):
        return []

    # Otherwise, convert to string and strip any whitespace
    x = str(x).strip()

    try:
        return ast.literal_eval(x)
    except Exception:
        return x
"""
clear_output()

# DATA MANILUPATION

##FIRST MANIPULATION

In [8]:
#@title PRE-IMPORT SANITY CHECK (schema + formats)

import os
import pandas as pd
import numpy as np
from datetime import datetime

SHEET_NAME = "Results"
NROWS_SAMPLE = 3000  # increase if you want a more robust check

# Columns we care about in the main dataset
COLUMNS_OF_INTEREST = [
    "Company name",
    "Tax code number",
    "Registered office address - Postal code",
    "Registered office address - Longitude",
    "Registered office address - Latitude",
    "Trading address - Postal code",
    "Legal status",
    "Legal form",
    "Company category",
    "Incorporation year",
    "No of available years",
    "Last accounting closing date",
    "Procedure/cessazione",
    "Date of open procedure/cessazione",
    "Date of closure procedure",
    "Unnamed: 0",
]

def _safe_read_excel(path, usecols=None, nrows=1000):
    # Some exports are messy; keep it resilient
    return pd.read_excel(
        path,
        sheet_name=SHEET_NAME,
        usecols=usecols,
        nrows=nrows,
        engine="openpyxl"
    )

def _coerce_str(series):
    # robust string conversion; preserves NaN as NaN
    s = series.copy()
    # keep NaNs
    s = s.astype("string")
    # strip whitespace
    return s.str.strip()

def _clean_tax_code_like(series):
    """
    Makes tax code analysis robust even if Excel imported numbers as floats.
    - Strips spaces
    - Removes trailing '.0'
    """
    s = _coerce_str(series)
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\s+", "", regex=True)
    return s

def _date_parse_rate(series):
    # returns % parseable dates (ignoring missing)
    s = series.copy()
    s = s[~pd.isna(s)]
    if len(s) == 0:
        return np.nan
    parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
    return float(parsed.notna().mean())

def _numeric_rate(series):
    s = series.copy()
    s = s[~pd.isna(s)]
    if len(s) == 0:
        return np.nan
    coerced = pd.to_numeric(s, errors="coerce")
    return float(coerced.notna().mean())

schema_rows = []
quality_rows = []
all_columns_sets = {}

for file_name in file_names:
    file_path = os.path.join(path_data, file_name)
    print(f"\n--- Checking file: {file_name} ---")
    if not os.path.exists(file_path):
        print(f"!! Missing file at: {file_path}")
        continue

    # 1) Read just header to get columns quickly
    try:
        df_head = _safe_read_excel(file_path, usecols=None, nrows=5)
    except Exception as e:
        print(f"!! Could not read {file_name}: {e}")
        continue

    cols = list(df_head.columns)
    all_columns_sets[file_name] = set(cols)

    # Column presence check
    missing = [c for c in COLUMNS_OF_INTEREST if c not in cols]
    present = [c for c in COLUMNS_OF_INTEREST if c in cols]

    print(f"Columns found: {len(cols)}")
    if missing:
        print("Missing columns of interest:")
        for c in missing:
            print(f"  - {c}")
    else:
        print("All columns of interest are present ✓")

    # 2) Read sample for quality checks (only columns present)
    try:
        df_s = _safe_read_excel(file_path, usecols=present, nrows=NROWS_SAMPLE)
    except Exception as e:
        print(f"!! Could not read sample rows for {file_name}: {e}")
        continue

    # 3) Schema report (dtype + missing)
    for c in present:
        schema_rows.append({
            "file": file_name,
            "column": c,
            "dtype_inferred": str(df_s[c].dtype),
            "missing_rate": float(df_s[c].isna().mean()),
            "example_1": df_s[c].dropna().iloc[0] if df_s[c].notna().any() else None,
        })

    # 4) Focused quality checks
    # Tax code number
    if "Tax code number" in df_s.columns:
        tc = _clean_tax_code_like(df_s["Tax code number"])
        tc_nonmiss = tc[tc.notna() & (tc != "") & (tc != "<NA>") & (tc.str.lower() != "nan")]

        lengths = tc_nonmiss.str.len()
        quality_rows.append({
            "file": file_name,
            "check": "Tax code number",
            "non_missing": int(tc_nonmiss.shape[0]),
            "missing_rate": float(df_s["Tax code number"].isna().mean()),
            "unique_rate_nonmissing": float(tc_nonmiss.nunique() / max(1, tc_nonmiss.shape[0])),
            "len_min": int(lengths.min()) if len(lengths) else None,
            "len_max": int(lengths.max()) if len(lengths) else None,
            "len_mode": int(lengths.mode().iloc[0]) if len(lengths) and not lengths.mode().empty else None,
            "share_len_11": float((lengths == 11).mean()) if len(lengths) else None,
            "share_len_16": float((lengths == 16).mean()) if len(lengths) else None,
            "share_has_dot": float(tc_nonmiss.str.contains(r"\.").mean()) if tc_nonmiss.shape[0] else None,
            "share_non_alnum": float((~tc_nonmiss.str.match(r"^[A-Za-z0-9]+$")).mean()) if tc_nonmiss.shape[0] else None,
            "example_clean": tc_nonmiss.iloc[0] if tc_nonmiss.shape[0] else None,
        })

    # No of available years
    if "No of available years" in df_s.columns:
        y = df_s["No of available years"]
        y_num = pd.to_numeric(y, errors="coerce")
        quality_rows.append({
            "file": file_name,
            "check": "No of available years",
            "numeric_parse_rate": float(y_num.notna().mean()),
            "min": float(y_num.min()) if y_num.notna().any() else None,
            "max": float(y_num.max()) if y_num.notna().any() else None,
            "share_outside_0_15": float(((y_num < 0) | (y_num > 15)).mean()) if y_num.notna().any() else None,
        })

    # Incorporation year
    if "Incorporation year" in df_s.columns:
        inc = pd.to_numeric(df_s["Incorporation year"], errors="coerce")
        quality_rows.append({
            "file": file_name,
            "check": "Incorporation year",
            "numeric_parse_rate": float(inc.notna().mean()),
            "min": float(inc.min()) if inc.notna().any() else None,
            "max": float(inc.max()) if inc.notna().any() else None,
            "share_outside_1800_2026": float(((inc < 1800) | (inc > 2026)).mean()) if inc.notna().any() else None,
        })

    # Dates parseability
    for date_col in [
        "Last accounting closing date",
        "Date of open procedure/cessazione",
        "Date of closure procedure"
    ]:
        if date_col in df_s.columns:
            quality_rows.append({
                "file": file_name,
                "check": f"Date parseability: {date_col}",
                "missing_rate": float(df_s[date_col].isna().mean()),
                "parseable_rate": _date_parse_rate(df_s[date_col]),
                "example": df_s[date_col].dropna().iloc[0] if df_s[date_col].notna().any() else None,
            })

    # Coordinates sanity (Italy-ish bounds)
    if "Registered office address - Longitude" in df_s.columns:
        lon = pd.to_numeric(df_s["Registered office address - Longitude"], errors="coerce")
        quality_rows.append({
            "file": file_name,
            "check": "Longitude (Italy range approx 5..19)",
            "numeric_parse_rate": float(lon.notna().mean()),
            "share_outside_5_19": float(((lon < 5) | (lon > 19)).mean()) if lon.notna().any() else None,
        })
    if "Registered office address - Latitude" in df_s.columns:
        lat = pd.to_numeric(df_s["Registered office address - Latitude"], errors="coerce")
        quality_rows.append({
            "file": file_name,
            "check": "Latitude (Italy range approx 35..48)",
            "numeric_parse_rate": float(lat.notna().mean()),
            "share_outside_35_48": float(((lat < 35) | (lat > 48)).mean()) if lat.notna().any() else None,
        })

# Compare column sets across files (schema consistency)
print("\n=== SCHEMA CONSISTENCY ACROSS FILES ===")
base_file = None
base_cols = None
for fn, colset in all_columns_sets.items():
    if base_cols is None:
        base_file = fn
        base_cols = colset
        continue
    if colset != base_cols:
        print(f"!! Columns differ between {base_file} and {fn}")
        only_in_base = sorted(list(base_cols - colset))
        only_in_this = sorted(list(colset - base_cols))
        if only_in_base:
            print(f"   - Only in {base_file}: {only_in_base[:10]}{' ...' if len(only_in_base)>10 else ''}")
        if only_in_this:
            print(f"   - Only in {fn}: {only_in_this[:10]}{' ...' if len(only_in_this)>10 else ''}")
    else:
        print(f"✓ {fn} matches {base_file}")

df_schema_report = pd.DataFrame(schema_rows).sort_values(["column", "file"])
df_quality_report = pd.DataFrame(quality_rows)

print("\n=== QUICK VIEW: SCHEMA REPORT (first 20 rows) ===")
display(df_schema_report.head(20))

print("\n=== QUICK VIEW: QUALITY REPORT ===")
display(df_quality_report)



--- Checking file: Aida_ALL_CODICI_000001_050000.xlsx ---
Columns found: 16
All columns of interest are present ✓

--- Checking file: Aida_ALL_CODICI_050001_100000.xlsx ---
Columns found: 16
All columns of interest are present ✓

--- Checking file: Aida_ALL_CODICI_100001_150000.xlsx ---
Columns found: 16
All columns of interest are present ✓

--- Checking file: Aida_ALL_CODICI_150001_190607.xlsx ---
Columns found: 16
All columns of interest are present ✓

=== SCHEMA CONSISTENCY ACROSS FILES ===
✓ Aida_ALL_CODICI_050001_100000.xlsx matches Aida_ALL_CODICI_000001_050000.xlsx
✓ Aida_ALL_CODICI_100001_150000.xlsx matches Aida_ALL_CODICI_000001_050000.xlsx
✓ Aida_ALL_CODICI_150001_190607.xlsx matches Aida_ALL_CODICI_000001_050000.xlsx

=== QUICK VIEW: SCHEMA REPORT (first 20 rows) ===


,file,column,dtype_inferred,missing_rate,example_1
8,Aida_ALL_CODICI_000001_050000.xlsx,Company category,object,0.034000,Limited company
24,Aida_ALL_CODICI_050001_100000.xlsx,Company category,object,0.049000,Limited company
40,Aida_ALL_CODICI_100001_150000.xlsx,Company category,object,0.064333,Limited company
56,Aida_ALL_CODICI_150001_190607.xlsx,Company category,object,0.145667,Limited company
0,Aida_ALL_CODICI_000001_050000.xlsx,Company name,object,0.034000,ENI S.P.A.
16,Aida_ALL_CODICI_050001_100000.xlsx,Company name,object,0.049000,CORNELIO CAPPELLINI S.R.L.
32,Aida_ALL_CODICI_100001_150000.xlsx,Company name,object,0.064333,CALLEGARI FRANCESCO S.R.L.
48,Aida_ALL_CODICI_150001_190607.xlsx,Company name,object,0.145667,PRISMA S.R.L.S. IN LIQUIDAZIONE
14,Aida_ALL_CODICI_000001_050000.xlsx,Date of closure procedure,object,0.986000,13/12/2017
30,Aida_ALL_CODICI_050001_100000.xlsx,Date of closure procedure,object,0.990667,25/02/2004



=== QUICK VIEW: QUALITY REPORT ===


,file,check,non_missing,missing_rate,unique_rate_nonmissing,len_min,len_max,len_mode,share_len_11,share_len_16,...,example_clean,numeric_parse_rate,min,max,share_outside_0_15,share_outside_1800_2026,parseable_rate,example,share_outside_5_19,share_outside_35_48
0,Aida_ALL_CODICI_000001_050000.xlsx,Tax code number,2898.0,0.034000,1.0,8.0,11.0,10.0,0.116287,0.0,...,484960588,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aida_ALL_CODICI_000001_050000.xlsx,No of available years,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.966000,1.0,10.0,0.0,NaN,NaN,NaN,NaN,NaN
2,Aida_ALL_CODICI_000001_050000.xlsx,Incorporation year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.957000,37.0,45567.0,NaN,0.957000,NaN,NaN,NaN,NaN
3,Aida_ALL_CODICI_000001_050000.xlsx,Date parseability: Last accounting closing date,NaN,0.034000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,45291.0,NaN,NaN
4,Aida_ALL_CODICI_000001_050000.xlsx,Date parseability: Date of open procedure/cess...,NaN,0.830333,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,25/05/2020,NaN,NaN
5,Aida_ALL_CODICI_000001_050000.xlsx,Date parseability: Date of closure procedure,NaN,0.986000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,13/12/2017,NaN,NaN
6,Aida_ALL_CODICI_000001_050000.xlsx,Longitude (Italy range approx 5..19),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.944333,NaN,NaN,NaN,NaN,NaN,NaN,0.000333,NaN
7,Aida_ALL_CODICI_000001_050000.xlsx,Latitude (Italy range approx 35..48),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.944333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000333
8,Aida_ALL_CODICI_050001_100000.xlsx,Tax code number,2853.0,0.049000,1.0,8.0,11.0,10.0,0.086225,0.0,...,2343160137,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Aida_ALL_CODICI_050001_100000.xlsx,No of available years,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.951000,1.0,10.0,0.0,NaN,NaN,NaN,NaN,NaN


### continue


In [7]:
#@title Main Dataset Import and Processing

# Define the PKL file path for the main dataset
df_main_pkl = os.path.join(path_data, 'df_main_processed.pkl')

print("MAIN DATASET IMPORT")


# Check if processed dataset already exists
if os.path.exists(df_main_pkl):
    print("Found existing processed dataset")
    try:
        print("Loading df_main from PKL file")
        with open(df_main_pkl, 'rb') as f:
            df_main = pickle.load(f)

        print(f"Successfully loaded df_main with shape: {df_main.shape}")
        print(f"   - Observations: {df_main.shape[0]:,}")
        print(f"   - Variables: {df_main.shape[1]}")
        print(f"   - Unique tax codes: {df_main['Tax code number'].nunique():,}")

        print("\nSKIPPING EXCEL IMPORT - Using existing processed data")

    except Exception as e:
        print(f"Error loading PKL file: {e}")
        print("New Excel import")
        df_main_exists = False
    else:
        df_main_exists = True

else:
    print("No existing processed dataset found")
    print("New Excel import")
    df_main_exists = False

# Only import Excel files if we don't have the processed dataset
if not df_main_exists:
    print("IMPORTING EXCEL FILES")

    # Import and concatenate all Excel files
    print("Starting data import")
    dataframes = []

    for file_name in file_names:
        file_path = os.path.join(path_data, file_name)
        print(f"Loading {file_name}...")
        df_temp = pd.read_excel(file_path, sheet_name='Results')
        dataframes.append(df_temp)
        print(f"  - Shape: {df_temp.shape}")

    # Concatenate all dataframes
    df_main = pd.concat(dataframes, ignore_index=True)
    print(f"\nCombined dataset shape: {df_main.shape}")

    # Display column names to check exact naming
    print("\nColumn names in the dataset:")
    for i, col in enumerate(df_main.columns):
        print(f"{i+1}. '{col}'")

    # Clean data - remove observations with empty Tax code number
    print(f"\nBefore removing empty Tax code numbers: {df_main.shape[0]} observations")
    df_main = df_main.dropna(subset=['Tax code number'])
    print(f"After removing empty Tax code numbers: {df_main.shape[0]} observations")

    # Drop specified variables
    columns_to_drop = ['Procedure/cessazione', 'Date of open procedure/cessazione', 'Date of closure procedure']
    existing_columns_to_drop = [col for col in columns_to_drop if col in df_main.columns]

    if existing_columns_to_drop:
        df_main = df_main.drop(columns=existing_columns_to_drop)
        print(f"\nDropped columns: {existing_columns_to_drop}")
    else:
        print(f"\nWarning: None of the specified columns to drop were found in the dataset")
        print("Available columns that might be similar:")
        for col in df_main.columns:
            if any(keyword in col.lower() for keyword in ['procedure', 'cessazione', 'closure', 'open']):
                print(f"  - {col}")

    print(f"Final dataset shape after cleaning: {df_main.shape}")

    # Save the processed dataset
    print(f"\nSaving processed dataset to PKL")
    with open(df_main_pkl, 'wb') as f:
        pickle.dump(df_main, f)
    print(f"Saved to: {df_main_pkl}")

# DESCRIPTIVE STATISTICS (always run)
print("DESCRIPTIVE STATISTICS")

# 1. Shape of database
print(f"\n1. DATABASE SHAPE:")
print(f"   - Number of observations: {df_main.shape[0]:,}")
print(f"   - Number of variables: {df_main.shape[1]}")

# 2. Check if all tax codes are unique
print(f"\n2. TAX CODE UNIQUENESS:")
unique_tax_codes = df_main['Tax code number'].nunique()
total_tax_codes = len(df_main)
duplicates = total_tax_codes - unique_tax_codes

print(f"   - Total observations: {total_tax_codes:,}")
print(f"   - Unique tax codes: {unique_tax_codes:,}")
print(f"   - Duplicate tax codes: {duplicates:,}")

if duplicates > 0:
    print(f"   - Percentage of duplicates: {(duplicates/total_tax_codes)*100:.2f}%")

    # Show some examples of duplicated tax codes
    duplicated_mask = df_main.duplicated(subset=['Tax code number'], keep=False)
    duplicated_codes = df_main[duplicated_mask]['Tax code number'].value_counts().head()
    print(f"   - Top duplicated tax codes:")
    for code, count in duplicated_codes.items():
        print(f"     * {code}: {count} occurrences")
else:
    print("   - All tax codes are unique")

# 3. Analysis of "No of available years"
print(f"\n3. NO OF AVAILABLE YEARS ANALYSIS:")

if 'No of available years' in df_main.columns:
    years_col = df_main['No of available years']

    # Remove missing values for analysis
    years_clean = years_col.dropna()

    print(f"   - Total observations: {len(years_col):,}")
    print(f"   - Non-missing values: {len(years_clean):,}")
    print(f"   - Missing values: {years_col.isna().sum():,}")

    if len(years_clean) > 0:
        print(f"\n   CENTRAL TENDENCY MEASURES:")
        print(f"   - Mean: {years_clean.mean():.2f}")
        print(f"   - Median: {years_clean.median():.2f}")

        print(f"\n   DISTRIBUTION MEASURES:")
        print(f"   - Standard deviation: {years_clean.std():.2f}")
        print(f"   - Minimum: {years_clean.min()}")
        print(f"   - Maximum: {years_clean.max()}")
        print(f"   - 25th percentile (Q1): {years_clean.quantile(0.25):.2f}")
        print(f"   - 75th percentile (Q3): {years_clean.quantile(0.75):.2f}")
        print(f"   - Interquartile Range (IQR): {years_clean.quantile(0.75) - years_clean.quantile(0.25):.2f}")

        print(f"\n   VALUE COUNTS:")
        value_counts = years_clean.value_counts().sort_index()
        for value, count in value_counts.items():
            percentage = (count / len(years_clean)) * 100
            print(f"   - {value} years: {count:,} observations ({percentage:.1f}%)")

    else:
        print("   - No valid data available for analysis")

else:
    print("   - Column 'No of available years' not found in dataset")
    print("   - Available columns that might be related:")
    for col in df_main.columns:
        if 'year' in col.lower() or 'anni' in col.lower():
            print(f"     * {col}")

print("DATA PROCESSING COMPLETED")

# Clear temporary variables
if 'dataframes' in locals():
    del dataframes
if 'df_temp' in locals():
    del df_temp
gc.collect()

MAIN DATASET IMPORT
No existing processed dataset found
New Excel import
IMPORTING EXCEL FILES
Starting data import
Loading Aida_ALL_CODICI_000001_050000.xlsx...
  - Shape: (51851, 16)
Loading Aida_ALL_CODICI_050001_100000.xlsx...
  - Shape: (52956, 16)
Loading Aida_ALL_CODICI_100001_150000.xlsx...
  - Shape: (55449, 16)
Loading Aida_ALL_CODICI_150001_190607.xlsx...
  - Shape: (55246, 16)

Combined dataset shape: (215502, 16)

Column names in the dataset:
1. 'Unnamed: 0'
2. 'Company name'
3. 'Tax code number'
4. 'Registered office address - Postal code'
5. 'Registered office address - Longitude'
6. 'Registered office address - Latitude'
7. 'Trading address - Postal code'
8. 'Legal status'
9. 'Legal form'
10. 'Company category'
11. 'Incorporation year'
12. 'No of available years'
13. 'Last accounting closing date'
14. 'Procedure/cessazione'
15. 'Date of open procedure/cessazione'
16. 'Date of closure procedure'

Before removing empty Tax code numbers: 215502 observations
After removing 

238

In [ ]:
#@title IMPORT CHECKED

# File paths
df_finance_pkl = os.path.join(path_data, 'df_finance.pkl')
imported_files_txt = os.path.join(path_data, 'imported_files_list.txt')

print("OPTIMIZED PIPELINE IMPORT")

# Step 1: Load existing progress and dataframe
xlsx_files = sorted(glob.glob(os.path.join(path_data2, "*.xlsx")))
print(f"Found {len(xlsx_files)} Excel files")

# Initialize variables
imported_files = set()
df_finance = pd.DataFrame()

# Load existing import tracking
if os.path.exists(imported_files_txt):
    print("Loading previously imported files list")
    with open(imported_files_txt, 'r') as f:
        imported_files = set(line.strip() for line in f.readlines())
    print(f"  - Found record of {len(imported_files)} previously imported files")
else:
    print("No previous import record found")

# Load existing dataframe
if os.path.exists(df_finance_pkl):
    print("Loading existing df_finance")
    try:
        with open(df_finance_pkl, 'rb') as f:
            df_finance = pickle.load(f)
        print(f"  - Successfully loaded df_finance with shape: {df_finance.shape}")
        print(f"  - Unique tax codes: {df_finance['Tax code number'].nunique():,}")
        print(f"  - Columns: {len(df_finance.columns)}")
    except Exception as e:
        print(f"  - Error loading df_finance: {e}")
        print("  - Starting with empty dataframe")
        df_finance = pd.DataFrame()
        imported_files = set()  # Reset if pkl is corrupted
else:
    print("No existing df_finance found - starting fresh")

# Determine what needs to be imported
files_to_import = [f for f in xlsx_files if os.path.basename(f) not in imported_files]

print(f"\nIMPORT STATUS CHECK:")
print(f"  - Total files available: {len(xlsx_files)}")
print(f"  - Files already imported: {len(imported_files)}")
print(f"  - Files remaining to import: {len(files_to_import)}")

# Validation check
if len(imported_files) > 0 and df_finance.empty:
    print("  - WARNING: Import list exists but df_finance is empty!")
    print("  - This might indicate corrupted files. Starting fresh")
    imported_files = set()
    files_to_import = xlsx_files

if len(files_to_import) == 0:
    print("\nALL FILES ALREADY IMPORTED!")
    print(f"  - df_finance shape: {df_finance.shape}")
    print(f"  - No new files to process")
    print("  - You can proceed to the next step")
elif len(imported_files) == 0:
    print(f"\nSTARTING FRESH IMPORT of {len(files_to_import)} files")
else:
    print(f"\nRESUMING IMPORT - {len(files_to_import)} files remaining")
    print(f"  - Will append new data to existing df_finance")
    print(f"  - Current df_finance will be preserved")

# Only proceed with import if there are files to process
if len(files_to_import) > 0:
    print(f"STARTING OPTIMIZED PIPELINE IMPORT WITH {len(files_to_import)} files")

    # Pipeline configuration
    max_read_threads = 4  # Number of parallel read operations

    # Queues for pipeline
    read_queue = queue.Queue()
    result_queue = queue.Queue()

    # Thread-safe counters
    stats_lock = threading.Lock()
    stats = {
        'read': 0,
        'errors': 0,
        'start_time': datetime.now()
    }

    def read_worker():
        """Worker thread that reads Excel files directly from drive and processes them"""
        while True:
            try:
                item = read_queue.get(timeout=2)
                if item is None:  # Shutdown signal
                    break

                file_path, file_name = item

                try:
                    read_start = datetime.now()
                    df_temp = pd.read_excel(
                        file_path,
                        sheet_name=1,  # Results sheet
                        nrows=200
                    )
                    read_time = (datetime.now() - read_start).total_seconds()

                    # Validate data
                    if df_temp.empty or 'Tax code number' not in df_temp.columns:
                        print(f"    Invalid data in {file_name}")
                        with stats_lock:
                            stats['errors'] += 1
                        continue

                    # Add tracking
                    df_temp['_source_file'] = file_name
                    df_temp['_import_order'] = len(imported_files) + stats['read']

                    # Add to result queue
                    result_queue.put((df_temp, file_name, read_time))

                    with stats_lock:
                        stats['read'] += 1
                        if stats['read'] % 5 == 0:
                            elapsed = datetime.now() - stats['start_time']
                            rate = stats['read'] / elapsed.total_seconds() * 60 if elapsed.total_seconds() > 0 else 0
                            remaining = len(files_to_import) - stats['read']
                            eta = remaining / rate if rate > 0 else 0
                            print(f"  📖 Read {stats['read']}/{len(files_to_import)} files ({rate:.1f}/min, ETA: {eta:.1f}min)")

                except Exception as e:
                    with stats_lock:
                        stats['errors'] += 1
                    print(f"  Read error {file_name}: {e}")

                read_queue.task_done()

            except queue.Empty:
                continue

    # Start worker threads
    read_threads = []
    for i in range(max_read_threads):
        t = threading.Thread(target=read_worker, daemon=True)
        t.start()
        read_threads.append(t)

    # Add all files to read queue
    for file_path in files_to_import:
        file_name = os.path.basename(file_path)
        read_queue.put((file_path, file_name))

    print(f"  Pipeline started: {max_read_threads} read workers")
    print(f"  Processing {len(files_to_import)} files...")

    # Collect results and save periodically
    batch_data = []
    successful_imports = 0

    while successful_imports < len(files_to_import) - stats['errors']:
        try:
            # Get result with timeout
            result = result_queue.get(timeout=30)
            if result is None:
                break

            df_temp, file_name, read_time = result
            batch_data.append(df_temp)
            imported_files.add(file_name)
            successful_imports += 1

            print(f"    {file_name}: Read={read_time:.1f}s, Shape={df_temp.shape}")

            # Clear df_temp immediately to save memory
            del df_temp

            # Save progress every 20 files
            if len(batch_data) >= 20:
                print(f"  Saving batch of {len(batch_data)} files")
                df_batch = pd.concat(batch_data, ignore_index=True)

                if df_finance.empty:
                    df_finance = df_batch
                else:
                    df_finance = pd.concat([df_finance, df_batch], ignore_index=True)

                # Clear batch data immediately
                del batch_data, df_batch
                batch_data = []

                # Save to disk
                with open(df_finance_pkl, 'wb') as f:
                    pickle.dump(df_finance, f)

                with open(imported_files_txt, 'w') as f:
                    for fname in sorted(imported_files):
                        f.write(fname + '\n')

                print(f"    Saved! Current shape: {df_finance.shape}")

            result_queue.task_done()

        except queue.Empty:
            # Check if we're done or if there's an issue
            if read_queue.empty() and result_queue.empty():
                print("  ⏳ Waiting for remaining files to be processed")
            continue
        except Exception as e:
            print(f"  Result processing error: {e}")
            break

    # Process final batch
    if batch_data:
        print(f"  Saving final batch of {len(batch_data)} files")
        df_batch = pd.concat(batch_data, ignore_index=True)

        # APPEND to existing df_finance (preserve old data)
        if df_finance.empty:
            df_finance = df_batch
            print(f"    - Created final df_finance with shape: {df_finance.shape}")
        else:
            original_shape = df_finance.shape
            df_finance = pd.concat([df_finance, df_batch], ignore_index=True)
            print(f"    - Final append: {original_shape} → {df_finance.shape}")

        # Clear batch data
        del batch_data, df_batch

        with open(df_finance_pkl, 'wb') as f:
            pickle.dump(df_finance, f)

        with open(imported_files_txt, 'w') as f:
            for fname in sorted(imported_files):
                f.write(fname + '\n')

        print(f"    Final save completed!")

    # Proper thread cleanup
    print("Cleaning up threads")
    for _ in range(max_read_threads):
        read_queue.put(None)

    for t in read_threads:
        t.join(timeout=5)
        if t.is_alive():
            print("Read thread didn't terminate cleanly")

    # Clear queue objects
    try:
        while not read_queue.empty():
            read_queue.get_nowait()
    except:
        pass

    try:
        while not result_queue.empty():
            result_queue.get_nowait()
    except:
        pass

    # Memory cleanup
    cleanup_vars = ['read_queue', 'result_queue', 'read_threads', 'stats', 'files_to_import', 'xlsx_files']
    for var in cleanup_vars:
        try:
            if var in locals():
                del locals()[var]
        except:
            pass

    import gc
    gc.collect()

    total_time = datetime.now() - stats['start_time'] if 'stats' in locals() else datetime.now()
    print(f"\nPIPELINE IMPORT COMPLETED!")
    print(f"  - Files successfully imported: {successful_imports}")
    print(f"  - Errors: {stats['errors'] if 'stats' in locals() else 0}")
    print(f"  - Total time: {total_time}")
    if total_time.total_seconds() > 0:
        print(f"  - Average rate: {successful_imports / total_time.total_seconds() * 60:.1f} files/min")

else:
    print(f"\nSKIPPING IMPORT - All files already processed!")
    print(f"  - Existing df_finance shape: {df_finance.shape}")
    print(f"  - Ready to proceed to next step")

# Final status and memory cleanup
print(f"\nFINAL STATUS:")
print(f"  - df_finance shape: {df_finance.shape}")
print(f"  - Total files imported: {len(imported_files)}")
if not df_finance.empty:
    print(f"  - Unique tax codes: {df_finance['Tax code number'].nunique():,}")
print(f"  - Saved to: {df_finance_pkl}")

# Final cleanup
try:
    del imported_files
    import gc
    gc.collect()
    print("Final memory cleanup completed")
except:
    pass

print("OPTIMIZED PIPELINE COMPLETED")

### Clearing: dates, nan, months NO EVENTS (in the end with CCIAA)

In [ ]:
#@title COMPLETE PIPELINE: CLEANING, MERGING, AND REMOVING >95% MISSING
import pandas as pd
import numpy as np
import re
import os
import gc
import pickle
from datetime import datetime

# ============================================================================
# CLEANING FUNCTIONS (from our earlier discussion)
# ============================================================================

def fix_date_columns(df, verbose=True):
    """
    Fix date columns that might be in Excel serial format or string format
    """
    date_pattern = re.compile(r'(year|date|anno|data)$', re.IGNORECASE)
    date_cols_fixed = 0

    for col in df.columns:
        if not date_pattern.search(str(col)): # Ensure column name is a string
            continue

        if verbose:
            print(f"\nProcessing date column: {col}")

        raw = df[col].copy()
        ser = pd.to_numeric(raw, errors="coerce")
        dt = pd.Series(pd.NaT, index=ser.index, dtype='datetime64[ns]')

        # Convert Excel serial dates (within safe range)
        safe_mask = ser.between(-10_000, 100_000) & ser.notna()
        if safe_mask.any():
            dt.loc[safe_mask] = pd.to_datetime(
                ser.loc[safe_mask].astype("int64"),
                unit="D",
                origin="1899-12-30",
                errors="coerce"
            )
            if verbose:
                print(f"  - Converted {safe_mask.sum()} Excel serial dates")

        # Try string parsing for remaining values
        str_mask = dt.isna() & raw.notna()
        if str_mask.any():
            dt.loc[str_mask] = pd.to_datetime(
                raw[str_mask],
                dayfirst=True,
                errors="coerce"
            )
            # Correctly calculate how many strings were converted
            converted_strings = (dt.notna().sum() - safe_mask.sum())
            if verbose and converted_strings > 0:
                print(f"  - Converted {converted_strings} string dates")

        df[col] = dt

        if verbose:
            valid_dates = dt.notna().sum()
            original_non_null = raw.notna().sum()
            if original_non_null > 0:
                print(f"  - Result: {valid_dates}/{original_non_null} valid dates ({valid_dates/original_non_null*100:.1f}% success rate)")

        date_cols_fixed += 1

    if verbose:
        print(f"\nTotal date columns fixed: {date_cols_fixed}")

    return df


def fix_year_columns(df, verbose=True):
    """
    Fix columns that contain only years (4-digit numbers)
    """
    year_cols_fixed = 0
    for col in df.columns:
        # *** ADD THIS LINE TO THE FUNCTION ***
        # If the column is already a date, skip it.
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        if 'year' in str(col).lower() or 'anno' in str(col).lower():
            if verbose:
                print(f"\nProcessing year column: {col}")

            years = pd.to_numeric(df[col], errors='coerce')
            valid_mask = years.between(1900, 2100)
            years_clean = years.where(valid_mask)
            df[col] = years_clean

            if verbose:
                valid_count = valid_mask.sum()
                original_count = years.notna().sum()
                if original_count > 0:
                    print(f"  - Valid years: {valid_count}/{original_count}")

            year_cols_fixed += 1

    if verbose:
        print(f"\nTotal year columns fixed: {year_cols_fixed}")

    return df


def standardize_missing_values(df, na_values=None, verbose=True):
    """
    Standardize missing values across the dataframe
    """
    if na_values is None:
        na_values = ["n.a.", "n.d.", "nd", "na", "N.A.", "N.D.", "ND", "NA", "#N/A", "-", "--", "..."]

    if verbose:
        print("\nStandardizing missing values...")
        print(f"Values to treat as NaN: {na_values}")

    total_replacements = 0

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Check if column has object type to apply string replacement
        if df[col].dtype == 'object':
            before_nan = df[col].isna().sum()
            df[col] = df[col].replace(na_values, np.nan)
            after_nan = df[col].isna().sum()

            replacements = after_nan - before_nan
            if replacements > 0:
                total_replacements += replacements
                # Only print for very large numbers of replacements to avoid clutter
                if verbose and replacements > 200000:
                    print(f"  - {col}: {replacements} values replaced")

    if verbose:
        print(f"\nTotal values replaced with NaN: {total_replacements:,}")

    return df


def add_group_prefixes(df, exclude_cols=None, verbose=True):
    """
    Add group prefixes to columns with years to avoid naming conflicts
    """
    if exclude_cols is None:
        exclude_cols = ['Tax code number', 'Company name']

    print("\n" + "="*60)
    print("ADDING GROUP PREFIXES TO COLUMNS")
    print("="*60)

    year_pattern = re.compile(r'(20\d{2})(?:\.\d+)?$', re.MULTILINE)

    def get_base_name(col_name):
        """Extract the base variable name without the year and suffix"""
        col_name_str = str(col_name) # Ensure it's a string
        suffix_pattern = re.compile(r'\.\d+$')
        col_name_cleaned = suffix_pattern.sub('', col_name_str)

        year_match = year_pattern.search(col_name_cleaned)
        if year_match:
            # This was the line with the AttributeError
            base = year_pattern.sub('', col_name_cleaned).strip()
            return base
        return None

    columns = list(df.columns)
    new_columns = []
    group_counter = 1
    seen_groups = {}
    current_base = None

    # Corrected logic for grouping
    for col in columns:
        if col in exclude_cols:
            new_columns.append(col)
            continue

        base_name = get_base_name(col)

        if base_name is None:
            new_columns.append(col)
        else:
            # This logic correctly handles finding the right group number
            # even if variables are not perfectly sequential in the dataframe
            if base_name != current_base:
                current_base = base_name
                if base_name not in seen_groups:
                    seen_groups[base_name] = group_counter
                    group_counter += 1

            group_num = seen_groups[base_name]
            new_col_name = f"{group_num:04d}_{col}"
            new_columns.append(new_col_name)

    df.columns = new_columns

    if verbose:
        print(f"\nTotal unique variable groups created: {len(seen_groups)}")
        print(f"Columns renamed: {len([old for old, new in zip(columns, new_columns) if old != new])}")

        duplicate_cols = df.columns[df.columns.duplicated()].tolist()
        if duplicate_cols:
            print(f"\n⚠️ Warning: Still have {len(duplicate_cols)} duplicate columns!")
            print(f"   Examples: {duplicate_cols[:5]}")
        else:
            print("\n✅ Success: No duplicate columns remaining!")

    return df


def clean_dataframe(df, add_prefixes=False, verbose=True):
    """
    Apply all cleaning steps to a dataframe
    """
    print("\n" + "="*60)
    print("STARTING DATA CLEANING PIPELINE")
    print("="*60)

    print("\n[STEP 1] Fixing date columns...")
    df = fix_date_columns(df, verbose=verbose)

    print("\n[STEP 2] Fixing year columns...")
    df = fix_year_columns(df, verbose=verbose)

    print("\n[STEP 3] Standardizing missing values...")
    df = standardize_missing_values(df, verbose=verbose)

    if add_prefixes:
        print("\n[STEP 4] Adding group prefixes...")
        df = add_group_prefixes(df, verbose=verbose)

    print("\n" + "="*60)
    print("CLEANING COMPLETED")
    print("="*60)

    return df


# ============================================================================
# MAIN PIPELINE
# ============================================================================

print("="*60)
print("COMPLETE DATA PROCESSING PIPELINE")
print("="*60)

# Memory check function
def check_memory():
    try:
        import psutil
        memory = psutil.virtual_memory()
        print(f"Available RAM: {memory.available / (1024**3):.1f} GB")
        print(f"Used RAM: {memory.percent:.1f}%")
    except ImportError:
        print("psutil not available - skipping memory check")

check_memory()

# ============================================================================
# STEP 1: CLEAN DF_MAIN
# ============================================================================
print("\n[STEP 1] CLEANING DF_MAIN...")

# Clean df_main (without prefixes)
# Making a copy to ensure original df_main is not modified
df_main_clean = clean_dataframe(df_main.copy(), add_prefixes=False, verbose=True)

print(f"\ndf_main_clean shape: {df_main_clean.shape}")
gc.collect()

# ============================================================================
# STEP 2: CLEAN DF_FINANCE AND ADD PREFIXES
# ============================================================================
print("\n[STEP 2] CLEANING DF_FINANCE...")

# First, analyze columns with high missing rates BEFORE cleaning
print("\n🔍 Analyzing df_finance columns for missing data...")
finance_info = []
for col in df_finance.columns:
    if col != 'Tax code number':
        total_count = len(df_finance)
        null_count = df_finance[col].isna().sum()
        null_percentage = (null_count / total_count) * 100
        finance_info.append((col, null_percentage, null_count))

# Sort by null percentage
finance_info.sort(key=lambda x: x[1], reverse=True)

print(f"Total columns in df_finance: {len(df_finance.columns)}")
print(f"Columns with >95% missing data: {sum(1 for _, pct, _ in finance_info if pct > 95)}")
print(f"Columns with >90% missing data: {sum(1 for _, pct, _ in finance_info if pct > 90)}")

# Drop columns with >95% missing BEFORE cleaning
missing_threshold = 95
cols_to_drop = [col for col, pct, _ in finance_info if pct > missing_threshold]
print(f"\n📉 Dropping {len(cols_to_drop)} columns with >{missing_threshold}% missing data")

df_finance_reduced = df_finance.drop(columns=cols_to_drop)
print(f"df_finance shape: {df_finance.shape} → {df_finance_reduced.shape}")

# Clear original df_finance
del df_finance
gc.collect()

# Clean df_finance_reduced (with prefixes to avoid column conflicts)
df_finance_clean = clean_dataframe(df_finance_reduced.copy(), add_prefixes=True, verbose=True)

print(f"\ndf_finance_clean shape: {df_finance_clean.shape}")
del df_finance_reduced
gc.collect()
check_memory()

# ============================================================================
# STEP 3: MERGE WITH TRACKING OF MISSING CODES
# ============================================================================
print("\n[STEP 3] MERGING DATAFRAMES...")

# Check for duplicates in df_finance_clean before merging
finance_duplicates = df_finance_clean['Tax code number'].duplicated().sum()
if finance_duplicates > 0:
    print(f"\n⚠️ Found {finance_duplicates} duplicate tax codes in df_finance_clean")
    print("Removing duplicates (keeping first occurrence)...")
    df_finance_clean = df_finance_clean.drop_duplicates(subset=['Tax code number'], keep='first')
    print(f"df_finance_clean shape after deduplication: {df_finance_clean.shape}")

# Get tax codes for analysis
main_codes = set(df_main_clean['Tax code number'].dropna().unique())
finance_codes = set(df_finance_clean['Tax code number'].dropna().unique())
missing_codes = main_codes - finance_codes

print(f"\nTax code analysis:")
print(f"  - Codes in df_main_clean: {len(main_codes):,}")
print(f"  - Codes in df_finance_clean: {len(finance_codes):,}")
print(f"  - Missing in df_finance: {len(missing_codes):,} ({len(missing_codes)/len(main_codes)*100:.1f}%)")

# Save missing codes
if len(missing_codes) > 0:
    missing_codes_sorted = sorted(list(map(str, missing_codes))) # Ensure codes are strings for sorting
    txt_path = os.path.join(path_data, 'missing_tax_codes.txt')
    with open(txt_path, 'w') as f:
        for code in missing_codes_sorted:
            f.write(f"{code}\n")
    print(f"\nMissing tax codes saved to: {txt_path}")
    print(f"First 10 missing codes: {missing_codes_sorted[:10]}")

# Perform chunked merge for memory efficiency
print(f"\n🔄 PERFORMING CHUNKED MERGE...")
chunk_size = 20000
chunks = []
total_chunks = (len(df_main_clean) + chunk_size - 1) // chunk_size

print(f"Processing {len(df_main_clean):,} rows in {total_chunks} chunks of {chunk_size:,} rows each")

for i in range(0, len(df_main_clean), chunk_size):
    chunk_num = (i // chunk_size) + 1
    print(f"  Processing chunk {chunk_num}/{total_chunks}...")

    # Get chunk of main dataframe
    df_chunk = df_main_clean.iloc[i:i+chunk_size].copy()

    # Merge chunk with finance data
    chunk_merged = pd.merge(
        df_chunk,
        df_finance_clean,
        on='Tax code number',
        how='left',
        suffixes=('', '_finance'),
        validate='one_to_one'
    )

    chunks.append(chunk_merged)

    # Clear chunk
    del df_chunk, chunk_merged
    gc.collect()

    if chunk_num % 5 == 0:
        check_memory()

print(f"\n🔗 COMBINING CHUNKS...")
df_merged = pd.concat(chunks, ignore_index=True)

# Clear chunks
del chunks, df_main_clean, df_finance_clean
gc.collect()

print(f"\n✅ MERGE COMPLETED!")
print(f"Final merged shape: {df_merged.shape}")

# ============================================================================
# STEP 4: REMOVE COLUMNS WITH >95% MISSING IN FINAL DATASET
# ============================================================================
print("\n[STEP 4] ANALYZING FINAL DATASET FOR MISSING DATA...")

# Analyze missing data in merged dataset
final_missing_info = []
for col in df_merged.columns:
    if col not in ['Tax code number', 'Company name']:
        total_count = len(df_merged)
        null_count = df_merged[col].isna().sum()
        null_percentage = (null_count / total_count) * 100
        final_missing_info.append((col, null_percentage))

# Sort by missing percentage
final_missing_info.sort(key=lambda x: x[1], reverse=True)

# Count columns by missing percentage
print(f"\nMissing data analysis in final dataset:")
print(f"  - Columns with >95% missing: {sum(1 for _, pct in final_missing_info if pct > 95)}")
print(f"  - Columns with >90% missing: {sum(1 for _, pct in final_missing_info if pct > 90)}")
print(f"  - Columns with >80% missing: {sum(1 for _, pct in final_missing_info if pct > 80)}")

# Drop columns with >95% missing from final dataset
final_cols_to_drop = [col for col, pct in final_missing_info if pct > 95]
if final_cols_to_drop:
    print(f"\n📉 Dropping {len(final_cols_to_drop)} columns with >95% missing from final dataset")
    df_final = df_merged.drop(columns=final_cols_to_drop)
    print(f"Final shape: {df_merged.shape} → {df_final.shape}")
else:
    df_final = df_merged
    print("\n✅ No columns with >95% missing in final dataset")

del df_merged
gc.collect()

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*60)
print("PIPELINE COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\nFinal dataset summary:")
print(f"  - Shape: {df_final.shape}")
print(f"  - Unique companies: {df_final['Tax code number'].nunique():,}")
print(f"  - Companies missing financial data: {len(missing_codes):,}")
print(f"  - Missing codes saved to: missing_tax_codes.txt")
print("\n✅ df_final is ready to be saved using the Save cell")
print("="*60)

check_memory()

In [ ]:
#@title Save
# Save df_final to pickle file
df_final_pkl = os.path.join(path_data, 'df_final_processed.pkl')

print("="*60)
print("SAVING FINAL DATASET")
print("="*60)

print(f"Saving df_final to: {df_final_pkl}")
print(f"Dataset shape: {df_final.shape}")
print(f"Columns: {len(df_final.columns)}")
print(f"Unique companies: {df_final['Tax code number'].nunique():,}")

# Save to pickle
with open(df_final_pkl, 'wb') as f:
    pickle.dump(df_final, f)

# Verify file was created
if os.path.exists(df_final_pkl):
    file_size = os.path.getsize(df_final_pkl) / (1024**3)  # Convert to GB
    print(f"\n✅ Successfully saved!")
    print(f"File size: {file_size:.2f} GB")
else:
    print("\n❌ Error: File was not saved properly!")

print("="*60)

## RESHAPE LONG

In [ ]:
#@title Load
# Load df_final from pickle file
df_final_pkl = os.path.join(path_data, 'df_final_processed.pkl')

print("="*60)
print("LOADING FINAL DATASET")
print("="*60)

if os.path.exists(df_final_pkl):
    print(f"Loading df_final from: {df_final_pkl}")

    # Load from pickle
    with open(df_final_pkl, 'rb') as f:
        df_final = pickle.load(f)

    print(f"\n✅ Successfully loaded!")
    print(f"Dataset shape: {df_final.shape}")
    print(f"Columns: {len(df_final.columns)}")
    print(f"Unique companies: {df_final['Tax code number'].nunique():,}")

else:
    print(f"❌ Error: File not found at {df_final_pkl}")
    print("Please make sure you have run the processing pipeline and saved the data first.")

print("="*60)

In [ ]:
#@title visualize big database

# Show *every* column
pd.set_option('display.max_columns', 1000)

# (Optionally widen your display so lines don’t wrap too crazily)
pd.set_option('display.width', 20)
pd.set_option('display.max_colwidth', 50)

# Now when you do…
#df

#print(final_df.iloc[:, :10])
#df.iloc[:30, :200]
#df.iloc[1000:1030, :200]

df_final.iloc[985:1015, :50]

In [ ]:
#@title RENAME COLUMNS: Move Suffix '.1'/'.2' to Prefix (Corrected)
import re

print("="*60)
print("STANDARDIZING COLUMN NAMES (MOVING SUFFIX TO PREFIX)")
print("="*60)

# This regex is designed to parse names like '0026_...2024.1'
# It captures three groups:
# 1. (\d{4})   - The 4-digit prefix
# 2. (.*\d{4}) - The middle part of the name, ensuring it ends with the 4-digit year
# 3. (\d+)     - The suffix number after the dot
pattern = re.compile(r'^(\d{4})_(.*\d{4})\.(\d+)$', re.DOTALL)

rename_map = {}
for col in df_final.columns:
    match = pattern.match(str(col))
    if match:
        # Extract the parts
        prefix = match.group(1)      # e.g., '0026'
        base_name = match.group(2)   # e.g., 'Additions in progress...\n2024'
        suffix = match.group(3)      # e.g., '1'

        # Construct the new column name
        new_col_name = f"{prefix}.{suffix}_{base_name}"
        rename_map[col] = new_col_name

if not rename_map:
    print("✅ No columns with a '.1', '.2', etc. suffix were found to rename.")
else:
    print(f"Found {len(rename_map)} columns to rename. Applying changes...")

    # Show some examples of the renaming
    print("\n--- Examples of Renaming ---")
    for i, (old, new) in enumerate(rename_map.items()):
        if i >= 5: # Limit to 5 examples
            break
        print(f"  '{old}'\n    will be renamed to\n  '{new}'\n")

    # Apply the renaming to the dataframe
    df_final.rename(columns=rename_map, inplace=True)
    print(f"\n✅ Successfully renamed {len(rename_map)} columns.")
    gc.collect()

In [ ]:
#@title MANUAL CLEANING
print("="*60)
print("STEP 1: MANUAL COLUMN CLEANING")
print("="*60)

# List of columns to remove
cols_to_drop = [
    '_source_file',
    '_import_order',
    'Unnamed: 0',
    'Company name',
    'Legal status',
    'Company category',
    'Last accounting closing date',
    'Unnamed: 0_finance',
    'Company name_finance'
]

print(f"Original shape: {df_final.shape}")

# Find which of these columns actually exist in the dataframe
existing_cols_to_drop = [col for col in cols_to_drop if col in df_final.columns]
print(f"\nFound {len(existing_cols_to_drop)} columns to drop: {existing_cols_to_drop}")

# Drop the columns
df_final.drop(columns=existing_cols_to_drop, inplace=True)
gc.collect()

print(f"\nNew shape after dropping columns: {df_final.shape}")


# --- 2. VARIABLE NAME VERIFICATION ---
print("\n" + "="*60)
print("STEP 2: VERIFYING COLUMN NAME FORMATS")
print("="*60)

# Whitelist of columns that are allowed to NOT end with a 4-digit number
whitelist = [
    'Tax code number',
    'Registered office address - Postal code',
    'Registered office address - Longitude',
    'Registered office address - Latitude',
    'Trading address - Postal code',
    'Incorporation year',
    'Legal form'
]

# Regex to check if a string ends with 4 digits
ends_with_4_digits = re.compile(r'\d{4}$')

non_conforming_columns = []
for col in df_final.columns:
    # Check if the column is in our whitelist
    if col in whitelist:
        continue

    # Check if the column name ends with a 4-digit number
    if not ends_with_4_digits.search(str(col)):
        non_conforming_columns.append(col)

# --- 3. REPORT RESULTS ---
if not non_conforming_columns:
    print("\n✅ SUCCESS: All columns (not in the whitelist) end with a 4-digit number.")
else:
    print(f"\n⚠️ WARNING: Found {len(non_conforming_columns)} columns that are not in the whitelist and do not end with a 4-digit year.")
    print("These columns may be static identifiers or need further cleaning before reshaping:")
    for col in non_conforming_columns:
        print(f"  - {col}")

In [ ]:
#@title visualize big database

# Show *every* column
pd.set_option('display.max_columns', 1000)

# (Optionally widen your display so lines don’t wrap too crazily)
pd.set_option('display.width', 20)
pd.set_option('display.max_colwidth', 50)

# Now when you do…
#df

#print(final_df.iloc[:, :10])
#df.iloc[:30, :200]
#df.iloc[1000:1030, :200]

df_final.iloc[985:1015, :50]


In [ ]:
#@title RESHAPE TO LONG FORMAT - MEMORY EFFICIENT VERSION
import pandas as pd
import numpy as np
import re
import gc
import os
import pickle
from datetime import datetime

print("="*60)
print("RESHAPING TO LONG FORMAT (MEMORY EFFICIENT)")
print("="*60)

# Define whitelist variables that should be constant across years
whitelist = [
    'Tax code number',
    'Registered office address - Postal code',
    'Registered office address - Longitude',
    'Registered office address - Latitude',
    'Trading address - Postal code',
    'Incorporation year',
    'Legal form'
]

# Define year range
years = list(range(2015, 2025))  # 2015 to 2024

print(f"Initial shape: {df_final.shape}")
print(f"Years to process: {years}")
print(f"Expected rows after reshape: ~{len(df_final) * len(years):,}")

# ============================================================================
# STEP 1: IDENTIFY COLUMNS WITH YEARS (same as before)
# ============================================================================
print("\n[STEP 1] IDENTIFYING COLUMNS WITH YEARS...")

# Pattern to match 4-digit year at the end of column name
year_pattern = re.compile(r'(\d{4})$')

# Categorize columns
columns_with_years = {}  # {base_name: {year: full_column_name}}
columns_without_years = []
whitelist_found = []

for col in df_final.columns:
    if col in whitelist:
        whitelist_found.append(col)
        continue

    match = year_pattern.search(col)
    if match:
        year = int(match.group(1))
        if year in years:
            # Extract base name (everything before the year)
            base_name = col[:match.start()]
            if base_name not in columns_with_years:
                columns_with_years[base_name] = {}
            columns_with_years[base_name][year] = col
        else:
            # Year outside our range
            columns_without_years.append(col)
    else:
        # No year found
        columns_without_years.append(col)

print(f"\nColumn analysis:")
print(f"  - Whitelist variables found: {len(whitelist_found)}")
print(f"  - Variables with years: {len(columns_with_years)}")
print(f"  - Columns without years (non-whitelist): {len(columns_without_years)}")

# Save column mapping for later use
column_mapping = {
    'whitelist_found': whitelist_found,
    'columns_with_years': columns_with_years,
    'columns_without_years': columns_without_years
}

mapping_path = os.path.join(path_data, 'reshape_column_mapping.pkl')
with open(mapping_path, 'wb') as f:
    pickle.dump(column_mapping, f)
print(f"\nColumn mapping saved to: {mapping_path}")

# ============================================================================
# STEP 2: RESHAPE IN CHUNKS AND SAVE
# ============================================================================
print("\n[STEP 2] RESHAPING DATA IN CHUNKS...")

# Process in chunks of companies
chunk_size = 5000  # Process 5000 companies at a time
num_chunks = (len(df_final) + chunk_size - 1) // chunk_size

print(f"Processing {len(df_final):,} companies in {num_chunks} chunks of up to {chunk_size:,} companies each")

# Track processed chunks
chunk_files = []

for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * chunk_size
    end_idx = min((chunk_idx + 1) * chunk_size, len(df_final))

    print(f"\nProcessing chunk {chunk_idx + 1}/{num_chunks} (companies {start_idx:,} to {end_idx:,})...")

    # Get chunk of companies
    df_chunk = df_final.iloc[start_idx:end_idx]

    # Initialize list for this chunk's data
    chunk_long_data = []

    # Process each company in the chunk
    for idx, row in df_chunk.iterrows():
        # Get whitelist values for this company
        company_static = {}
        for col in whitelist_found:
            company_static[col] = row[col]

        # Create a row for each year
        for year in years:
            year_row = company_static.copy()
            year_row['Year'] = year

            # Add financial variables for this year
            for base_name, year_cols in columns_with_years.items():
                if year in year_cols:
                    # Data exists for this year
                    year_row[base_name] = row[year_cols[year]]
                else:
                    # No data for this year
                    year_row[base_name] = np.nan

            chunk_long_data.append(year_row)

    # Convert chunk to DataFrame
    df_chunk_long = pd.DataFrame(chunk_long_data)

    # Sort by company and year
    df_chunk_long = df_chunk_long.sort_values(['Tax code number', 'Year'], ignore_index=True)

    # Save chunk
    chunk_file = os.path.join(path_data, f'df_long_chunk_{chunk_idx:04d}.pkl')
    with open(chunk_file, 'wb') as f:
        pickle.dump(df_chunk_long, f)

    chunk_files.append(chunk_file)
    print(f"  Saved chunk {chunk_idx + 1} with {len(df_chunk_long):,} rows to: {chunk_file}")

    # Clear memory
    del df_chunk, chunk_long_data, df_chunk_long
    gc.collect()

# Save list of chunk files
chunk_list_file = os.path.join(path_data, 'df_long_chunk_list.pkl')
with open(chunk_list_file, 'wb') as f:
    pickle.dump(chunk_files, f)

print(f"\n✅ RESHAPE COMPLETED!")
print(f"Created {len(chunk_files)} chunk files")
print(f"Chunk list saved to: {chunk_list_file}")

print("\n" + "="*60)
print("RESHAPE TO PANEL DATA COMPLETED!")
print("="*60)
print("Data saved in chunks to avoid memory issues")
print("Use the Load cell to combine chunks when needed")

In [ ]:
#@title Save Long Format (Chunked)
# This cell combines all chunks into one file and saves to your path
import pandas as pd
import pickle
import gc
import os

print("="*60)
print("SAVING LONG FORMAT DATASET")
print("="*60)

# Load chunk list
chunk_list_file = os.path.join(path_data, 'df_long_chunk_list.pkl')

if os.path.exists(chunk_list_file):
    with open(chunk_list_file, 'rb') as f:
        chunk_files = pickle.load(f)
    print(f"Found {len(chunk_files)} chunk files to combine")

    # Load and combine all chunks
    print("\nLoading and combining chunks...")
    all_chunks = []

    for i, chunk_file in enumerate(chunk_files):
        if os.path.exists(chunk_file):
            print(f"  Loading chunk {i+1}/{len(chunk_files)}...")
            with open(chunk_file, 'rb') as f:
                chunk = pickle.load(f)
            all_chunks.append(chunk)
        else:
            print(f"  ❌ Missing chunk file: {chunk_file}")

    # Combine all chunks
    print("\nCombining all chunks into single dataframe...")
    df_long = pd.concat(all_chunks, ignore_index=True)

    # Clear chunks from memory
    del all_chunks
    gc.collect()

    # Save combined file
    df_long_pkl = os.path.join(path_data, 'df_long_panel.pkl')

    print(f"\nSaving df_long to: {df_long_pkl}")
    print(f"Dataset shape: {df_long.shape}")
    print(f"Companies: {df_long['Tax code number'].nunique():,}")

    with open(df_long_pkl, 'wb') as f:
        pickle.dump(df_long, f)

    # Verify file was created
    if os.path.exists(df_long_pkl):
        file_size = os.path.getsize(df_long_pkl) / (1024**3)  # Convert to GB
        print(f"\n✅ Successfully saved!")
        print(f"File size: {file_size:.2f} GB")

        # Optional: Clean up chunk files
        print("\n🧹 Cleaning up chunk files...")
        for chunk_file in chunk_files:
            if os.path.exists(chunk_file):
                os.remove(chunk_file)
        os.remove(chunk_list_file)
        print("✅ Chunk files removed")
    else:
        print("\n❌ Error: File was not saved properly!")
else:
    print("❌ No chunk files found. Please run the reshape process first.")

print("="*60)

## LAST MANIPULATION

In [ ]:
#@title Load Long Format
# Load df_long from pickle file
df_long_pkl = os.path.join(path_data, 'df_long_panel.pkl')

print("="*60)
print("LOADING LONG FORMAT DATASET")
print("="*60)

if os.path.exists(df_long_pkl):
    print(f"Loading df_long from: {df_long_pkl}")

    # Load from pickle
    with open(df_long_pkl, 'rb') as f:
        df_long = pickle.load(f)

    print(f"\n✅ Successfully loaded!")
    print(f"Dataset shape: {df_long.shape}")
    print(f"Companies: {df_long['Tax code number'].nunique():,}")
    print(f"Years covered: {df_long['Year'].min()} - {df_long['Year'].max()}")
    print(f"Rows per company: {df_long.groupby('Tax code number').size().value_counts().sort_index().to_dict()}")

    # Memory usage
    memory_usage = df_long.memory_usage(deep=True).sum() / 1024**3
    print(f"Memory usage: {memory_usage:.2f} GB")

else:
    print(f"❌ Error: File not found at {df_long_pkl}")
    print("Please make sure you have run the reshape process and save cell first.")

print("="*60)

In [ ]:
#@title CREATE YEAR FROM INCORPORATION & CLEAN DATA - FIXED
import pandas as pd
import numpy as np
from datetime import datetime

print("="*60)
print("CREATING YEAR FROM INCORPORATION & CLEANING DATA")
print("="*60)

print(f"Initial shape: {df_long.shape}")

# ============================================================================
# STEP 1: CREATE 'YEAR FROM INCORPORATION' VARIABLE
# ============================================================================
print("\n[STEP 1] CREATING 'YEAR FROM INCORPORATION' VARIABLE...")

# First, check the actual column name and format
incorporation_cols = [col for col in df_long.columns if 'incorporat' in col.lower()]
print(f"Found incorporation columns: {incorporation_cols}")

# Assuming the column is 'Incorporation year' based on whitelist
if 'Incorporation year' in df_long.columns:
    # Check data type and sample values
    print(f"\nIncorporation year data type: {df_long['Incorporation year'].dtype}")
    print(f"Sample values: {df_long['Incorporation year'].dropna().head()}")

    # Convert to datetime if it's not already
    if pd.api.types.is_datetime64_any_dtype(df_long['Incorporation year']):
        # Extract year from datetime
        df_long['Incorporation_year_numeric'] = pd.to_datetime(df_long['Incorporation year']).dt.year
    else:
        # Try to convert to datetime first
        df_long['Incorporation_year_numeric'] = pd.to_datetime(df_long['Incorporation year'], errors='coerce').dt.year

    # Create 'Year from incorporation'
    df_long['Year from incorporation'] = df_long['Year'] - df_long['Incorporation_year_numeric']

    # Show statistics
    print(f"\nYear from incorporation statistics:")
    print(f"  - Min: {df_long['Year from incorporation'].min()}")
    print(f"  - Max: {df_long['Year from incorporation'].max()}")
    print(f"  - Mean: {df_long['Year from incorporation'].mean():.1f}")
    print(f"  - Negative values (before incorporation): {(df_long['Year from incorporation'] < 0).sum():,}")

    # Clean up temporary column
    df_long = df_long.drop('Incorporation_year_numeric', axis=1)
else:
    print("❌ 'Incorporation year' column not found!")
    print("Available columns:", df_long.columns.tolist())

# ============================================================================
# STEP 2: DELETE ALL OBSERVATIONS BEFORE INCORPORATION
# ============================================================================
print("\n[STEP 2] REMOVING ALL OBSERVATIONS BEFORE INCORPORATION...")

initial_rows = len(df_long)

# Remove ALL rows where Year from incorporation is negative
if 'Year from incorporation' in df_long.columns:
    df_long = df_long[df_long['Year from incorporation'] >= 0].copy()
    removed_rows = initial_rows - len(df_long)
    print(f"  - Removed {removed_rows:,} observations ({removed_rows/initial_rows*100:.1f}%) before incorporation")
    print(f"  - Remaining observations: {len(df_long):,}")

# ============================================================================
# STEP 3: FINAL STATISTICS
# ============================================================================
print("\n[STEP 3] FINAL STATISTICS...")

# Companies statistics
companies_remaining = df_long['Tax code number'].nunique()
obs_per_company = df_long.groupby('Tax code number').size()

print(f"\nDataset summary:")
print(f"  - Total observations: {len(df_long):,}")
print(f"  - Unique companies: {companies_remaining:,}")
print(f"  - Average observations per company: {obs_per_company.mean():.1f}")
print(f"  - Distribution of observations per company:")
for n_obs, count in obs_per_company.value_counts().sort_index().items():
    print(f"    * {n_obs} observations: {count:,} companies ({count/companies_remaining*100:.1f}%)")

# Year coverage
print(f"\nYear coverage after cleaning:")
year_counts = df_long['Year'].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  - {year}: {count:,} observations")

print("\n" + "="*60)
print("DATA CLEANING COMPLETED!")
print("="*60)
print(f"Original shape: {initial_rows:,} rows → Final shape: {len(df_long):,} rows")
print(f"Removed: {initial_rows - len(df_long):,} observations ({(initial_rows - len(df_long))/initial_rows*100:.1f}%)")

# The cleaned data remains in df_long

In [ ]:
#@title DIAGNOSE YEAR MISMATCH FOR EVENTS
import pandas as pd
import numpy as np

print("="*60)
print("DIAGNOSING YEAR MISMATCH FOR EVENTS")
print("="*60)

# Load the event data again
event_dataframes = []
for file_name in file_names:
    file_path = os.path.join(path_data, file_name)
    df_temp = pd.read_excel(
        file_path,
        sheet_name='Results',
        usecols=['Tax code number', 'Procedure/cessazione', 'Date of open procedure/cessazione']
    )
    event_dataframes.append(df_temp)

df_events = pd.concat(event_dataframes, ignore_index=True)
df_events['Tax code number'] = df_events['Tax code number'].fillna(method='ffill')
df_events = df_events.dropna(subset=['Tax code number'])
df_events['Date of open procedure/cessazione'] = pd.to_datetime(df_events['Date of open procedure/cessazione'], errors='coerce')
df_events['Year of event'] = df_events['Date of open procedure/cessazione'].dt.year
df_events = df_events.rename(columns={'Procedure/cessazione': 'Event'})

# Focus on bankruptcy and judicial liquidation WITH VALID YEARS
bankruptcy_events = df_events[
    (df_events['Event'].str.contains('Bankruptcy', case=False, na=False)) &
    (df_events['Year of event'].notna())
]
liquidation_events = df_events[
    (df_events['Event'].str.contains('Judicial liquidation', case=False, na=False)) &
    (df_events['Year of event'].notna())
]

print(f"\nEvents with valid years:")
print(f"  - Bankruptcy: {len(bankruptcy_events)}")
print(f"  - Judicial liquidation: {len(liquidation_events)}")

# Check which companies are in our panel
panel_companies = set(df_long['Tax code number'].unique())

# For each event, check if the company-year combination exists in panel
bankruptcy_matches = []
liquidation_matches = []

print("\nChecking company-year matches...")

# Check bankruptcy matches
for _, event in bankruptcy_events.iterrows():
    tax_code = event['Tax code number']
    year = int(event['Year of event'])

    # Check if this company-year exists in panel
    exists = ((df_long['Tax code number'] == tax_code) & (df_long['Year'] == year)).any()
    bankruptcy_matches.append({
        'tax_code': tax_code,
        'year': year,
        'in_panel': tax_code in panel_companies,
        'year_match': exists
    })

# Check liquidation matches
for _, event in liquidation_events.iterrows():
    tax_code = event['Tax code number']
    year = int(event['Year of event'])

    # Check if this company-year exists in panel
    exists = ((df_long['Tax code number'] == tax_code) & (df_long['Year'] == year)).any()
    liquidation_matches.append({
        'tax_code': tax_code,
        'year': year,
        'in_panel': tax_code in panel_companies,
        'year_match': exists
    })

# Analyze results
bankruptcy_df = pd.DataFrame(bankruptcy_matches)
liquidation_df = pd.DataFrame(liquidation_matches)

print("\nBankruptcy analysis:")
print(f"  - Total events: {len(bankruptcy_df)}")
print(f"  - Companies in panel: {bankruptcy_df['in_panel'].sum()}")
print(f"  - Company-year matches: {bankruptcy_df['year_match'].sum()}")

print("\nJudicial liquidation analysis:")
print(f"  - Total events: {len(liquidation_df)}")
print(f"  - Companies in panel: {liquidation_df['in_panel'].sum()}")
print(f"  - Company-year matches: {liquidation_df['year_match'].sum()}")

# Year distribution for events where company is in panel but year doesn't match
print("\nYear distribution for companies IN panel but year NOT matching:")

bankruptcy_mismatch = bankruptcy_df[bankruptcy_df['in_panel'] & ~bankruptcy_df['year_match']]
if len(bankruptcy_mismatch) > 0:
    print("\nBankruptcy events - year distribution:")
    year_counts = bankruptcy_mismatch['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        in_range = "✓" if 2015 <= year <= 2024 else "✗"
        print(f"  {year}: {count} events {in_range}")

liquidation_mismatch = liquidation_df[liquidation_df['in_panel'] & ~liquidation_df['year_match']]
if len(liquidation_mismatch) > 0:
    print("\nJudicial liquidation events - year distribution:")
    year_counts = liquidation_mismatch['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        in_range = "✓" if 2015 <= year <= 2024 else "✗"
        print(f"  {year}: {count} events {in_range}")

# Check what years these companies have in the panel
print("\n🔍 SAMPLE CHECK - First 5 bankruptcy companies with year mismatch:")
sample_companies = bankruptcy_mismatch.head()['tax_code'].unique()[:5]
for tax_code in sample_companies:
    event_year = bankruptcy_mismatch[bankruptcy_mismatch['tax_code'] == tax_code]['year'].iloc[0]
    panel_years = sorted(df_long[df_long['Tax code number'] == tax_code]['Year'].unique())
    print(f"  Company {tax_code}:")
    print(f"    - Event year: {event_year}")
    print(f"    - Years in panel: {panel_years}")

print("="*60)

In [ ]:
#@title IMPORT AND PROCESS EVENT DATA - FIXED VERSION
import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print("="*60)
print("IMPORTING AND PROCESSING EVENT DATA (FIXED)")
print("="*60)

# ====================================================================
# STEP 1-4: IMPORT & PREPARE RAW EVENT DATA (UNCHANGED)
# ====================================================================
print("\n[STEP 1-4] IMPORTING AND PREPARING EVENT DATA...")

event_dataframes = []
for file_name in file_names:
    file_path = os.path.join(path_data, file_name)
    df_temp = pd.read_excel(
        file_path,
        sheet_name='Results',
        usecols=['Tax code number', 'Procedure/cessazione', 'Date of open procedure/cessazione']
    )
    event_dataframes.append(df_temp)

df_events = pd.concat(event_dataframes, ignore_index=True)
print(f"Combined event dataset shape: {df_events.shape}")

df_events['Tax code number'] = df_events['Tax code number'].fillna(method='ffill')
df_events = df_events.dropna(subset=['Tax code number'])
df_events['Date of open procedure/cessazione'] = pd.to_datetime(
    df_events['Date of open procedure/cessazione'], errors='coerce'
)
df_events['Year'] = df_events['Date of open procedure/cessazione'].dt.year
df_events = df_events.rename(columns={'Procedure/cessazione': 'Event'})
df_events_merge = df_events[['Tax code number', 'Year', 'Event']].dropna(subset=['Year'])
df_events_merge['Year'] = df_events_merge['Year'].astype(int)

del event_dataframes, df_events
gc.collect()

# ====================================================================
# STEP 5: CREATE DUMMIES FOR ALL EVENTS
# ====================================================================
print("\n[STEP 5] CREATING EVENT DUMMY VARIABLES FOR ALL EVENTS...")

# Ensure event names are clean
df_events_merge['Event'] = df_events_merge['Event'].str.strip()

# One-hot encode each unique event type
event_dummies = pd.get_dummies(df_events_merge['Event'], prefix='Event')
df_events_dummies = pd.concat(
    [df_events_merge[['Tax code number', 'Year']], event_dummies],
    axis=1
)

# If a company-year has multiple of the same event, keep it marked
df_events_dummies = (
    df_events_dummies
    .groupby(['Tax code number', 'Year'], as_index=False)
    .max()
)

print("Dummy columns created:", list(event_dummies.columns))

# ====================================================================
# STEP 6: MERGE EVENTS INTO PANEL DATA
# ====================================================================
print("\n[STEP 6] MERGING EVENT DUMMIES INTO PANEL DATA...")

df_long_before = len(df_long)
df_long = pd.merge(
    df_long,
    df_events_dummies,
    on=['Tax code number', 'Year'],
    how='left'
)

# Fill any NaNs in dummy columns with 0
for col in event_dummies.columns:
    df_long[col] = df_long[col].fillna(0).astype(int)

print(f"Rows before merge: {df_long_before:,}")
print(f"Rows after merge:  {len(df_long):,}")
print("Total event flags set:",
      df_long[event_dummies.columns].sum().sum())

# ====================================================================
# STEP 7: IMPROVED VISUALIZATION (UNCHANGED)
# ====================================================================
print("\n[STEP 7] CREATING IMPROVED VISUALIZATION...")

# Aggregate by year
yearly_events = df_long.groupby('Year')[event_dummies.columns].sum()

plt.figure(figsize=(14, 8))
plt.style.use('seaborn-v0_8-darkgrid')

years = yearly_events.index
for col in event_dummies.columns:
    # derive marker and label from column name
    label = col.replace('Event_', '')
    marker = 'o' if 'Bankruptcy' in label else 's'
    plt.plot(
        years,
        yearly_events[col].values,
        f"{marker}-",
        linewidth=3,
        markersize=10,
        label=label,
        alpha=0.8
    )
    # annotate non-zero points
    for x, y in zip(years, yearly_events[col].values):
        if y > 0:
            offset = (0, 10) if 'Bankruptcy' in label else (0, -15)
            plt.annotate(
                str(y),
                (x, y),
                textcoords="offset points",
                xytext=offset,
                ha='center',
                fontsize=10,
                fontweight='bold'
            )

plt.xlabel('Year', fontsize=14, fontweight='bold')
plt.ylabel('Number of Events', fontsize=14, fontweight='bold')
plt.title('Bankruptcy and Judicial Liquidation Events by Year', fontsize=16, fontweight='bold', pad=20)
plt.legend(fontsize=12, frameon=True, fancybox=True, shadow=True)
plt.ylim(0, yearly_events.values.max() * 1.2)
plt.xticks(years, fontsize=11)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print("\nDetailed yearly statistics:")
print(yearly_events)

print("\nTotal events in dataset:")
for col in event_dummies.columns:
    print(f"  - {col}: {df_long[col].sum()}")

print("\nCompanies with each event type:")
for col in event_dummies.columns:
    n_companies = df_long[df_long[col] == 1]['Tax code number'].nunique()
    print(f"  - {col}: {n_companies}")

print("\n" + "="*60)
print("EVENT PROCESSING COMPLETED!")
print("="*60)

# Clean up
del df_events_merge, df_events_dummies
gc.collect()


In [ ]:
#@title visualization
# ====================================================================
# STEP 7: UPDATED VISUALIZATIONS
# ====================================================================
print("\n[STEP 7] CREATING UPDATED VISUALIZATIONS...")

# Aggregate by year
yearly_events = df_long.groupby('Year')[event_dummies.columns].sum()
years = yearly_events.index

# 1) Plot: Event_Bankruptcy only
plt.figure(figsize=(12, 6))
plt.style.use('seaborn-v0_8-darkgrid')

plt.plot(
    years,
    yearly_events['Event_Bankruptcy'],
    'o-',
    linewidth=3,
    markersize=8,
    label='Bankruptcy',
    alpha=0.8
)
for x, y in zip(years, yearly_events['Event_Bankruptcy']):
    if y > 0:
        plt.annotate(
            str(y),
            (x, y),
            textcoords="offset points",
            xytext=(0,10),
            ha='center',
            fontsize=9,
            fontweight='bold'
        )

plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Bankruptcy Events', fontsize=12, fontweight='bold')
plt.title('Bankruptcy Events by Year', fontsize=14, fontweight='bold', pad=15)
plt.ylim(0, yearly_events['Event_Bankruptcy'].max() * 1.2)
plt.xticks(years)
plt.tight_layout()
plt.show()

# 2) Plot: Bankruptcy vs Judicial Liquidation
plt.figure(figsize=(12, 6))
plt.style.use('seaborn-v0_8-darkgrid')

plt.plot(
    years,
    yearly_events['Event_Bankruptcy'],
    'o-',
    linewidth=3,
    markersize=8,
    label='Bankruptcy',
    alpha=0.8
)
plt.plot(
    years,
    yearly_events['Event_Judicial liquidation'],
    's-',
    linewidth=3,
    markersize=8,
    label='Judicial Liquidation',
    alpha=0.8
)
for label, marker_offset in [('Event_Bankruptcy', (0,10)),
                              ('Event_Judicial liquidation', (0,-12))]:
    for x, y in zip(years, yearly_events[label]):
        if y > 0:
            plt.annotate(
                str(y),
                (x, y),
                textcoords="offset points",
                xytext=marker_offset,
                ha='center',
                fontsize=9,
                fontweight='bold'
            )

plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Events', fontsize=12, fontweight='bold')
plt.title('Bankruptcy vs Judicial Liquidation by Year', fontsize=14, fontweight='bold', pad=15)
plt.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)
max_val = yearly_events[['Event_Bankruptcy', 'Event_Judicial liquidation']].values.max()
plt.ylim(0, max_val * 1.2)
plt.xticks(years)
plt.tight_layout()
plt.show()


### ADD EVENTS, CUT AT EVENT BANKRUPTCY/JL

In [ ]:
#@title CREATE TARGET VARIABLES - OPTIMIZED VERSION
import pandas as pd
import numpy as np

print("="*60)
print("CREATING TARGET VARIABLES - OPTIMIZED VERSION")
print("="*60)

print(f"Initial shape: {df_long.shape}")

# ============================================================================
# STEP 1: INITIALIZE TARGET VARIABLE
# ============================================================================
print("\n[STEP 1] INITIALIZING TARGET VARIABLE...")
df_long['Target'] = 0

# ============================================================================
# STEP 2: IDENTIFY FIRST EVENT YEAR PER COMPANY
# ============================================================================
print("\n[STEP 2] IDENTIFYING FIRST EVENT YEAR PER COMPANY...")
# Create a combined indicator for any event

#df_long['Any_Event'] = df_long['Event_Bankruptcy'] | df_long['Event_Judicial liquidation']
df_long['Any_Event'] = df_long['Event_Bankruptcy']

# Find the first event year for each company
first_event_year = (
    df_long[df_long['Any_Event'] == 1]
      .groupby('Tax code number')['Year']
      .min()
      .rename('First_Event_Year')
)

# Merge back into df_long
df_long = df_long.merge(
    first_event_year,
    on='Tax code number',
    how='left'
)

# ============================================================================
# STEP 3: ASSIGN TARGET = 1 FOR THE YEAR BEFORE THE FIRST EVENT
# ============================================================================
print("\n[STEP 3] ASSIGNING TARGET FOR YEAR BEFORE FIRST EVENT...")
df_long.loc[
    df_long['Year'] == df_long['First_Event_Year'] - 1,
    'Target'
] = 1

# Clean up helper columns
df_long.drop(columns=['Any_Event', 'First_Event_Year'], inplace=True)

print("Target variable created!")

# ============================================================================
# STEP 4: STATISTICS AND VALIDATION
# ============================================================================
print("\n[STEP 4] TARGET VARIABLE STATISTICS...")
total_targets = df_long['Target'].sum()
companies_with_target = df_long[df_long['Target'] == 1]['Tax code number'].nunique()
print(f"  - Observations with Target = 1: {total_targets:,}")
print(f"  - Companies with Target = 1: {companies_with_target:,}")

# Yearly distribution of Target
yearly_targets = df_long.groupby('Year')['Target'].sum().rename('Target_count')
print("\nYearly Target counts:")
print(yearly_targets)

print("\n" + "="*60)
print("TARGET VARIABLE CREATION COMPLETED!")
print("="*60)
print(f"Final shape: {df_long.shape} (unchanged)")
print("Created variable: Target")


In [ ]:
#@title IMPORT AND PROCESS CCIAA DATA
import pandas as pd
import numpy as np
from datetime import datetime

print("="*60)
print("IMPORTING AND PROCESSING CCIAA DATA")
print("="*60)

# ============================================================================
# STEP 1: IMPORT CCIAA FILES
# ============================================================================
print("\n[STEP 1] IMPORTING CCIAA FILES...")

# Import all CCIAA files
cciaa_dataframes = []

for file_name in file_CCIAA:
    file_path = os.path.join(path_data, file_name)
    print(f"Loading {file_name}...")

    # Read the CCIAA data
    df_temp = pd.read_excel(
        file_path,
        sheet_name='Results',
        usecols=['Tax code number', 'CCIAA number', 'Previous CCIAA', 'CCIAA change date']
    )
    cciaa_dataframes.append(df_temp)
    print(f"  - Shape: {df_temp.shape}")

# Concatenate all dataframes
df_cciaa_raw = pd.concat(cciaa_dataframes, ignore_index=True)
print(f"\nCombined CCIAA dataset shape: {df_cciaa_raw.shape}")

# Clear temporary dataframes
del cciaa_dataframes, df_temp
gc.collect()

# ============================================================================
# STEP 2: FILL TAX CODE AND CCIAA NUMBER
# ============================================================================
print("\n[STEP 2] FILLING TAX CODE AND CCIAA NUMBERS...")

# Show sample before filling
print("Sample before filling:")
print(df_cciaa_raw.head(10))

# Forward fill Tax code number and CCIAA number
df_cciaa_raw['Tax code number'] = df_cciaa_raw['Tax code number'].fillna(method='ffill')
df_cciaa_raw['CCIAA number'] = df_cciaa_raw['CCIAA number'].fillna(method='ffill')

print(f"\nAfter filling:")
print(f"  - Rows with Tax code: {df_cciaa_raw['Tax code number'].notna().sum():,}")
print(f"  - Rows with CCIAA number: {df_cciaa_raw['CCIAA number'].notna().sum():,}")

# Remove any rows without Tax code
df_cciaa_raw = df_cciaa_raw.dropna(subset=['Tax code number'])

# ============================================================================
# STEP 3: CREATE YEAR FROM CCIAA CHANGE DATE
# ============================================================================
print("\n[STEP 3] EXTRACTING YEAR FROM CCIAA CHANGE DATE...")

# Convert date to datetime and extract year
df_cciaa_raw['CCIAA change date'] = pd.to_datetime(df_cciaa_raw['CCIAA change date'], errors='coerce')
df_cciaa_raw['Year CCIAA'] = df_cciaa_raw['CCIAA change date'].dt.year

print(f"Year extraction results:")
print(f"  - Valid years extracted: {df_cciaa_raw['Year CCIAA'].notna().sum():,}")
print(f"  - Missing years: {df_cciaa_raw['Year CCIAA'].isna().sum():,}")

# ============================================================================
# STEP 4: PROCESS CCIAA CHANGES FOR EACH COMPANY
# ============================================================================
print("\n[STEP 4] PROCESSING CCIAA CHANGES...")

# Define year range
years = list(range(2015, 2025))  # 2015 to 2024

# Initialize list to collect all company-year data
all_company_years = []

# Get unique companies
unique_companies = df_cciaa_raw['Tax code number'].unique()
print(f"Processing {len(unique_companies):,} unique companies...")

for idx, company in enumerate(unique_companies):
    if idx % 5000 == 0:
        print(f"  Processing company {idx:,}/{len(unique_companies):,}...")

    # Get all rows for this company
    company_data = df_cciaa_raw[df_cciaa_raw['Tax code number'] == company].copy()

    # Sort by Year CCIAA to process changes chronologically
    company_data = company_data.sort_values('Year CCIAA')

    # Get the current CCIAA number (from first row after forward fill)
    current_cciaa = company_data['CCIAA number'].iloc[0]

    # Check if company has any CCIAA changes
    has_changes = company_data['Previous CCIAA'].notna().any()

    if not has_changes:
        # No changes - use current CCIAA for all years
        for year in years:
            all_company_years.append({
                'Tax code number': company,
                'Year': year,
                'CCIAA number adjusted': current_cciaa
            })
    else:
        # Company has changes - need to process them
        # Create a timeline of CCIAA numbers
        cciaa_timeline = {}

        # Start with current CCIAA for recent years
        for year in years:
            cciaa_timeline[year] = current_cciaa

        # Process each change (working backwards in time)
        for _, row in company_data.iterrows():
            if pd.notna(row['Previous CCIAA']) and pd.notna(row['Year CCIAA']):
                change_year = int(row['Year CCIAA'])
                previous_cciaa = row['Previous CCIAA']

                # For all years before the change, use the previous CCIAA
                for year in years:
                    if year < change_year:
                        cciaa_timeline[year] = previous_cciaa

        # Create rows for all years with the correct CCIAA
        for year in years:
            all_company_years.append({
                'Tax code number': company,
                'Year': year,
                'CCIAA number adjusted': cciaa_timeline[year]
            })

# ============================================================================
# STEP 5: CREATE FINAL CCIAA DATAFRAME
# ============================================================================
print("\n[STEP 5] CREATING FINAL CCIAA DATAFRAME...")

# Convert to DataFrame
df_cciaa_final = pd.DataFrame(all_company_years)

# Sort by company and year
df_cciaa_final = df_cciaa_final.sort_values(['Tax code number', 'Year']).reset_index(drop=True)

print(f"\nFinal CCIAA data shape: {df_cciaa_final.shape}")
print(f"Unique companies: {df_cciaa_final['Tax code number'].nunique():,}")
print(f"Years per company: {df_cciaa_final.groupby('Tax code number').size().value_counts().sort_index().to_dict()}")

# Show sample
print("\nSample of final CCIAA data:")
sample_company = df_cciaa_final['Tax code number'].iloc[0]
print(df_cciaa_final[df_cciaa_final['Tax code number'] == sample_company])

# ============================================================================
# STEP 6: MERGE WITH DF_LONG
# ============================================================================
print("\n[STEP 6] MERGING CCIAA DATA WITH PANEL DATA...")

# Check overlap
cciaa_companies = set(df_cciaa_final['Tax code number'].unique())
panel_companies = set(df_long['Tax code number'].unique())
overlap = cciaa_companies.intersection(panel_companies)

print(f"Company overlap:")
print(f"  - Companies with CCIAA data: {len(cciaa_companies):,}")
print(f"  - Companies in panel: {len(panel_companies):,}")
print(f"  - Companies in both: {len(overlap):,} ({len(overlap)/len(panel_companies)*100:.1f}%)")

# Perform merge
df_long_before = len(df_long)
df_long = pd.merge(
    df_long,
    df_cciaa_final,
    on=['Tax code number', 'Year'],
    how='left'
)

print(f"\nMerge completed:")
print(f"  - Rows before merge: {df_long_before:,}")
print(f"  - Rows after merge: {len(df_long):,}")
print(f"  - Rows with CCIAA data: {df_long['CCIAA number adjusted'].notna().sum():,}")

# ============================================================================
# STEP 7: FINAL STATISTICS
# ============================================================================
print("\n[STEP 7] CCIAA DATA STATISTICS...")

# CCIAA coverage
print(f"\nCCIAA coverage:")
print(f"  - Observations with CCIAA: {df_long['CCIAA number adjusted'].notna().sum():,} ({df_long['CCIAA number adjusted'].notna().mean()*100:.1f}%)")
print(f"  - Observations without CCIAA: {df_long['CCIAA number adjusted'].isna().sum():,}")

# Unique CCIAA numbers
print(f"\nUnique CCIAA numbers: {df_long['CCIAA number adjusted'].nunique():,}")

# Distribution by year
print(f"\nCCIAA coverage by year:")
cciaa_by_year = df_long.groupby('Year')['CCIAA number adjusted'].agg(['count', 'size'])
cciaa_by_year['coverage_pct'] = (cciaa_by_year['count'] / cciaa_by_year['size'] * 100).round(1)
print(cciaa_by_year[['coverage_pct']])

print("\n" + "="*60)
print("CCIAA PROCESSING COMPLETED!")
print("="*60)

# Clean up
del df_cciaa_raw, df_cciaa_final, all_company_years
gc.collect()

In [ ]:
#@title FINAL MERGE - KEEPING ONLY MAIN DATASET OBSERVATIONS
import pandas as pd
import numpy as np

print("="*60)
print("FINAL MERGE - MAINTAINING MAIN DATASET STRUCTURE")
print("="*60)

# Store the shape before merge
print(f"Main dataset shape before merge: {df_long.shape}")
print(f"Unique companies in main dataset: {df_long['Tax code number'].nunique():,}")

# Since df_long already has the CCIAA data merged, we don't need another merge
# The previous step already did the merge correctly with a left join

# Verify we haven't added any extra rows
print(f"\nVerification:")
print(f"Main dataset shape after CCIAA merge: {df_long.shape}")
print(f"Unique companies after CCIAA merge: {df_long['Tax code number'].nunique():,}")

# Show CCIAA coverage in the final dataset
print(f"\nCCIAA data coverage in final dataset:")
print(f"  - Observations with CCIAA: {df_long['CCIAA number adjusted'].notna().sum():,} ({df_long['CCIAA number adjusted'].notna().mean()*100:.1f}%)")
print(f"  - Observations without CCIAA: {df_long['CCIAA number adjusted'].isna().sum():,}")

# Final dataset is ready - it's still called df_long
print("\n✅ Final dataset ready with all features including CCIAA!")
print("="*60)

In [ ]:
#@title Save Final Dataset
# Save final df_long to pickle file
df_final_complete_pkl = os.path.join(path_data, 'df_final_complete.pkl')

print("="*60)
print("SAVING FINAL COMPLETE DATASET")
print("="*60)

print(f"Saving final dataset to: {df_final_complete_pkl}")
print(f"Dataset shape: {df_long.shape}")
print(f"Companies: {df_long['Tax code number'].nunique():,}")
print(f"Years covered: {df_long['Year'].min()} - {df_long['Year'].max()}")
print(f"Target variable distribution:")
print(f"  - Target = 0: {(df_long['Target'] == 0).sum():,} ({(df_long['Target'] == 0).mean()*100:.1f}%)")
print(f"  - Target = 1: {(df_long['Target'] == 1).sum():,} ({(df_long['Target'] == 1).mean()*100:.1f}%)")

# Save to pickle
with open(df_final_complete_pkl, 'wb') as f:
    pickle.dump(df_long, f)

# Verify file was created
if os.path.exists(df_final_complete_pkl):
    file_size = os.path.getsize(df_final_complete_pkl) / (1024**3)  # Convert to GB
    print(f"\n✅ Successfully saved!")
    print(f"File size: {file_size:.2f} GB")

    # Quick summary of what's included
    print(f"\nDataset includes:")
    print(f"  - Base company information: ✓")
    print(f"  - Financial data (cleaned): ✓")
    print(f"  - Event data (bankruptcy, liquidation): ✓")
    print(f"  - Target variable: ✓")
    print(f"  - CCIAA data: ✓")
else:
    print("\n❌ Error: File was not saved properly!")

print("="*60)

# MODEL general

---



---



## Data Ultimation before model


In [ ]:
#@title Load Final Dataset
# Load final df_long from pickle file
df_final_complete_pkl = os.path.join(path_data, 'df_final_complete.pkl')

print("="*60)
print("LOADING FINAL COMPLETE DATASET")
print("="*60)

if os.path.exists(df_final_complete_pkl):
    print(f"Loading final dataset from: {df_final_complete_pkl}")

    # Load from pickle
    with open(df_final_complete_pkl, 'rb') as f:
        df_long = pickle.load(f)

    print(f"\n✅ Successfully loaded!")
    print(f"Dataset shape: {df_long.shape}")
    print(f"Companies: {df_long['Tax code number'].nunique():,}")
    print(f"Years covered: {df_long['Year'].min()} - {df_long['Year'].max()}")

    # Show key features
    print(f"\nKey features present:")
    print(f"  - Target variable: {'Target' in df_long.columns}")
    print(f"  - CCIAA data: {'CCIAA number adjusted' in df_long.columns}")
    print(f"  - Event data: {'Event' in df_long.columns}")
    print(f"  - Financial variables: {len([col for col in df_long.columns if col.startswith('0')])}")

    # Target distribution
    print(f"\nTarget variable distribution:")
    print(f"  - Target = 0: {(df_long['Target'] == 0).sum():,} ({(df_long['Target'] == 0).mean()*100:.1f}%)")
    print(f"  - Target = 1: {(df_long['Target'] == 1).sum():,} ({(df_long['Target'] == 1).mean()*100:.1f}%)")

else:
    print(f"❌ Error: File not found at {df_final_complete_pkl}")
    print("Please make sure you have run the complete pipeline and saved the data first.")

print("="*60)

In [ ]:
#@title VISUALIZE EMPLOYEE STATS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
# Set a modern plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12

# --- Step 1: Identify the Employee Column ---
# This code dynamically finds the column name that contains "Number of employees".
# This makes the script robust even if the exact column name has prefixes/suffixes.
try:
    employee_col = [col for col in df_long.columns if 'Number of employees' in str(col)][0]
    print(f"✅ Successfully identified employee column: '{employee_col}'")
except IndexError:
    print("❌ ERROR: Could not find a column containing 'Number of employees' in the dataframe.")
    # As a fallback, you can manually set the column name here if needed:
    # employee_col = 'your_column_name_here'
    employee_col = None
except NameError:
    print("❌ ERROR: The dataframe 'df_long' was not found. Please ensure it is loaded in your environment.")
    employee_col = None


if employee_col:
    # --- Step 2: Prepare Data for Overall Distribution Plot ---
    # We calculate the mean number of employees for each unique company.
    # This gives us one value per company, representing its average size over the years.
    print("\nCalculating the average number of employees per company...")

    # Create a copy to avoid modifying the original df_long
    company_mean_employees = df_long[['Tax code number', employee_col]].copy()

    # Drop rows where employee data is missing to ensure accurate means
    company_mean_employees.dropna(subset=[employee_col], inplace=True)

    # Group by company and calculate the mean
    mean_employees_per_company = company_mean_employees.groupby('Tax code number')[employee_col].mean()

    print(f"Calculated average size for {mean_employees_per_company.nunique():,} unique companies.")

    # --- Step 3: Plot 1 - Overall Distribution of Company Sizes ---
    print("\nGenerating Plot 1: Overall distribution of average company sizes...")
    plt.figure(figsize=(14, 7))

    # We cap the range at a reasonable number (e.g., 500) to make the histogram readable,
    # as very large companies can skew the visualization.
    capped_data = mean_employees_per_company[mean_employees_per_company <= 500]

    sns.histplot(capped_data, bins=100, kde=True, color='#3498db', edgecolor='white')

    plt.title('Overall Distribution of Average Company Size', fontsize=16, fontweight='bold')
    plt.xlabel('Average Number of Employees (Capped at 500)', fontsize=12)
    plt.ylabel('Number of Companies', fontsize=12)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

    # Add a vertical line for the median
    median_val = capped_data.median()
    plt.axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median Size: {median_val:.1f}')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # --- Step 4: Prepare Data for Temporal Distribution Plot ---
    print("\nPreparing data for Plot 2: Company size distribution over time...")

    # Create a copy with the necessary columns
    temporal_data = df_long[['Year', 'Tax code number', employee_col]].copy()
    temporal_data.dropna(subset=[employee_col], inplace=True)

    # Define the employee size categories (bins)
    # Bins are defined as: (0, 30], (30, 80], (80, 200], (200, infinity)
    bins = [0, 30, 80, 200, np.inf]
    labels = ['< 30', '31-80', '81-200', '200+']

    temporal_data['size_class'] = pd.cut(temporal_data[employee_col], bins=bins, labels=labels, right=True)

    # Group by year and size class, then count the number of unique companies
    # .unstack() pivots the 'size_class' into columns for plotting
    yearly_distribution = temporal_data.groupby(['Year', 'size_class'], observed=False)['Tax code number'].nunique().unstack(fill_value=0)

    print("Yearly distribution data:")
    print(yearly_distribution)

    # --- Step 5: Plot 2 - Temporal Distribution of Company Sizes ---
    print("\nGenerating Plot 2: Stacked bar chart of company sizes by year...")

    # Use a color palette that is easy to distinguish
    colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

    ax = yearly_distribution.plot(
        kind='bar',
        stacked=True,
        figsize=(16, 8),
        color=colors,
        width=0.8
    )

    plt.title('Distribution of Company Sizes Per Year', fontsize=16, fontweight='bold')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Companies', fontsize=12)
    plt.xticks(rotation=45)
    plt.legend(title='Employee Size Class', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', linewidth=0.5)

    # --- CORRECTED LABELING LOGIC ---
    # This logic correctly calculates the percentage for each segment of the stacked bar.
    totals_by_year = yearly_distribution.sum(axis=1)

    for c in ax.containers:
        labels = []
        for v in c:
            height = v.get_height()
            if height > 0:
                # Get the integer index of the bar (0, 1, 2, ...)
                bar_index = int(round(v.get_x()))
                # Get the corresponding year from the dataframe's index
                year_for_bar = yearly_distribution.index[bar_index]
                # Get the total for that year
                total_for_year = totals_by_year[year_for_bar]

                if total_for_year > 0:
                    percentage = (height / total_for_year) * 100
                    labels.append(f'{percentage:.1f}%')
                else:
                    labels.append('')
            else:
                labels.append('')

        ax.bar_label(c, labels=labels, label_type='center', color='white', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()



In [ ]:
#@title visualize big database
"""
# Show *every* column
pd.set_option('display.max_columns', 1000)

# (Optionally widen your display so lines don’t wrap too crazily)
pd.set_option('display.width', 20)
pd.set_option('display.max_colwidth', 50)

# Now when you do…
#df

#print(final_df.iloc[:, :10])
#df.iloc[:30, :200]
#df.iloc[1000:1030, :200]

df_long.iloc[985:1015, :50]
"""

In [ ]:
#@title PREPARE DATA FOR Target TESTING - OPTIMIZED
import pandas as pd
import numpy as np
from datetime import datetime

print("="*60)
print("PREPARING DATA FOR Target TESTING")
print("="*60)

print(f"Initial shape: {df_long.shape}")
initial_companies = df_long['Tax code number'].nunique()

# ============================================================================
# STEP 1: DELETE OBSERVATIONS AFTER THE TARGET YEAR
# ============================================================================
print("\n[STEP 1] REMOVING OBSERVATIONS AFTER TARGET YEAR...")

# Map each company to its single target year
first_target_year = (
    df_long[df_long['Target'] == 1]
      .groupby('Tax code number')['Year']
      .min()
)

# Attach it and filter out any rows occurring after that year
df_long['First_Target_Year'] = df_long['Tax code number'].map(first_target_year)
rows_before = len(df_long)
df_long = df_long[
    df_long['First_Target_Year'].isna()  # companies with no target keep all rows
    | (df_long['Year'] <= df_long['First_Target_Year'])
].copy()
rows_removed = rows_before - len(df_long)
df_long.drop(columns=['First_Target_Year'], inplace=True)

print(f"  - Rows removed after target year: {rows_removed:,} ({rows_removed/rows_before*100:.1f}%)")
print(f"  - Remaining rows: {len(df_long):,}")

# ============================================================================
# STEP 2: DELETE ALL OBSERVATIONS FROM 2024
# ============================================================================
print("\n[STEP 2] REMOVING ALL 2024 OBSERVATIONS...")
rows_before = len(df_long)
df_long = df_long[df_long['Year'] != 2024].copy()
rows_removed = rows_before - len(df_long)
print(f"  - Rows removed from 2024: {rows_removed:,} ({rows_removed/rows_before*100:.1f}%)")
print(f"  - Remaining rows: {len(df_long):,}")

# ============================================================================
# STEP 3: CONVERT DATE VARIABLES TO MONTH ONLY
# ============================================================================
print("\n[STEP 3] CONVERTING DATE VARIABLES TO MONTH...")
date_columns = [col for col in df_long.columns
                if pd.api.types.is_datetime64_any_dtype(df_long[col])]
print(f"  - Found {len(date_columns)} date columns to convert")
for col in date_columns:
    df_long[f'{col}_month'] = df_long[col].dt.month
    df_long.drop(columns=[col], inplace=True)
    print(f"    * Converted {col} → {col}_month")

# ============================================================================
# STEP 4: CHECK COMPANIES WITH ALL NaN FINANCIAL VARIABLES
# ============================================================================
print("\n[STEP 4] CHECKING COMPANIES WITH ALL NaN FINANCIAL VARIABLES...")
whitelist_extended = [
    'Tax code number',
    'Registered office address - Postal code',
    'Registered office address - Longitude',
    'Registered office address - Latitude',
    'Trading address - Postal code',
    'Incorporation year',
    'Legal form',
    'Year',
    'Year from incorporation',
    'Event',
    'Target',
    'CCIAA number adjusted'
]
# include any month columns
whitelist_extended += [c for c in df_long.columns if c.endswith('_month')]

financial_cols = [c for c in df_long.columns if c not in whitelist_extended]
print(f"  - Total columns: {len(df_long.columns)}")
print(f"  - Whitelist columns: {len(whitelist_extended)}")
print(f"  - Financial columns: {len(financial_cols)}")

companies_all_nan = df_long.groupby('Tax code number')[financial_cols] \
    .apply(lambda x: x.isna().all().all())
companies_with_all_nan = companies_all_nan[companies_all_nan].index.tolist()
print(f"\n  - Companies with ALL financial variables NaN: {len(companies_with_all_nan):,}")
print(f"  - Percentage: {len(companies_with_all_nan)/initial_companies*100:.1f}%")

rows_with_all_nan = df_long[df_long['Tax code number'].isin(companies_with_all_nan)].shape[0]
print(f"  - Total rows for these companies: {rows_with_all_nan:,}")

# ============================================================================
# STEP 5: FINAL SUMMARY
# ============================================================================
print("\n[STEP 5] FINAL SUMMARY...")
print(f"\nDataset transformation summary:")
print(f"  - Initial companies: {initial_companies:,}")
print(f"  - Final shape: {df_long.shape}")
print(f"  - Companies remaining: {df_long['Tax code number'].nunique():,}")
print(f"  - Years covered: {df_long['Year'].min()} - {df_long['Year'].max()}")

print(f"\nTarget distribution after cleaning:")
print(f"  - Target = 0: {(df_long['Target']==0).sum():,} ({(df_long['Target']==0).mean()*100:.1f}%)")
print(f"  - Target = 1: {(df_long['Target']==1).sum():,} ({(df_long['Target']==1).mean()*100:.1f}%)")

print(f"\nObservations by year:")
year_counts = df_long['Year'].value_counts().sort_index()
for year, count in year_counts.items():
    tcount = df_long[df_long['Year']==year]['Target'].sum()
    print(f"  - {year}: {count:,} obs ({tcount} with Target=1)")

print("\n" + "="*60)
print("DATA PREPARATION COMPLETED!")
print("="*60)

# Optional prompt for removing all-NaN companies
print(f"\n❓ To drop companies with all-NaN financials: remove {len(companies_with_all_nan):,} companies ({rows_with_all_nan:,} rows).")


In [ ]:
#@title Diagnostic: Identify Potential Categorical Variables

import pandas as pd
import numpy as np

print("="*80)
print("DIAGNOSTIC: IDENTIFYING POTENTIAL CATEGORICAL VARIABLES")
print("="*80)

# First, let's see all columns and their basic info
print("\n1. DATASET OVERVIEW:")
print(f"Total columns: {len(df_long.columns)}")
print(f"Total rows: {len(df_long):,}")

# Analyze each column for discrete patterns
discrete_candidates = {}

print("\n2. ANALYZING COLUMNS FOR DISCRETE PATTERNS:")
print("-"*80)

for col in df_long.columns:
    # Skip known identifiers and target
    if col in ['Year', 'Tax code number', 'Target', 'CCIAA number adjusted']:
        continue

    # Get column info
    dtype = df_long[col].dtype
    unique_count = df_long[col].nunique()
    total_count = df_long[col].count()  # Non-null count
    null_count = df_long[col].isnull().sum()

    # For numeric columns, check if they might be categorical
    if dtype in ['int64', 'float64']:
        unique_ratio = unique_count / total_count if total_count > 0 else 0

        # Criteria for potential categorical:
        # - Less than 20 unique values OR
        # - Less than 1% unique ratio (many repeated values)
        if unique_count <= 20 or unique_ratio < 0.01:
            # Get value counts
            value_counts = df_long[col].value_counts().head(10)

            # Check if values are "round" numbers (0, 1, 2, etc.)
            unique_values = df_long[col].dropna().unique()
            try:
                are_integers = all(v == int(v) for v in unique_values if not pd.isna(v))
            except:
                are_integers = False

            discrete_candidates[col] = {
                'unique_count': unique_count,
                'unique_ratio': unique_ratio,
                'null_count': null_count,
                'null_percent': (null_count / len(df_long)) * 100,
                'are_integers': are_integers,
                'top_values': value_counts.head(5).to_dict(),
                'sample_values': sorted(unique_values)[:10]  # First 10 sorted values
            }

# Sort by number of unique values
sorted_candidates = sorted(discrete_candidates.items(),
                         key=lambda x: x[1]['unique_count'])

print("\n3. POTENTIAL CATEGORICAL VARIABLES:")
print("(Sorted by number of unique values)")
print("-"*80)

for col, info in sorted_candidates:
    print(f"\n📊 {col}")
    print(f"   Unique values: {info['unique_count']}")
    print(f"   Unique ratio: {info['unique_ratio']:.2%}")
    print(f"   Missing: {info['null_percent']:.1f}%")
    print(f"   Integer values: {'Yes' if info['are_integers'] else 'No'}")
    print(f"   Sample values: {info['sample_values']}")
    print(f"   Top 5 frequencies:")
    for val, count in list(info['top_values'].items())[:5]:
        print(f"      {val}: {count:,} ({count/len(df_long)*100:.2f}%)")

# Special focus on very low unique count variables
print("\n4. HIGH PRIORITY CANDIDATES (≤ 10 unique values):")
print("-"*80)

high_priority = [(col, info) for col, info in sorted_candidates
                 if info['unique_count'] <= 10]

if high_priority:
    for col, info in high_priority:
        print(f"\n🎯 {col}")
        print(f"   All values: {info['sample_values']}")
        print(f"   Likely interpretation: ", end="")

        # Try to guess what it might be
        if info['unique_count'] == 2 and info['are_integers']:
            if set(info['sample_values']) <= {0, 1}:
                print("Binary indicator (Yes/No, Active/Inactive, etc.)")
            else:
                print("Binary category")
        elif info['unique_count'] <= 5 and info['are_integers']:
            print("Ordinal category (rating, level, class)")
        elif info['unique_count'] <= 10:
            print("Nominal category (type, group, segment)")
        else:
            print("Multi-category variable")
else:
    print("No variables with ≤ 10 unique values found")

# Check for variables that might be codes or IDs
print("\n5. CHECKING FOR CODED VARIABLES:")
print("-"*80)

code_patterns = []
for col in df_long.columns:
    if col in ['Year', 'Tax code number', 'Target', 'CCIAA number adjusted']:
        continue

    # Check if column name suggests it might be a code
    if any(pattern in col.lower() for pattern in ['code', 'id', 'number', 'type', 'class', 'category', 'status']):
        dtype = df_long[col].dtype
        unique_count = df_long[col].nunique()
        sample = df_long[col].dropna().head(5).tolist()

        code_patterns.append({
            'column': col,
            'dtype': str(dtype),
            'unique_count': unique_count,
            'sample': sample
        })

if code_patterns:
    for pattern in code_patterns:
        print(f"\n📌 {pattern['column']}")
        print(f"   Type: {pattern['dtype']}")
        print(f"   Unique values: {pattern['unique_count']}")
        print(f"   Sample: {pattern['sample']}")
else:
    print("No obvious coded variables found based on column names")

# Summary
print("\n" + "="*80)
print("SUMMARY:")
print(f"Found {len(discrete_candidates)} potential categorical variables")
print(f"High priority (≤10 unique): {len(high_priority)}")
print(f"Column name suggests code/category: {len(code_patterns)}")
print("\nNext step: Review these variables and decide which should be treated as categorical")
print("="*80)

In [ ]:
#@title Data Type Check and Encoding for df_long (Corrected)

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("="*60)
print("CHECKING AND ENCODING NON-NUMERIC VARIABLES")
print("="*60)

# Check initial data types
print("\n1. INITIAL DATA TYPE SUMMARY:")
print(df_long.dtypes.value_counts())

# Identify truly non-numeric columns (object or category types only)
non_numeric_cols = df_long.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = df_long.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

print(f"\n2. COLUMN CLASSIFICATION:")
print(f"   - Total columns: {len(df_long.columns)}")
print(f"   - Numeric columns: {len(numeric_cols)}")
print(f"   - Non-numeric columns: {len(non_numeric_cols)}")

# Drop Event_Bankruptcy (all zeros - no variance)
if 'Event_Bankruptcy' in df_long.columns:
    print("\n3. DROPPING ZERO-VARIANCE COLUMNS:")
    print("   - Dropping Event_Bankruptcy (all values are 0)")
    df_long = df_long.drop('Event_Bankruptcy', axis=1)

# Drop postal codes to avoid data leakage and reduce noise
print("\n4. DROPPING POSTAL CODE COLUMNS:")
postal_columns_to_drop = [
    'Trading address - Postal code',
    'Registered office address - Postal code'
]
for col in postal_columns_to_drop:
    if col in df_long.columns:
        print(f"   - Dropping '{col}' to avoid potential data leakage")
        df_long = df_long.drop(col, axis=1)
    else:
        print(f"   - '{col}' not found in columns")
print(f"   Shape after dropping postal codes: {df_long.shape}")

# Check if columns that look like they should be numeric are stored as objects
print("\n5. CHECKING FOR NUMERIC COLUMNS STORED AS OBJECTS:")
potentially_numeric = []
for col in non_numeric_cols:
    if col not in ['Year', 'Tax code number', 'Target', 'Legal form', 'CCIAA number adjusted'] + postal_columns_to_drop:
        # Check if column name suggests it's numeric (contains %, EUR, ratio, etc.)
        if any(indicator in col for indicator in ['%', 'EUR', 'ratio', 'turnover', 'ROA', 'ROE', 'ROS', 'EBITDA',
                                                  'Leverage', 'days', 'times', 'Costs']):
            potentially_numeric.append(col)
            print(f"   • {col} - might be numeric stored as text")

# Create a copy to work with
df_long_encoded = df_long.copy()

# First, try to convert potentially numeric columns to numeric
print("\n6. CONVERTING POTENTIALLY NUMERIC COLUMNS:")
converted_cols = []
failed_conversions = []

for col in potentially_numeric:
    try:
        # For these specific columns that failed, they might have commas as decimal separators
        if col in ['0314_Leverage\n', '0326_Stocks/Turnover (days)\n', '0327_Stocks/Cost goods sold (days)\n',
                   '0328_Durata media dei crediti al lordo IVA (days)\n', '0329_Durata media dei debiti al lordo IVA (days)\n',
                   '0330_Durata Ciclo Commerciale (days)\n', '0340_Turnover/Staff Costs\n']:
            # Try replacing comma with dot if that's the issue
            df_long_encoded[col] = df_long_encoded[col].astype(str).str.replace(',', '.', regex=False)

        # Try to convert to numeric, coercing errors to NaN
        df_long_encoded[col] = pd.to_numeric(df_long_encoded[col], errors='coerce')

        if df_long_encoded[col].notna().any():  # If at least some values converted successfully
            converted_cols.append(col)
            print(f"   ✓ {col} - successfully converted to numeric")
        else:
            print(f"   ✗ {col} - conversion failed, will try alternative methods")
            failed_conversions.append(col)
            # Revert to original for now
            df_long_encoded[col] = df_long[col]
    except Exception as e:
        print(f"   ✗ {col} - conversion failed with error: {str(e)}")
        failed_conversions.append(col)

# For failed conversions, try more aggressive cleaning
print("\n7. ATTEMPTING AGGRESSIVE CONVERSION FOR FAILED COLUMNS:")
for col in failed_conversions:
    try:
        # Get the original data
        original_data = df_long[col].copy()

        # Convert to string and clean
        cleaned_data = original_data.astype(str)

        # Remove any whitespace
        cleaned_data = cleaned_data.str.strip()

        # Replace common European number formats
        cleaned_data = cleaned_data.str.replace(',', '.', regex=False)  # European decimal
        cleaned_data = cleaned_data.str.replace(' ', '', regex=False)   # Remove spaces

        # Try conversion again
        df_long_encoded[col] = pd.to_numeric(cleaned_data, errors='coerce')

        # Check if conversion was successful
        success_rate = df_long_encoded[col].notna().sum() / len(df_long_encoded[col])

        if success_rate > 0.1:  # If at least 10% converted successfully
            print(f"   ✓ {col} - converted with {success_rate:.1%} success rate")
            converted_cols.append(col)
        else:
            print(f"   ✗ {col} - still failing, reverting to original")
            df_long_encoded[col] = original_data
    except:
        print(f"   ✗ {col} - aggressive conversion failed")
        df_long_encoded[col] = df_long[col]

# Handle CCIAA - Just drop it without creating province dummies
print("\n8. HANDLING CCIAA NUMBER:")
if 'CCIAA number adjusted' in df_long_encoded.columns:
    print("   - Dropping CCIAA number adjusted (not creating province dummies to reduce noise)")
    df_long_encoded = df_long_encoded.drop('CCIAA number adjusted', axis=1)
else:
    print("   - CCIAA number adjusted not found in columns")

# Update the list of non-numeric columns after conversion
non_numeric_cols = df_long_encoded.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\n9. REMAINING NON-NUMERIC COLUMNS AFTER CONVERSION:")
for col in non_numeric_cols:
    unique_count = df_long_encoded[col].nunique()
    sample_values = df_long_encoded[col].dropna().unique()[:5]
    print(f"   • {col}:")
    print(f"     - Unique values: {unique_count}")
    print(f"     - Sample values: {sample_values}")

# Strategy for encoding remaining non-numeric columns
print("\n10. ENCODING STRATEGY FOR REMAINING NON-NUMERIC COLUMNS:")

# Lists to track what we do with each column
label_encoded_cols = []
one_hot_encoded_cols = []
kept_as_is_cols = []
dropped_cols = []

for col in non_numeric_cols:
    unique_count = df_long_encoded[col].nunique()

    # Skip if column is 'Year' or 'Tax code number' (identifiers)
    if col in ['Year', 'Tax code number']:
        print(f"   - {col}: Keeping as identifier (will be excluded from features)")
        kept_as_is_cols.append(col)
        continue

    # If it's the target variable, skip
    if col == 'Target':
        print(f"   - {col}: Target variable, keeping as is")
        kept_as_is_cols.append(col)
        continue

    # For Legal form - use one-hot encoding
    if col == 'Legal form' and unique_count <= 50:
        print(f"   - {col}: One-hot encoding ({unique_count} unique values)")

        # Create dummy variables
        dummies = pd.get_dummies(df_long_encoded[col], prefix=col, dummy_na=False)

        # Convert boolean to int immediately
        dummies = dummies.astype(int)

        # Add to dataframe
        df_long_encoded = pd.concat([df_long_encoded, dummies], axis=1)

        # Drop original column
        df_long_encoded = df_long_encoded.drop(col, axis=1)
        one_hot_encoded_cols.append(col)

    # For any remaining object columns, we need to handle them
    else:
        print(f"   - {col}: High cardinality ({unique_count} unique values)")

        # Let's try one more time to convert to numeric
        temp_col = df_long_encoded[col].copy()

        try:
            # Convert to string first
            temp_col = temp_col.astype(str)
            # Remove any non-numeric characters except . and -
            temp_col = temp_col.str.extract(r'([-+]?\d*\.?\d+)', expand=False)
            # Convert to numeric
            temp_numeric = pd.to_numeric(temp_col, errors='coerce')

            if temp_numeric.notna().sum() > len(temp_numeric) * 0.1:  # If >10% successful
                df_long_encoded[col] = temp_numeric
                print(f"     → Forced numeric conversion successful")
            else:
                # If still failing, drop the column as last resort
                print(f"     → WARNING: Cannot convert to numeric, dropping column")
                df_long_encoded = df_long_encoded.drop(col, axis=1)
                dropped_cols.append(col)
        except:
            print(f"     → ERROR: Cannot process, dropping column")
            df_long_encoded = df_long_encoded.drop(col, axis=1)
            dropped_cols.append(col)

# Convert any remaining boolean columns to int
print("\n11. CONVERTING BOOLEAN COLUMNS TO INT:")
bool_columns = df_long_encoded.select_dtypes(include=['bool']).columns
if len(bool_columns) > 0:
    print(f"   - Found {len(bool_columns)} boolean columns from one-hot encoding")
    df_long_encoded[bool_columns] = df_long_encoded[bool_columns].astype(int)
    print("   - Converted all boolean columns to int")
else:
    print("   - No boolean columns found")

# Final results
print("\n12. ENCODING RESULTS:")
print(f"   - Columns converted to numeric: {len(converted_cols)}")
if converted_cols:
    print(f"     First 5: {converted_cols[:5]}")
print(f"   - Columns one-hot encoded: {len(one_hot_encoded_cols)}")
if one_hot_encoded_cols:
    print(f"     {one_hot_encoded_cols}")
print(f"   - Columns label encoded: {len(label_encoded_cols)}")
if label_encoded_cols:
    print(f"     {label_encoded_cols}")
print(f"   - Columns kept as is: {len(kept_as_is_cols)}")
if kept_as_is_cols:
    print(f"     {kept_as_is_cols}")
print(f"   - Columns dropped: {len(dropped_cols)}")
if dropped_cols:
    print(f"     {dropped_cols}")

# Shape comparison
print(f"\n13. SHAPE COMPARISON:")
print(f"   - Original shape: {df_long.shape}")
print(f"   - After encoding: {df_long_encoded.shape}")
print(f"   - Columns added: {df_long_encoded.shape[1] - df_long.shape[1]}")

# Replace the original dataframe
df_long = df_long_encoded

# Final verification
print("\n14. FINAL VERIFICATION:")
final_dtypes = df_long.dtypes.value_counts()
print("Data types after processing:")
print(final_dtypes)

# Verify no boolean types remain
if 'bool' in final_dtypes.index:
    print("\n   ⚠️ WARNING: Boolean columns still present! This will cause issues.")
else:
    print("\n   ✅ No boolean columns - all properly converted!")

# List any remaining object columns that aren't expected
remaining_objects = df_long.select_dtypes(include=['object']).columns.tolist()
expected_objects = ['Year', 'Tax code number', 'Target']
remaining_objects = [col for col in remaining_objects if col not in expected_objects]

if not remaining_objects:
    print("   ✅ All unexpected object columns successfully handled!")
    print(f"   ℹ️  Note: {expected_objects} remain as object type but will be excluded from features")
else:
    print(f"\n   ⚠️ WARNING: {len(remaining_objects)} unexpected object columns remain:")
    for col in remaining_objects:
        print(f"      - {col}")
    print("\n   These columns will cause errors in XGBoost/LightGBM!")

# Memory cleanup
import gc
gc.collect()

print("\n" + "="*60)
print("ENCODING COMPLETED - df_long UPDATED")
print("="*60)

In [ ]:
#@title 💾 Save Encoded Dataset (Pickle to path_MODEL)
import pandas as pd
import pickle
import os
from datetime import datetime

# ------------------------------------------------------------------
# 1) Define / re-use a single location for all model artefacts
# ------------------------------------------------------------------

os.makedirs(path_MODEL, exist_ok=True)

# ------------------------------------------------------------------
# 2) Usual save configuration
# ------------------------------------------------------------------
CUSTOM_FILENAME = 'italian_bankruptcy_encoded'   # ↩️ keep your own name
ADD_TIMESTAMP   = True                           # ↩️ keep timestamp?
SAVE_PATH       = path_MODEL                     # ✅ now points to path_MODEL

# ------------------------------------------------------------------
# 3) Build filename + save
# ------------------------------------------------------------------
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S') if ADD_TIMESTAMP else ''
filename  = f"{CUSTOM_FILENAME}{'_' + timestamp if timestamp else ''}.pkl"
filepath  = os.path.join(SAVE_PATH, filename)

print("💾 SAVING ENCODED DATASET")
print("=" * 60)

try:
    df_long.to_pickle(filepath)

    sz_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"✅ Saved successfully!")
    print(f"   • Filename : {filename}")
    print(f"   • Full path: {os.path.abspath(filepath)}")
    print(f"   • File size: {sz_mb:.2f} MB")
    print(f"   • Shape    : {df_long.shape}")
    print(f"   • Columns  : {len(df_long.columns)}")
    print(f"   • Mem usage: {df_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

except Exception as e:
    print(f"❌ Error saving file: {e}")

print("=" * 60)


## MODEL - application

---



In [ ]:
#@title LOAD Most Recent Dataset (from path_MODEL)
import pandas as pd
import os
from glob import glob

# ------------------------------------------------------------------
# Use the same folder defined in the save cell
# ------------------------------------------------------------------

pattern    = os.path.join(path_MODEL, '*bankruptcy_encoded*.pkl')

pkl_files = glob(pattern)

if pkl_files:
    latest_file = max(pkl_files, key=os.path.getctime)
    df_long = pd.read_pickle(latest_file)

    print(f"✅ Loaded: {os.path.basename(latest_file)}")
    print(f"   Shape: {df_long.shape}")
    print("   Data types:")
    for dtype, count in df_long.dtypes.value_counts().items():
        print(f"      • {dtype}: {count}")

    if 'Trading address - Postal code' in df_long.columns:
        print("\n⚠️  Warning: Dataset contains 'Trading address - Postal code' – consider dropping it")
else:
    print("❌ No saved bankruptcy dataset found in", path_MODEL)


### model autosampling healthy


In [ ]:
#@title XGBoost & LightGBM Model Training (Optimized with Sampling)

#================================================================================
# CONFIGURATION PARAMETERS (MODIFY HERE)
#================================================================================
# Define temporal splits
train_years = [2016, 2017, 2018]
valid_years = [2019]
test_years = [2020, 2021]

# Sampling Configuration
SAMPLING_RATIOS = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]  # Fraction of healthy companies to keep

# Optimization Configuration
OPTIMIZATION_CONFIG = {
    'metric': 'roc_auc',
        # Options: 'roc_auc', 'pr_auc', 'partial_pr_auc', 'precision_at_recall'
    'partial_range': [0.90, 0.95],  # For partial_pr_auc (recall range)
    'target_recall': 0.90,          # For precision_at_recall
}

# Other Optimization Settings
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 10

# GPU Configuration
USE_GPU = True
GPU_DEVICE_ID = 0

# Class Weight Configuration
USE_SCALE_POS_WEIGHT = True  # Will be calculated for each sampling ratio

# Optuna Configuration
MODEL_TRIALS = {
    'xgboost':  700,    # Number of trials for XGBoost
    'lightgbm': 700     # Number of trials for LightGBM
}
OPTUNA_TIMEOUT = {
    'xgboost':  12000,  # seconds per model
    'lightgbm': 12000   # seconds per model
}

# Pruning Configuration
USE_PRUNING = True
PRUNING_N_STARTUP_TRIALS = 5
PRUNING_N_WARMUP_STEPS = 5
PRUNING_INTERVAL_STEPS = 1

# Hyperparameter Search Ranges
PARAM_RANGES = {
    'xgboost': {
        'max_depth':        (3    , 25  ),
        'learning_rate':    (0.01 , 0.4 ),
        'n_estimators':     (200  , 1500),
        'subsample':        (0.3  , 1   ),
        'colsample_bytree': (0.3  , 1   ),
        'reg_alpha':        (0.01 , 15.0),
        'reg_lambda':       (0.01 , 15.0),
        'min_child_weight': (1    , 100  )
    },
    'lightgbm': {
        'num_leaves':       (2    , 600 ),
        'learning_rate':    (0.005, 0.17),
        'n_estimators':     (30   , 800 ),
        'feature_fraction': (0.3  , 1   ),
        'bagging_fraction': (0.3  , 1   ),
        'bagging_freq':     (20   , 150 ),
        'reg_alpha':        (0.02 , 20.0),
        'reg_lambda':       (0.02 , 20.0),
        'min_child_weight': (1   , 150 )
    }
}
"""
PARAM_RANGES = {
    'xgboost': {
        'max_depth': (3, 25),
        'learning_rate': (0.01, 0.4),
        'n_estimators': (300, 800),
        'subsample': (0.3, 1),
        'colsample_bytree': (0.3, 1),
        'reg_alpha': (0.04, 10.0),
        'reg_lambda': (0.05, 10.0),
        'min_child_weight': (1, 60)
    },
    'lightgbm': {
        'num_leaves': (5, 600),
        'learning_rate': (0.005, 0.17),
        'n_estimators': (30, 800),
        'feature_fraction': (0.3, 1),
        'bagging_fraction': (0.3, 1),
        'bagging_freq': (20, 100),
        'reg_alpha': (0.05, 15.0),
        'reg_lambda': (0.03, 10.0),
        'min_child_weight': (10, 120)
    }
}
"""
# Business Metric Configuration
BANKRUPTCY_RECALL_TARGETS = [0.80, 0.90, 0.95, 0.99]  # Recall targets for business metrics

# Trial Logging Configuration
LOG_FAILED_TRIALS = True  # Log details of failed trials

#================================================================================
# IMPORTS AND SETUP
#================================================================================

import subprocess
import gc
import psutil
import os
import time
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    accuracy_score,
    classification_report,
    auc
)
from IPython.display import clear_output
import warnings
import traceback

# Suppress warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# Store failed trials for debugging
failed_trials = {'XGBoost': [], 'LightGBM': []}

#================================================================================
# CUSTOM EVALUATION METRICS
#================================================================================

def partial_pr_auc_score(y_true, y_pred_proba, recall_range):
    """Calculate partial PR-AUC for a specific recall range"""
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

        # Check if we have enough points
        if len(precision) < 2:
            return 0.0

        # Reverse arrays to make recall increasing
        precision = precision[::-1]
        recall = recall[::-1]

        # Find indices for recall range
        idx_start = np.searchsorted(recall, recall_range[0])
        idx_end = np.searchsorted(recall, recall_range[1])

        # Ensure we have at least 2 points
        if idx_end - idx_start < 2:
            # If not enough points in the range, return 0
            return 0.0

        # Calculate partial AUC
        partial_auc = auc(recall[idx_start:idx_end], precision[idx_start:idx_end])

        # Normalize by the width of the recall range
        normalized_auc = partial_auc / (recall_range[1] - recall_range[0])

        return normalized_auc

    except Exception as e:
        # If any error occurs (e.g., all predictions are the same), return 0
        print(f"Warning in partial_pr_auc_score: {e}")
        return 0.0


def precision_at_recall_score(y_true, y_pred_proba, target_recall):
    """
    Calculate precision at the highest-precision threshold whose recall is ≥ target_recall.
    Returns 0.0 if the target recall cannot be reached.
    """
    try:
        # Check if we have any positive samples
        if np.sum(y_true) == 0:
            return 0.0

        # Check if predictions have any variation
        if len(np.unique(y_pred_proba)) < 2:
            # All predictions are the same
            if target_recall == 0:
                return 1.0 if np.all(y_pred_proba < 0.5) else 0.0
            else:
                # If we predict all as negative and need recall > 0, precision is 0
                # If we predict all as positive, precision equals positive rate
                if np.all(y_pred_proba >= 0.5):
                    return np.mean(y_true)
                else:
                    return 0.0

        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

        # Find all indices where the recall is at least the target
        valid = np.where(recall >= target_recall)[0]

        if len(valid) == 0:
            # The target recall level was never reached
            return 0.0

        # Of the valid points, take the one with the highest precision
        idx = valid[-1]
        return precision[idx]

    except Exception as e:
        print(f"Warning in precision_at_recall_score: {e}")
        return 0.0

def create_eval_metric_function(metric_config):
    """Create the appropriate evaluation metric function based on config"""
    metric_type = metric_config['metric']

    if metric_type == 'roc_auc':
        def eval_metric(y_true, y_pred):
            return 'roc_auc', roc_auc_score(y_true, y_pred)
        native_metric = 'auc'

    elif metric_type == 'pr_auc':
        def eval_metric(y_true, y_pred):
            return 'pr_auc', average_precision_score(y_true, y_pred)
        native_metric = 'aucpr'

    elif metric_type == 'partial_pr_auc':
        recall_range = metric_config['partial_range']
        def eval_metric(y_true, y_pred):
            return f'partial_pr_auc_{recall_range[0]}-{recall_range[1]}', partial_pr_auc_score(y_true, y_pred, recall_range)
        native_metric = 'aucpr'  # Use PR-AUC as proxy for early stopping

    elif metric_type == 'precision_at_recall':
        target_recall = metric_config['target_recall']
        def eval_metric(y_true, y_pred):
            return f'precision@{target_recall}', precision_at_recall_score(y_true, y_pred, target_recall)
        native_metric = 'aucpr'  # Use PR-AUC as proxy for early stopping

    else:
        raise ValueError(f"Unknown metric type: {metric_type}")

    return eval_metric, native_metric

# Create evaluation metric function
eval_metric_func, native_metric = create_eval_metric_function(OPTIMIZATION_CONFIG)

#================================================================================
# GPU DETECTION AND SETUP
#================================================================================

# GPU Detection
GPU_PARAMS = None
if USE_GPU:
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                               '--format=csv,noheader,nounits'],
                               capture_output=True, text=True)
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(', ')
            gpu_name = gpu_info[0]
            gpu_memory = f"{int(gpu_info[1])/1024:.1f} GB"

            print(f"GPU detected: {gpu_name} ({gpu_memory})")

            # Set GPU parameters based on GPU type
            if 'A100' in gpu_name:
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID,
                        'sampling_method': 'gradient_based',
                        'max_bin': 256
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False,
                        'max_bin': 255
                    }
                }
            elif 'T4' in gpu_name:
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID,
                        'sampling_method': 'uniform',
                        'max_bin': 64
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False,
                        'max_bin': 63
                    }
                }
            else:
                # Generic GPU settings
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False
                    }
                }
        else:
            print("GPU not available, using CPU")
            USE_GPU = False
    except:
        print("GPU detection failed, using CPU")
        USE_GPU = False

#================================================================================
# HELPER FUNCTIONS
#================================================================================

def get_memory_usage():
    """Get current memory usage in GB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024 / 1024

def clean_memory():
    """Force garbage collection"""
    gc.collect()

def get_metric_display_name(config):
    """Get a display name for the current metric configuration"""
    if config['metric'] == 'partial_pr_auc':
        return f"Partial PR-AUC [{config['partial_range'][0]}-{config['partial_range'][1]}]"
    elif config['metric'] == 'precision_at_recall':
        return f"Precision@{config['target_recall']}Recall"
    else:
        return config['metric'].upper().replace('_', '-')

#================================================================================
# ENHANCED PROGRESS TRACKING
#================================================================================

class ModelTracker:
    def __init__(self, total_models=2):
        self.total_models = total_models
        self.current_model = 0
        self.results_summary = []
        self.start_time = time.time()
        self.trial_stats = {}
        self.model_start_times = {}
        self.trial_details = {'XGBoost': [], 'LightGBM': []}
        self.metric_name = get_metric_display_name(OPTIMIZATION_CONFIG)
        self.best_sampling_ratios = {}

    def update(self, model_name, status, metrics=None, trial=None, total_trials=None, study=None):
        self.current_time = time.time() - self.start_time

        if model_name not in self.model_start_times:
            self.model_start_times[model_name] = time.time()

        clear_output(wait=True)

        # Header
        print("XGBOOST & LIGHTGBM BANKRUPTCY PREDICTION MODEL TRAINING")
        print("=" * 80)
        print(f"Total Elapsed Time: {self.format_time(self.current_time)}")
        print(f"Optimization Metric: {self.metric_name}")
        print(f"Sampling Ratios: {SAMPLING_RATIOS}")
        print(f"GPU: {'Enabled' if USE_GPU and GPU_PARAMS else 'Disabled'}")
        print(f"Memory Usage: {get_memory_usage():.2f} GB")
        print("=" * 80)

        # Current model status
        model_time = time.time() - self.model_start_times[model_name]
        print(f"\nCurrent Model: {model_name}")
        print(f"Status: {status}")
        print(f"Model Time: {self.format_time(model_time)}")

        # Optuna progress with detailed trial info
        if trial is not None and total_trials is not None:
            actual_trial_num = trial + 1
            progress = actual_trial_num / total_trials
            bar_length = 50
            filled_length = int(bar_length * progress)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)

            trial_info = f"{actual_trial_num}/{total_trials} trials"

            if study is not None:
                completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
                pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
                failed = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

                self.trial_stats[model_name] = {
                    'completed': completed,
                    'pruned': pruned,
                    'failed': failed,
                    'total_attempted': len(study.trials)
                }

                if completed > 0:
                    trial_info += f" ({completed} completed"
                    if pruned > 0:
                        trial_info += f", {pruned} pruned"
                    if failed > 0:
                        trial_info += f", {failed} failed"
                    trial_info += ")"

                    # Show best score with appropriate metric name
                    try:
                        if study.best_trial is not None:
                            trial_info += f" | Best: {study.best_value:.4f}"
                            # Show best sampling ratio if available
                            if 'sampling_ratio' in study.best_params:
                                trial_info += f" (sampling={study.best_params['sampling_ratio']})"
                    except:
                        pass

            print(f"Optimization: [{bar}] {trial_info}")

        # Results summary
        if self.results_summary:
            print("\nCOMPLETED MODELS:")
            print("-" * 80)
            for result in self.results_summary:
                print(f"{result['model']}: {result['metric_name']} = {result['metric_value']} (sampling={result.get('sampling_ratio', 'N/A')})")

    def format_time(self, seconds):
        if seconds < 60:
            return f"{seconds:.1f}s"
        elif seconds < 3600:
            return f"{seconds/60:.1f}m"
        else:
            hours = seconds // 3600
            minutes = (seconds % 3600) // 60
            return f"{hours:.0f}h {minutes:.0f}m"

    def complete_model(self, model_name, train_time, metric_value, sampling_ratio):
        self.current_model += 1
        self.results_summary.append({
            'model': model_name,
            'time': f"{train_time:.1f}s",
            'metric_name': self.metric_name,
            'metric_value': f"{metric_value:.4f}",
            'sampling_ratio': sampling_ratio
        })
        self.best_sampling_ratios[model_name] = sampling_ratio

#================================================================================
# DATA VERIFICATION
#================================================================================

tracker = ModelTracker(total_models=2)
tracker.update("Data Verification", "Checking if df_long exists...")

# Check if df_long exists first
if 'df_long' not in globals():
    raise ValueError("df_long not found! Please ensure the data is loaded before running this cell.")

print(f"\n✓ df_long found with shape: {df_long.shape}")
time.sleep(1)

#================================================================================
# DATA PREPARATION AND TEMPORAL SPLITTING
#================================================================================

tracker.update("Data Preparation", "Preparing temporal splits...")

import re

# Function to sanitize column names
def sanitize_column_names(df):
    """
    Replace special characters in column names that cause issues with XGBoost/LightGBM
    """
    # Characters that cause issues: []<>{},%
    df.columns = df.columns.str.replace('[', '_', regex=False)
    df.columns = df.columns.str.replace(']', '_', regex=False)
    df.columns = df.columns.str.replace('<', '_', regex=False)
    df.columns = df.columns.str.replace('>', '_', regex=False)
    df.columns = df.columns.str.replace('{', '_', regex=False)
    df.columns = df.columns.str.replace('}', '_', regex=False)
    df.columns = df.columns.str.replace(',', '_', regex=False)
    df.columns = df.columns.str.replace('%', 'pct', regex=False)
    df.columns = df.columns.str.replace('/', '_', regex=False)
    df.columns = df.columns.str.replace('\\', '_', regex=False)
    df.columns = df.columns.str.replace(':', '_', regex=False)
    df.columns = df.columns.str.replace('\n', '', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)

    # Remove any remaining special characters
    df.columns = [re.sub(r'[^\w]', '_', col) for col in df.columns]

    # Clean up multiple underscores
    df.columns = [re.sub(r'_+', '_', col) for col in df.columns]

    # Remove leading/trailing underscores
    df.columns = [col.strip('_') for col in df.columns]

    return df

# Check initial data
print(f"Total samples in df_long: {len(df_long):,}")
print(f"Years available: {sorted(df_long['Year'].unique())}")

# Temporal split
df_train = df_long[df_long['Year'].isin(train_years)].copy()
df_valid = df_long[df_long['Year'].isin(valid_years)].copy()
df_test = df_long[df_long['Year'].isin(test_years)].copy()

print(f"\nTemporal splits:")
print(f"Training set ({train_years}): {len(df_train):,} samples")
print(f"Validation set ({valid_years}): {len(df_valid):,} samples")
print(f"Test set ({test_years}): {len(df_test):,} samples")

# Sanitize column names
df_train = sanitize_column_names(df_train)
df_valid = sanitize_column_names(df_valid)
df_test = sanitize_column_names(df_test)
print("\n✓ Column names sanitized for XGBoost/LightGBM compatibility")

# Drop problematic columns
columns_to_drop = []

# Check for postal code columns
postal_columns = [col for col in df_train.columns if 'Postal_code' in col or 'postal_code' in col]
if postal_columns:
    columns_to_drop.extend(postal_columns)

# Drop CCIAA-derived province dummies
province_dummies = [col for col in df_train.columns if col.startswith('Province_')]
if province_dummies:
    print(f"Found {len(province_dummies)} province dummy columns from CCIAA - will be dropped")
    columns_to_drop.extend(province_dummies)

# Drop all identified columns
if columns_to_drop:
    df_train = df_train.drop(columns=columns_to_drop, errors='ignore')
    df_valid = df_valid.drop(columns=columns_to_drop, errors='ignore')
    df_test = df_test.drop(columns=columns_to_drop, errors='ignore')
    print(f"✓ Dropped {len(columns_to_drop)} columns total")

# Define columns to exclude from features
exclude_cols = ['Year', 'Tax_code_number', 'Target']

# Get feature columns
feature_cols = [col for col in df_train.columns if col not in exclude_cols]
feature_names = feature_cols

# Create feature matrices and target vectors for ORIGINAL data (before sampling)
X_train = df_train[feature_cols].values
y_train = df_train['Target'].values

X_valid = df_valid[feature_cols].values
y_valid = df_valid['Target'].values

X_test = df_test[feature_cols].values
y_test = df_test['Target'].values

print(f"\nFeatures: {len(feature_cols)}")
print(f"Original bankruptcy rates:")
print(f"  Training: {y_train.mean():.3%}")
print(f"  Validation: {y_valid.mean():.3%}")
print(f"  Test: {y_test.mean():.3%}")

# Convert to proper data types if needed
X_train = X_train.astype(np.float64)
X_valid = X_valid.astype(np.float64)
X_test = X_test.astype(np.float64)

# Calculate no information rate (baseline)
no_info_rate_train = 1 - y_train.mean()
no_info_rate_valid = 1 - y_valid.mean()
no_info_rate_test = 1 - y_test.mean()

print("\nData preparation completed!")
time.sleep(2)

#================================================================================
# CREATE SAMPLED DATASETS
#================================================================================

print("\n" + "="*80)
print("CREATING SAMPLED DATASETS")
print("="*80)

sampled_datasets = {}

# Get indices of bankrupt and healthy companies
bankrupt_indices = np.where(y_train == 1)[0]
healthy_indices = np.where(y_train == 0)[0]

n_bankrupt = len(bankrupt_indices)
n_healthy = len(healthy_indices)

print(f"\nOriginal training set:")
print(f"Bankrupt companies: {n_bankrupt:,}")
print(f"Healthy companies: {n_healthy:,}")

for ratio in SAMPLING_RATIOS:
    print(f"\nCreating dataset with sampling ratio {ratio}:")

    if ratio == 1.0:
        # No sampling, use original data
        X_train_sampled = X_train
        y_train_sampled = y_train
        sampled_healthy = n_healthy
    else:
        # Sample healthy companies
        n_healthy_to_keep = int(n_healthy * ratio)
        np.random.seed(RANDOM_STATE)
        sampled_healthy_indices = np.random.choice(healthy_indices, n_healthy_to_keep, replace=False)

        # Combine with all bankrupt companies
        all_indices = np.concatenate([bankrupt_indices, sampled_healthy_indices])
        np.random.shuffle(all_indices)

        X_train_sampled = X_train[all_indices]
        y_train_sampled = y_train[all_indices]
        sampled_healthy = n_healthy_to_keep

    # Calculate scale_pos_weight for this sample
    scale_pos_weight = (y_train_sampled == 0).sum() / (y_train_sampled == 1).sum()
    bankruptcy_rate = y_train_sampled.mean()

    # Store in dictionary
    sampled_datasets[ratio] = {
        'X_train': X_train_sampled,
        'y_train': y_train_sampled,
        'scale_pos_weight': scale_pos_weight,
        'n_samples': len(y_train_sampled),
        'n_bankrupt': n_bankrupt,
        'n_healthy': sampled_healthy,
        'bankruptcy_rate': bankruptcy_rate
    }

    print(f"  • Total samples: {len(y_train_sampled):,}")
    print(f"  • Healthy companies kept: {sampled_healthy:,} ({ratio*100:.0f}%)")
    print(f"  • Bankruptcy rate: {bankruptcy_rate:.3%}")
    print(f"  • Scale pos weight: {scale_pos_weight:.2f}")
    print(f"  • Distribution shift vs test: {bankruptcy_rate/y_test.mean():.1f}x")

time.sleep(3)

# Initialize storage
results = []
models = {}
predictions = {}
best_params = {}

#================================================================================
# MODEL 1: XGBOOST
#================================================================================

tracker.update("XGBoost", "Initializing...")

def objective_xgb(trial):
    try:
        # Add sampling ratio as a hyperparameter
        sampling_ratio = trial.suggest_categorical('sampling_ratio', SAMPLING_RATIOS)

        # Get the sampled dataset
        dataset = sampled_datasets[sampling_ratio]
        X_train_sampled = dataset['X_train']
        y_train_sampled = dataset['y_train']
        scale_pos_weight = dataset['scale_pos_weight']

        tracker.update("XGBoost", f"Optimizing hyperparameters (sampling={sampling_ratio})...",
                      trial=trial.number, total_trials=MODEL_TRIALS['xgboost'],
                      study=trial.study)

        params = {
            'max_depth': trial.suggest_int('max_depth', *PARAM_RANGES['xgboost']['max_depth']),
            'learning_rate': trial.suggest_float('learning_rate', *PARAM_RANGES['xgboost']['learning_rate'], log=True),
            'n_estimators': trial.suggest_int('n_estimators', *PARAM_RANGES['xgboost']['n_estimators']),
            'subsample': trial.suggest_float('subsample', *PARAM_RANGES['xgboost']['subsample']),
            'colsample_bytree': trial.suggest_float('colsample_bytree', *PARAM_RANGES['xgboost']['colsample_bytree']),
            'reg_alpha': trial.suggest_float('reg_alpha', *PARAM_RANGES['xgboost']['reg_alpha'], log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', *PARAM_RANGES['xgboost']['reg_lambda'], log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', *PARAM_RANGES['xgboost']['min_child_weight'])
        }

        # Model configuration
        model_params = {
            **params,
            'objective': 'binary:logistic',
            'seed': RANDOM_STATE,
            'eval_metric': native_metric,
            'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
            'verbosity': 0
        }

        # Add GPU parameters
        if USE_GPU and GPU_PARAMS:
            model_params.update(GPU_PARAMS['xgboost'])

        if USE_SCALE_POS_WEIGHT:
            model_params['scale_pos_weight'] = scale_pos_weight

        model = xgb.XGBClassifier(**model_params)

        # Train with sampled data
        model.fit(
            X_train_sampled, y_train_sampled,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        # Evaluate with our custom metric on validation set
        y_pred_proba = model.predict_proba(X_valid)[:, 1]
        _, metric_value = eval_metric_func(y_valid, y_pred_proba)

        return metric_value

    except Exception as e:
        # Log failed trial
        if LOG_FAILED_TRIALS:
            failed_trials['XGBoost'].append({
                'trial': trial.number,
                'params': trial.params,
                'error': str(e),
                'traceback': traceback.format_exc()
            })
        raise

# Create and run study
study_xgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=PRUNING_N_STARTUP_TRIALS,
        n_warmup_steps=PRUNING_N_WARMUP_STEPS,
        interval_steps=PRUNING_INTERVAL_STEPS
    ) if USE_PRUNING else None
)

try:
    study_xgb.optimize(
        objective_xgb,
        n_trials=MODEL_TRIALS['xgboost'],
        timeout=OPTUNA_TIMEOUT['xgboost'],
        n_jobs=1
    )
except Exception as e:
    print(f"\nOptimization stopped: {e}")
    if len(study_xgb.trials) == 0:
        raise ValueError("No trials completed!")

best_params['XGBoost'] = study_xgb.best_params
best_sampling_ratio_xgb = study_xgb.best_params['sampling_ratio']

# Train final model with best parameters
tracker.update("XGBoost", "Training final model...")

# Get the best sampled dataset
best_dataset_xgb = sampled_datasets[best_sampling_ratio_xgb]
X_train_best = best_dataset_xgb['X_train']
y_train_best = best_dataset_xgb['y_train']
scale_pos_weight_best = best_dataset_xgb['scale_pos_weight']

final_xgb_params = {k: v for k, v in study_xgb.best_params.items() if k != 'sampling_ratio'}
final_xgb_params.update({
    'objective': 'binary:logistic',
    'seed': RANDOM_STATE,
    'eval_metric': native_metric,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'verbosity': 0
})

if USE_GPU and GPU_PARAMS:
    final_xgb_params.update(GPU_PARAMS['xgboost'])

if USE_SCALE_POS_WEIGHT:
    final_xgb_params['scale_pos_weight'] = scale_pos_weight_best

xgb_model = xgb.XGBClassifier(**final_xgb_params)

start_time = time.time()
xgb_model.fit(
    X_train_best, y_train_best,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)
train_time = time.time() - start_time

# Predictions on all sets
y_train_pred_xgb = xgb_model.predict_proba(X_train)[:, 1]
y_valid_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
y_test_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Store
models['XGBoost'] = xgb_model
predictions['XGBoost'] = {
    'train': y_train_pred_xgb,
    'valid': y_valid_pred_xgb,
    'test': y_test_pred_xgb
}

# Calculate all metrics
test_roc_auc = roc_auc_score(y_test, y_test_pred_xgb)
test_pr_auc = average_precision_score(y_test, y_test_pred_xgb)
test_partial_pr_auc = partial_pr_auc_score(y_test, y_test_pred_xgb, OPTIMIZATION_CONFIG['partial_range'])
test_prec_at_recall = precision_at_recall_score(y_test, y_test_pred_xgb, OPTIMIZATION_CONFIG['target_recall'])

# Store the optimization metric value
_, test_metric_value = eval_metric_func(y_test, y_test_pred_xgb)

results.append({
    'Model': 'XGBoost',
    'Train_Time': train_time,
    'Test_Metric': test_metric_value,
    'Test_ROCAUC': test_roc_auc,
    'Test_PRAUC': test_pr_auc,
    'Test_PartialPRAUC': test_partial_pr_auc,
    'Test_PrecAtRecall': test_prec_at_recall,
    'Sampling_Ratio': best_sampling_ratio_xgb,
    'Scale_Pos_Weight': scale_pos_weight_best
})

tracker.complete_model("XGBoost", train_time, test_metric_value, best_sampling_ratio_xgb)
clean_memory()
time.sleep(1)

#================================================================================
# MODEL 2: LIGHTGBM
#================================================================================

tracker.update("LightGBM", "Initializing...")

def objective_lgb(trial):
    try:
        # Add sampling ratio as a hyperparameter
        sampling_ratio = trial.suggest_categorical('sampling_ratio', SAMPLING_RATIOS)

        # Get the sampled dataset
        dataset = sampled_datasets[sampling_ratio]
        X_train_sampled = dataset['X_train']
        y_train_sampled = dataset['y_train']
        scale_pos_weight = dataset['scale_pos_weight']

        tracker.update("LightGBM", f"Optimizing hyperparameters (sampling={sampling_ratio})...",
                      trial=trial.number, total_trials=MODEL_TRIALS['lightgbm'],
                      study=trial.study)

        params = {
            'num_leaves': trial.suggest_int('num_leaves', *PARAM_RANGES['lightgbm']['num_leaves']),
            'learning_rate': trial.suggest_float('learning_rate', *PARAM_RANGES['lightgbm']['learning_rate'], log=True),
            'n_estimators': trial.suggest_int('n_estimators', *PARAM_RANGES['lightgbm']['n_estimators']),
            'feature_fraction': trial.suggest_float('feature_fraction', *PARAM_RANGES['lightgbm']['feature_fraction']),
            'bagging_fraction': trial.suggest_float('bagging_fraction', *PARAM_RANGES['lightgbm']['bagging_fraction']),
            'bagging_freq': trial.suggest_int('bagging_freq', *PARAM_RANGES['lightgbm']['bagging_freq']),
            'reg_alpha': trial.suggest_float('reg_alpha', *PARAM_RANGES['lightgbm']['reg_alpha'], log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', *PARAM_RANGES['lightgbm']['reg_lambda'], log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', *PARAM_RANGES['lightgbm']['min_child_weight'])
        }

        # Model configuration
        model_params = {
            **params,
            'objective': 'binary',
            'boosting_type': 'gbdt',
            'random_state': RANDOM_STATE,
            'n_jobs': -1 if not USE_GPU else 1,
            'verbosity': -1
        }

        # Add GPU parameters
        if USE_GPU and GPU_PARAMS:
            model_params.update(GPU_PARAMS['lightgbm'])

        if USE_SCALE_POS_WEIGHT:
            model_params['scale_pos_weight'] = scale_pos_weight

        model = lgb.LGBMClassifier(**model_params)

        # Train with sampled data
        model.fit(
            X_train_sampled, y_train_sampled,
            eval_set=[(X_valid, y_valid)],
            eval_metric='average_precision' if native_metric == 'aucpr' else native_metric,
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
        )

        # Evaluate with our custom metric on validation set
        y_pred_proba = model.predict_proba(X_valid)[:, 1]
        _, metric_value = eval_metric_func(y_valid, y_pred_proba)

        return metric_value

    except Exception as e:
        # Log failed trial
        if LOG_FAILED_TRIALS:
            failed_trials['LightGBM'].append({
                'trial': trial.number,
                'params': trial.params,
                'error': str(e),
                'traceback': traceback.format_exc()
            })
        raise

# Create and run study
study_lgb = optuna.create_study(
   direction='maximize',
   pruner=optuna.pruners.MedianPruner(
       n_startup_trials=PRUNING_N_STARTUP_TRIALS,
       n_warmup_steps=PRUNING_N_WARMUP_STEPS,
       interval_steps=PRUNING_INTERVAL_STEPS
   ) if USE_PRUNING else None
)

try:
   study_lgb.optimize(
       objective_lgb,
       n_trials=MODEL_TRIALS['lightgbm'],
       timeout=OPTUNA_TIMEOUT['lightgbm'],
       n_jobs=1
   )
except Exception as e:
   print(f"\nOptimization stopped: {e}")
   if len(study_lgb.trials) == 0:
       raise ValueError("No trials completed!")

best_params['LightGBM'] = study_lgb.best_params
best_sampling_ratio_lgb = study_lgb.best_params['sampling_ratio']

# Train final model with best parameters
tracker.update("LightGBM", "Training final model...")

# Get the best sampled dataset
best_dataset_lgb = sampled_datasets[best_sampling_ratio_lgb]
X_train_best = best_dataset_lgb['X_train']
y_train_best = best_dataset_lgb['y_train']
scale_pos_weight_best = best_dataset_lgb['scale_pos_weight']

final_lgb_params = {k: v for k, v in study_lgb.best_params.items() if k != 'sampling_ratio'}
final_lgb_params.update({
   'objective': 'binary',
   'boosting_type': 'gbdt',
   'random_state': RANDOM_STATE,
   'n_jobs': -1 if not USE_GPU else 1,
   'verbosity': -1
})

if USE_GPU and GPU_PARAMS:
   final_lgb_params.update(GPU_PARAMS['lightgbm'])

if USE_SCALE_POS_WEIGHT:
   final_lgb_params['scale_pos_weight'] = scale_pos_weight_best

lgb_model = lgb.LGBMClassifier(**final_lgb_params)

start_time = time.time()
lgb_model.fit(
   X_train_best, y_train_best,
   eval_set=[(X_valid, y_valid)],
   eval_metric='average_precision' if native_metric == 'aucpr' else native_metric,
   callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
)
train_time = time.time() - start_time

# Predictions on all sets
y_train_pred_lgb = lgb_model.predict_proba(X_train)[:, 1]
y_valid_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
y_test_pred_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Store
models['LightGBM'] = lgb_model
predictions['LightGBM'] = {
   'train': y_train_pred_lgb,
   'valid': y_valid_pred_lgb,
   'test': y_test_pred_lgb
}

# Calculate all metrics
test_roc_auc = roc_auc_score(y_test, y_test_pred_lgb)
test_pr_auc = average_precision_score(y_test, y_test_pred_lgb)
test_partial_pr_auc = partial_pr_auc_score(y_test, y_test_pred_lgb, OPTIMIZATION_CONFIG['partial_range'])
test_prec_at_recall = precision_at_recall_score(y_test, y_test_pred_lgb, OPTIMIZATION_CONFIG['target_recall'])

# Store the optimization metric value
_, test_metric_value = eval_metric_func(y_test, y_test_pred_lgb)

results.append({
   'Model': 'LightGBM',
   'Train_Time': train_time,
   'Test_Metric': test_metric_value,
   'Test_ROCAUC': test_roc_auc,
   'Test_PRAUC': test_pr_auc,
   'Test_PartialPRAUC': test_partial_pr_auc,
   'Test_PrecAtRecall': test_prec_at_recall,
   'Sampling_Ratio': best_sampling_ratio_lgb,
   'Scale_Pos_Weight': scale_pos_weight_best
})

tracker.complete_model("LightGBM", train_time, test_metric_value, best_sampling_ratio_lgb)
clean_memory()

#================================================================================
# FINAL RESULTS SUMMARY
#================================================================================

clear_output(wait=True)

print("BANKRUPTCY PREDICTION MODEL TRAINING COMPLETED")
print("=" * 80)
print(f"Total time: {tracker.format_time(time.time() - tracker.start_time)}")
print(f"Optimization Metric: {get_metric_display_name(OPTIMIZATION_CONFIG)}")
print("=" * 80)

# Create results DataFrame
results_df = pd.DataFrame(results)

# Sort by test metric
results_df = results_df.sort_values('Test_Metric', ascending=False)

# Determine best model
best_model_name = results_df.iloc[0]['Model']

# Function to create metric table for a model
def create_metric_table(model_name):
   """Create a formatted metric table for a specific model"""

   # Calculate all metrics for all sets
   train_roc_auc = roc_auc_score(y_train, predictions[model_name]['train'])
   valid_roc_auc = roc_auc_score(y_valid, predictions[model_name]['valid'])
   test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])

   train_pr_auc = average_precision_score(y_train, predictions[model_name]['train'])
   valid_pr_auc = average_precision_score(y_valid, predictions[model_name]['valid'])
   test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

   train_partial_pr = partial_pr_auc_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['partial_range'])
   valid_partial_pr = partial_pr_auc_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['partial_range'])
   test_partial_pr = partial_pr_auc_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['partial_range'])

   train_prec_recall = precision_at_recall_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['target_recall'])
   valid_prec_recall = precision_at_recall_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['target_recall'])
   test_prec_recall = precision_at_recall_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['target_recall'])

   # Get optimization metric values
   train_opt_metric_name, train_opt_metric = eval_metric_func(y_train, predictions[model_name]['train'])
   valid_opt_metric_name, valid_opt_metric = eval_metric_func(y_valid, predictions[model_name]['valid'])
   test_opt_metric_name, test_opt_metric = eval_metric_func(y_test, predictions[model_name]['test'])

   # Create metric data
   metrics_data = []

   # Determine which metric was used for optimization
   opt_metric_type = OPTIMIZATION_CONFIG['metric']

   # Add optimization metric first (highlighted)
   if opt_metric_type == 'roc_auc':
       metrics_data.append({
           'Metric': '★ ROC-AUC',
           'Train': f"{train_roc_auc:.4f}",
           'Validation': f"{valid_roc_auc:.4f}",
           'Test': f"{test_roc_auc:.4f}"
       })
   elif opt_metric_type == 'pr_auc':
       metrics_data.append({
           'Metric': '★ PR-AUC',
           'Train': f"{train_pr_auc:.4f}",
           'Validation': f"{valid_pr_auc:.4f}",
           'Test': f"{test_pr_auc:.4f}"
       })
   elif opt_metric_type == 'partial_pr_auc':
       range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
       metrics_data.append({
           'Metric': f'★ Partial PR-AUC {range_str}',
           'Train': f"{train_partial_pr:.4f}",
           'Validation': f"{valid_partial_pr:.4f}",
           'Test': f"{test_partial_pr:.4f}"
       })
   elif opt_metric_type == 'precision_at_recall':
       metrics_data.append({
           'Metric': f"★ Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
           'Train': f"{train_prec_recall:.4f}",
           'Validation': f"{valid_prec_recall:.4f}",
           'Test': f"{test_prec_recall:.4f}"
       })

   # Add other metrics
   if opt_metric_type != 'roc_auc':
       metrics_data.append({
           'Metric': 'ROC-AUC',
           'Train': f"{train_roc_auc:.4f}",
           'Validation': f"{valid_roc_auc:.4f}",
           'Test': f"{test_roc_auc:.4f}"
       })

   if opt_metric_type != 'pr_auc':
       metrics_data.append({
           'Metric': 'PR-AUC',
           'Train': f"{train_pr_auc:.4f}",
           'Validation': f"{valid_pr_auc:.4f}",
           'Test': f"{test_pr_auc:.4f}"
       })

   if opt_metric_type != 'partial_pr_auc':
       range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
       metrics_data.append({
           'Metric': f'Partial PR-AUC {range_str}',
           'Train': f"{train_partial_pr:.4f}",
           'Validation': f"{valid_partial_pr:.4f}",
           'Test': f"{test_partial_pr:.4f}"
       })

   if opt_metric_type != 'precision_at_recall':
       metrics_data.append({
           'Metric': f"Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
           'Train': f"{train_prec_recall:.4f}",
           'Validation': f"{valid_prec_recall:.4f}",
           'Test': f"{test_prec_recall:.4f}"
       })

   return pd.DataFrame(metrics_data)

# Display metric tables for each model
print("\nMODEL PERFORMANCE SUMMARY:")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
   if model_name not in predictions:
       continue

   print(f"\n{model_name.upper()} METRICS:")
   print("-" * 60)

   # Create and display metric table
   metric_table = create_metric_table(model_name)

   # Format table with aligned columns
   for _, row in metric_table.iterrows():
       metric_name = row['Metric']
       train_val = row['Train']
       valid_val = row['Validation']
       test_val = row['Test']
       print(f"{metric_name:<30} | Train: {train_val} | Valid: {valid_val} | Test: {test_val}")

   # Get sampling info for this model
   model_result = results_df[results_df['Model'] == model_name].iloc[0]
   sampling_ratio = model_result['Sampling_Ratio']
   scale_pos_weight = model_result['Scale_Pos_Weight']

   # Get dataset info
   dataset_info = sampled_datasets[sampling_ratio]

   print(f"\nSampling Configuration:")
   print(f"  • Sampling ratio: {sampling_ratio} (kept {sampling_ratio*100:.0f}% of healthy companies)")
   print(f"  • Scale pos weight: {scale_pos_weight:.2f}")
   print(f"  • Training bankruptcy rate: {dataset_info['bankruptcy_rate']:.3%}")
   print(f"  • Training bankruptcies: {dataset_info['n_bankrupt']:,}")

print(f"\nBest Model: {best_model_name} (based on {get_metric_display_name(OPTIMIZATION_CONFIG)})")

# Display optimized hyperparameters
print("\nOPTIMIZED HYPERPARAMETERS:")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
   if model_name in best_params:
       print(f"\n{model_name}:")
       params = best_params[model_name]
       # Sort parameters by name for consistent display
       for param in sorted(params.keys()):
           if param == 'sampling_ratio':
               continue  # Already shown above
           value = params[param]
           if isinstance(value, float):
               print(f"  • {param}: {value:.4f}")
           else:
               print(f"  • {param}: {value}")

# Detailed business metrics for best model
print("\nBUSINESS ANALYSIS FOR BEST MODEL:")
print("=" * 80)
best_pred = predictions[best_model_name]['test']

# Calculate total bankruptcies and companies
total_companies = len(y_test)
total_bankruptcies = int(y_test.sum())
bankruptcy_rate = y_test.mean()

print(f"Test Set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

# Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, best_pred)

# Business metrics at different recall levels
for target_recall in [0.90, 0.95, 0.99]:
   # Find where recall drops below target (since recall is decreasing)
   valid_indices = np.where(recall >= target_recall)[0]

   if len(valid_indices) == 0:
       print(f"\n⚠️ Cannot achieve {target_recall:.0%} recall")
       continue

   # Take the last valid index (highest precision for this recall level)
   idx = valid_indices[-1]

   # Get actual values at this point
   actual_recall = recall[idx]
   actual_precision = precision[idx]

   # Handle threshold
   if idx < len(thresholds):
       threshold = thresholds[idx]
   else:
       threshold = thresholds[-1] - 0.0001

   # Calculate business metrics
   companies_flagged = np.sum(best_pred >= threshold)

   # If companies_flagged is too small, recalculate properly
   if companies_flagged < 10:
       sorted_indices = np.argsort(best_pred)[::-1]
       bankruptcies_needed = int(np.ceil(total_bankruptcies * target_recall))
       bankruptcies_found = 0
       companies_flagged = 0

       for i, idx_company in enumerate(sorted_indices):
           companies_flagged = i + 1
           if y_test.iloc[idx_company] == 1:
               bankruptcies_found += 1
           if bankruptcies_found >= bankruptcies_needed:
               break

       actual_precision = bankruptcies_found / companies_flagged
       threshold = best_pred[sorted_indices[companies_flagged-1]]

   investigation_rate = companies_flagged / total_companies
   bankruptcies_to_catch = int(total_bankruptcies * actual_recall)

   # Improvement over random
   random_companies = int(total_companies * target_recall)
   improvement = random_companies / companies_flagged if companies_flagged > 0 else 1.0

   print(f"\nAt {target_recall:.0%} recall target:")
   print(f"  • Actual recall: {actual_recall:.3f} ({bankruptcies_to_catch}/{total_bankruptcies} bankruptcies)")
   print(f"  • Precision: {actual_precision:.3%}")
   print(f"  • Must investigate: {companies_flagged:,} companies ({investigation_rate:.1%})")
   print(f"  • Threshold: {threshold:.4f}")
   print(f"  • Improvement over random: {improvement:.2f}x")

# Model diagnostic
print("\nMODEL DIAGNOSTIC:")
print("-" * 80)

# Score statistics
bankrupt_scores = best_pred[y_test == 1]
healthy_scores = best_pred[y_test == 0]

print(f"Bankruptcy scores: mean={bankrupt_scores.mean():.4f}, median={np.median(bankrupt_scores):.4f}")
print(f"Healthy scores: mean={healthy_scores.mean():.4f}, median={np.median(healthy_scores):.4f}")

# Check score overlap
overlap_threshold = np.percentile(healthy_scores, 95)
bankrupt_below_threshold = np.sum(bankrupt_scores < overlap_threshold) / len(bankrupt_scores)
print(f"Score overlap: {bankrupt_below_threshold:.1%} of bankruptcies score below 95th percentile of healthy")

# Distribution shift analysis
print("\nDISTRIBUTION SHIFT ANALYSIS:")
print("-" * 80)
print(f"Original training bankruptcy rate: {y_train.mean():.3%}")

# Get best model's sampling info
best_model_result = results_df.iloc[0]
best_sampling_ratio = best_model_result['Sampling_Ratio']
best_dataset = sampled_datasets[best_sampling_ratio]

print(f"Sampled training bankruptcy rate: {best_dataset['bankruptcy_rate']:.3%}")
print(f"Validation bankruptcy rate: {y_valid.mean():.3%}")
print(f"Test bankruptcy rate: {y_test.mean():.3%}")
print(f"Distribution shift (sampled train vs test): {best_dataset['bankruptcy_rate'] / y_test.mean():.1f}x")

# Trial statistics
print("\nOPTIMIZATION SUMMARY:")
print("-" * 80)
for model_name, stats in tracker.trial_stats.items():
   completion_rate = stats['completed'] / stats['total_attempted'] * 100 if stats['total_attempted'] > 0 else 0
   print(f"{model_name}: {stats['completed']} completed ({completion_rate:.0f}% success rate)")

print("\n" + "="*80)

# Store metadata
training_metadata = {
   'optimization_config': OPTIMIZATION_CONFIG,
   'sampling_ratios': SAMPLING_RATIOS,
   'best_model': best_model_name,
   'best_sampling_ratios': tracker.best_sampling_ratios,
   'sampled_datasets_info': {ratio: {k: v for k, v in info.items() if k not in ['X_train', 'y_train']}
                             for ratio, info in sampled_datasets.items()},
   'trial_stats': tracker.trial_stats,
   'best_params': best_params,
   'results_df': results_df,
   'training_time': time.time() - tracker.start_time,
   'test_bankruptcies': total_bankruptcies,
   'test_companies': total_companies,
   'bankruptcy_rate': bankruptcy_rate,
   'predictions': predictions,
   'models': models
}

print(f"All data stored in 'training_metadata' dictionary for analysis.")

In [ ]:
#@title latex table
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Data structure (paste your data here)
data = {
    'ROC_AUC_opt': {
        'trial_1': {
            'ROC_AUC': {'Train': 0.9787, 'Valid': 0.9481, 'Test': 0.9438},
            'PR_AUC': {'Train': 0.0736, 'Valid': 0.0190, 'Test': 0.0145},
            'Partial_PR_AUC': {'Train': 0.0207, 'Valid': 0.0089, 'Test': 0.0088},
            'Precision_at_90_Recall': {'Train': 0.0230, 'Valid': 0.0107, 'Test': 0.0095}
        },
        'trial_2': {
            'ROC_AUC': {'Train': 0.9716, 'Valid': 0.9477, 'Test': 0.9445},
            'PR_AUC': {'Train': 0.0403, 'Valid': 0.0188, 'Test': 0.0146},
            'Partial_PR_AUC': {'Train': 0.0158, 'Valid': 0.0094, 'Test': 0.0088},
            'Precision_at_90_Recall': {'Train': 0.0172, 'Valid': 0.0111, 'Test': 0.0099}
        },
        'trial_3': {
            'ROC_AUC': {'Train': 0.9788, 'Valid': 0.9479, 'Test': 0.9470},
            'PR_AUC': {'Train': 0.0868, 'Valid': 0.0192, 'Test': 0.0151},
            'Partial_PR_AUC': {'Train': 0.0208, 'Valid': 0.0093, 'Test': 0.0091},
            'Precision_at_90_Recall': {'Train': 0.0221, 'Valid': 0.0111, 'Test': 0.0102}
        },
        'trial_4': {
            'ROC_AUC': {'Train': 0.9780, 'Valid': 0.9485, 'Test': 0.9436},
            'PR_AUC': {'Train': 0.0616, 'Valid': 0.0180, 'Test': 0.0147},
            'Partial_PR_AUC': {'Train': 0.0180, 'Valid': 0.0092, 'Test': 0.0079},
            'Precision_at_90_Recall': {'Train': 0.0205, 'Valid': 0.0107, 'Test': 0.0096}
        },
        'trial_5': {
            'ROC_AUC': {'Train': 0.9763, 'Valid': 0.9475, 'Test': 0.9445},
            'PR_AUC': {'Train': 0.0501, 'Valid': 0.0181, 'Test': 0.0147},
            'Partial_PR_AUC': {'Train': 0.0180, 'Valid': 0.0091, 'Test': 0.0086},
            'Precision_at_90_Recall': {'Train': 0.0205, 'Valid': 0.0107, 'Test': 0.0098}
        }
    },
    'PR_AUC_opt': {
        'trial_1': {
            'ROC_AUC': {'Train': 0.9905, 'Valid': 0.9398, 'Test': 0.9390},
            'PR_AUC': {'Train': 0.1219, 'Valid': 0.0269, 'Test': 0.0133},
            'Partial_PR_AUC': {'Train': 0.0422, 'Valid': 0.0087, 'Test': 0.0085},
            'Precision_at_90_Recall': {'Train': 0.0462, 'Valid': 0.0103, 'Test': 0.0092}
        },
        'trial_2': {
            'ROC_AUC': {'Train': 0.9660, 'Valid': 0.9359, 'Test': 0.9365},
            'PR_AUC': {'Train': 0.0424, 'Valid': 0.0161, 'Test': 0.0161},
            'Partial_PR_AUC': {'Train': 0.0115, 'Valid': 0.0078, 'Test': 0.0074},
            'Precision_at_90_Recall': {'Train': 0.0159, 'Valid': 0.0091, 'Test': 0.0080}
        },
        'trial_3': {
            'ROC_AUC': {'Train': 0.9762, 'Valid': 0.9398, 'Test': 0.9427},
            'PR_AUC': {'Train': 0.0453, 'Valid': 0.0264, 'Test': 0.0152},
            'Partial_PR_AUC': {'Train': 0.0200, 'Valid': 0.0089, 'Test': 0.0086},
            'Precision_at_90_Recall': {'Train': 0.0215, 'Valid': 0.0104, 'Test': 0.0095}
        },
        'trial_4': {
            'ROC_AUC': {'Train': 0.9748, 'Valid': 0.9424, 'Test': 0.9403},
            'PR_AUC': {'Train': 0.0316, 'Valid': 0.0277, 'Test': 0.0142},
            'Partial_PR_AUC': {'Train': 0.0205, 'Valid': 0.0081, 'Test': 0.0082},
            'Precision_at_90_Recall': {'Train': 0.0219, 'Valid': 0.0097, 'Test': 0.0090}
        },
        'trial_5': {
            'ROC_AUC': {'Train': 0.9816, 'Valid': 0.9336, 'Test': 0.9414},
            'PR_AUC': {'Train': 0.0445, 'Valid': 0.0272, 'Test': 0.0148},
            'Partial_PR_AUC': {'Train': 0.0278, 'Valid': 0.0091, 'Test': 0.0085},
            'Precision_at_90_Recall': {'Train': 0.0295, 'Valid': 0.0109, 'Test': 0.0093}
        }
    },
    'Partial_PR_AUC_opt': {
        'trial_1': {
            'ROC_AUC': {'Train': 0.9612, 'Valid': 0.9444, 'Test': 0.9441},
            'PR_AUC': {'Train': 0.0234, 'Valid': 0.0177, 'Test': 0.0135},
            'Partial_PR_AUC': {'Train': 0.0100, 'Valid': 0.0094, 'Test': 0.0083},
            'Precision_at_90_Recall': {'Train': 0.0137, 'Valid': 0.0113, 'Test': 0.0099}
        },
        'trial_2': {
            'ROC_AUC': {'Train': 0.9723, 'Valid': 0.9452, 'Test': 0.9426},
            'PR_AUC': {'Train': 0.0298, 'Valid': 0.0193, 'Test': 0.0141},
            'Partial_PR_AUC': {'Train': 0.0188, 'Valid': 0.0093, 'Test': 0.0089},
            'Precision_at_90_Recall': {'Train': 0.0198, 'Valid': 0.0110, 'Test': 0.0098}
        },
        'trial_3': {
            'ROC_AUC': {'Train': 0.9603, 'Valid': 0.9445, 'Test': 0.9414},
            'PR_AUC': {'Train': 0.0224, 'Valid': 0.0168, 'Test': 0.0136},
            'Partial_PR_AUC': {'Train': 0.0112, 'Valid': 0.0094, 'Test': 0.0065},
            'Precision_at_90_Recall': {'Train': 0.0126, 'Valid': 0.0113, 'Test': 0.0095}
        },
        'trial_4': {
            'ROC_AUC': {'Train': 0.9715, 'Valid': 0.9449, 'Test': 0.9414},
            'PR_AUC': {'Train': 0.0384, 'Valid': 0.0172, 'Test': 0.0135},
            'Partial_PR_AUC': {'Train': 0.0145, 'Valid': 0.0097, 'Test': 0.0057},
            'Precision_at_90_Recall': {'Train': 0.0170, 'Valid': 0.0110, 'Test': 0.0096}
        },
        'trial_5': {
            'ROC_AUC': {'Train': 0.9611, 'Valid': 0.9441, 'Test': 0.9403},
            'PR_AUC': {'Train': 0.0234, 'Valid': 0.0174, 'Test': 0.0131},
            'Partial_PR_AUC': {'Train': 0.0124, 'Valid': 0.0094, 'Test': 0.0063},
            'Precision_at_90_Recall': {'Train': 0.0134, 'Valid': 0.0111, 'Test': 0.0092}
        }
    },
    'Precision_opt': {
        'trial_1': {
            'ROC_AUC': {'Train': 0.9636, 'Valid': 0.9450, 'Test': 0.9416},
            'PR_AUC': {'Train': 0.0258, 'Valid': 0.0173, 'Test': 0.0140},
            'Partial_PR_AUC': {'Train': 0.0132, 'Valid': 0.0095, 'Test': 0.0039},
            'Precision_at_90_Recall': {'Train': 0.0142, 'Valid': 0.0113, 'Test': 0.0094}
        },
        'trial_2': {
            'ROC_AUC': {'Train': 0.9634, 'Valid': 0.9453, 'Test': 0.9444},
            'PR_AUC': {'Train': 0.0276, 'Valid': 0.0174, 'Test': 0.0141},
            'Partial_PR_AUC': {'Train': 0.0107, 'Valid': 0.0094, 'Test': 0.0073},
            'Precision_at_90_Recall': {'Train': 0.0138, 'Valid': 0.0114, 'Test': 0.0099}
        },
        'trial_3': {
            'ROC_AUC': {'Train': 0.9562, 'Valid': 0.9398, 'Test': 0.9377},
            'PR_AUC': {'Train': 0.0227, 'Valid': 0.0138, 'Test': 0.0115},
            'Partial_PR_AUC': {'Train': 0.0081, 'Valid': 0.0093, 'Test': 0.0072},
            'Precision_at_90_Recall': {'Train': 0.0128, 'Valid': 0.0112, 'Test': 0.0091}
        },
        'trial_4': {
            'ROC_AUC': {'Train': 0.9605, 'Valid': 0.9449, 'Test': 0.9431},
            'PR_AUC': {'Train': 0.0258, 'Valid': 0.0170, 'Test': 0.0140},
            'Partial_PR_AUC': {'Train': 0.0117, 'Valid': 0.0094, 'Test': 0.0087},
            'Precision_at_90_Recall': {'Train': 0.0123, 'Valid': 0.0111, 'Test': 0.0098}
        },
        'trial_5': {
            'ROC_AUC': {'Train': 0.9701, 'Valid': 0.9399, 'Test': 0.9416},
            'PR_AUC': {'Train': 0.0541, 'Valid': 0.0176, 'Test': 0.0143},
            'Partial_PR_AUC': {'Train': 0.0147, 'Valid': 0.0094, 'Test': 0.0073},
            'Precision_at_90_Recall': {'Train': 0.0164, 'Valid': 0.0110, 'Test': 0.0100}
        }
    }
}

print("📊 GENERATING LATEX TABLES WITH FIXED DIMENSIONS")
print("=" * 80)

def create_phase_comparison_table(data, phase_name):
    """Create a comparison table for a specific phase"""

    methods = ['ROC_AUC_opt', 'PR_AUC_opt', 'Partial_PR_AUC_opt', 'Precision_opt']
    metrics = ['ROC_AUC', 'PR_AUC', 'Partial_PR_AUC', 'Precision_at_90_Recall']

    # Create results table
    results = []

    for metric in metrics:
        row = {'Metric': metric.replace('_', ' ')}

        # Extract data for all methods for this metric-phase combination
        method_data = {}
        for method in methods:
            values = []
            for trial_name, trial_data in data[method].items():
                values.append(trial_data[metric][phase_name])
            method_data[method] = values

        # Calculate mean and std for each method
        method_stats = {}
        for method in methods:
            values = method_data[method]
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1) if len(values) > 1 else 0
            method_stats[method] = {'mean': mean_val, 'std': std_val, 'values': values}

        # Perform ANOVA to test for significance
        group_values = [method_data[method] for method in methods]
        f_stat, p_value = f_oneway(*group_values)

        # Find the best performing method
        best_method = max(method_stats.keys(), key=lambda x: method_stats[x]['mean'])
        best_mean = method_stats[best_method]['mean']

        # Check if the best method is significantly better than ALL others
        significance_level = ""
        if p_value < 0.1:  # Check for any significance first
            # Perform post-hoc test (Tukey HSD)
            all_values = []
            all_labels = []
            for method in methods:
                all_values.extend(method_data[method])
                all_labels.extend([method] * len(method_data[method]))

            try:
                tukey_result = pairwise_tukeyhsd(all_values, all_labels, alpha=0.1)

                # Check if best method is significantly better than all others
                best_vs_others_significant = []
                for i, comparison in enumerate(tukey_result.summary().data[1:]):  # Skip header
                    group1, group2 = comparison[0], comparison[1]
                    reject = comparison[5]  # reject null hypothesis

                    if (group1 == best_method or group2 == best_method) and group1 != group2:
                        best_vs_others_significant.append(reject)

                # If best method is significantly better than all others
                if len(best_vs_others_significant) >= 3 and all(best_vs_others_significant[:3]):
                    # Determine significance level
                    if p_value < 0.01:
                        significance_level = "***"
                    elif p_value < 0.05:
                        significance_level = "**"
                    elif p_value < 0.1:
                        significance_level = "*"

            except:
                # If post-hoc test fails, just use ANOVA p-value
                if p_value < 0.01:
                    significance_level = "***"
                elif p_value < 0.05:
                    significance_level = "**"
                elif p_value < 0.1:
                    significance_level = "*"

        # Format results for each method
        for method in methods:
            mean_val = method_stats[method]['mean']
            std_val = method_stats[method]['std']

            # Format with significance marker for LaTeX
            if method == best_method and significance_level:
                formatted_value = f"\\textbf{{{mean_val:.4f}±{std_val:.4f}{significance_level}}}"
            else:
                formatted_value = f"{mean_val:.4f}±{std_val:.4f}"

            row[method] = formatted_value

        results.append(row)

    return pd.DataFrame(results)

def generate_latex_table(df, phase_name):
    """Generate LaTeX table code from DataFrame with fixed dimensions"""

    # Table header with fixed width columns that fit within page
    latex_code = f"""% Table: {phase_name} Phase
\\begin{{table}}[htbp]
\\centering
\\caption{{Performance Comparison - {phase_name} Phase}}
\\label{{tab:{phase_name.lower()}_phase}}
\\begin{{threeparttable}}
\\footnotesize
\\setlength{{\\tabcolsep}}{{4pt}}
\\renewcommand{{\\arraystretch}}{{1.3}}
\\begin{{tabular}}{{@{{}}l*{{4}}{{>{{\centering\\arraybackslash}}p{{3.2cm}}}}@{{}}}}
\\toprule
\\textbf{{Metric}} & \\multicolumn{{4}}{{c}}{{\\textbf{{Optimization Method}}}} \\\\
\\cmidrule(lr){{2-5}}
& \\textbf{{ROC-AUC}} & \\textbf{{PR-AUC}} & \\textbf{{Partial}} & \\textbf{{Precision}} \\\\
& & & \\textbf{{PR-AUC}} & \\textbf{{@90Recall}} \\\\
\\midrule
"""

    # Table rows
    for _, row in df.iterrows():
        metric = row['Metric']
        values = [row['ROC_AUC_opt'], row['PR_AUC_opt'], row['Partial_PR_AUC_opt'], row['Precision_opt']]

        # Shorten metric names to fit better
        metric_short = metric.replace('Precision at 90 Recall', 'Prec@90Rec')
        metric_short = metric_short.replace('Partial PR AUC', 'Part. PR-AUC')
        metric_short = metric_short.replace('ROC AUC', 'ROC-AUC')
        metric_short = metric_short.replace('PR AUC', 'PR-AUC')

        latex_code += f"{metric_short} & {' & '.join(values)} \\\\[0.3em]\n"

    # Table footer
    latex_code += """\\bottomrule
\\end{tabular}
\\begin{tablenotes}
\\footnotesize
\\item[*] p < 0.1, ** p < 0.05, *** p < 0.01 (ANOVA + Tukey HSD)
\\item Bold values indicate significantly better performance than all other methods
\\item Values: mean ± std (n=5)
\\end{tablenotes}
\\end{threeparttable}
\\end{table}

"""

    return latex_code

# Generate LaTeX tables for each phase
phases = ['Train', 'Valid', 'Test']
all_latex_code = ""

# Add required packages at the beginning
all_latex_code += """% Required packages (add to preamble):
% \\usepackage{booktabs}
% \\usepackage{threeparttable}
% \\usepackage{array}
% \\usepackage{multirow}

% Alternative: Use resizebox to fit any table to text width
% Wrap each table in: \\resizebox{\\textwidth}{!}{...existing table...}

"""

for phase in phases:
    print(f"\n📊 Generating {phase.upper()} Phase LaTeX Table...")

    # Create comparison table
    comparison_df = create_phase_comparison_table(data, phase)

    # Generate LaTeX code
    latex_table = generate_latex_table(comparison_df, phase)
    all_latex_code += latex_table

    print(f"✅ {phase} phase table generated")

# Output all LaTeX code
print("\n" + "=" * 80)
print("📝 COMPLETE LATEX CODE")
print("=" * 80)
print(all_latex_code)

# Save to file (optional)
with open('comparison_tables_fixed.tex', 'w') as f:
    f.write(all_latex_code)

print("\n✅ LaTeX code saved to 'comparison_tables_fixed.tex'")
print("📋 Copy the code above and paste it into your LaTeX document!")

# Also generate a summary table showing the best method for each metric-phase combination
print("\n" + "=" * 80)
print("📊 SUMMARY: Best Methods by Metric and Phase")
print("=" * 80)

summary_data = []
for phase in phases:
    for metric in ['ROC_AUC', 'PR_AUC', 'Partial_PR_AUC', 'Precision_at_90_Recall']:
        best_method = None
        best_value = -1

        for method in ['ROC_AUC_opt', 'PR_AUC_opt', 'Partial_PR_AUC_opt', 'Precision_opt']:
            values = []
            for trial_name, trial_data in data[method].items():
                values.append(trial_data[metric][phase])
            mean_val = np.mean(values)

            if mean_val > best_value:
                best_value = mean_val
                best_method = method

        summary_data.append({
            'Phase': phase,
            'Metric': metric.replace('_', ' '),
            'Best Method': best_method.replace('_opt', ''),
            'Mean Value': f"{best_value:.4f}"
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

### graphs

---



In [ ]:
#@title Visualization Preparation

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, roc_auc_score, precision_score, recall_score
)
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Check if we're loading from saved data or using fresh training
if 'save_path' not in globals():
    # If coming from training, create save path
    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    metric_name = OPTIMIZATION_CONFIG['metric']
    n_trials = MODEL_TRIALS['xgboost']
    folder_name = f"bankruptcy_model_{metric_name}_{n_trials}trials_{current_time}"
    save_path = os.path.join(path_MODEL, folder_name)
    os.makedirs(save_path, exist_ok=True)

# Create visualization subdirectory
viz_dir = os.path.join(save_path, 'visualizations')
os.makedirs(viz_dir, exist_ok=True)

print(f"📊 Visualization directory: {viz_dir}")

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Define color schemes
model_colors = {
    'XGBoost': '#FF6B6B',   # Coral red
    'LightGBM': '#3498DB',  # Blue
    'Random' : '#95A5A6'    # Gray
}

# Modern color palette for other visualizations
modern_colors = {
    'primary': '#3498DB',    # Blue
    'secondary': '#E74C3C',  # Red
    'success': '#2ECC71',    # Green
    'warning': '#F39C12',    # Orange
    'info': '#9B59B6',       # Purple
    'light': '#ECF0F1',      # Light gray
    'dark': '#34495E'        # Dark gray
}

# Helper function to save figures
def save_fig(fig, name, dpi=300):
    """Save figure to visualization directory"""
    filepath = os.path.join(viz_dir, f"{name}.png")
    fig.savefig(filepath, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  ✅ Saved: {name}.png")

# Helper function to save plotly figures
def save_plotly_fig(fig, name):
    """Save plotly figure as static image"""
    filepath = os.path.join(viz_dir, f"{name}.png")
    fig.write_image(filepath, width=1200, height=800, scale=2)
    print(f"  ✅ Saved: {name}.png")

# Check for required Optuna visualization
try:
    import optuna.visualization as optuna_vis
    HAS_OPTUNA_VIS = True
except:
    HAS_OPTUNA_VIS = False
    print("⚠️ Optuna visualization not available")

print("✅ Visualization preparation complete")

In [ ]:
#@title 📝 Final Results Summary (with .txt Export)

import os
import pandas as pd
from IPython.display import clear_output
from datetime import datetime
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, confusion_matrix

# This list will hold all the lines of our report
report_lines = []

# ==============================================================================
# ENSURE WE HAVE THE REQUIRED DATA
# ==============================================================================

# Force creation of model results (ignore existing 'results' which contains business metrics)
model_results = []

# Check what we have available
if 'predictions' not in globals() or not predictions:
    raise ValueError("No predictions found. Please run the model training first.")

# Create results from whatever models we have
for model_name in predictions.keys():
    if model_name in predictions:
        # Calculate metrics
        test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])
        test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

        # Use default values if optimization config not available
        if 'OPTIMIZATION_CONFIG' in globals():
            test_partial_pr_auc = partial_pr_auc_score(y_test, predictions[model_name]['test'],
                                                       OPTIMIZATION_CONFIG['partial_range'])
            test_prec_at_recall = precision_at_recall_score(y_test, predictions[model_name]['test'],
                                                           OPTIMIZATION_CONFIG['target_recall'])
            _, test_metric_value = eval_metric_func(y_test, predictions[model_name]['test'])
        else:
            test_partial_pr_auc = 0.0
            test_prec_at_recall = 0.0
            test_metric_value = test_roc_auc  # Use ROC-AUC as default

        # Get sampling info from best_params if available
        if 'best_params' in globals() and model_name in best_params:
            sampling_ratio = best_params[model_name].get('sampling_ratio', 0.3)
        else:
            sampling_ratio = 0.3  # Default

        # Calculate scale_pos_weight
        if 'sampled_datasets' in globals() and sampling_ratio in sampled_datasets:
            scale_pos_weight = sampled_datasets[sampling_ratio]['scale_pos_weight']
        else:
            scale_pos_weight = 233.24  # Default from your output

        model_results.append({
            'Model': model_name,
            'Test_Metric': test_metric_value,
            'Test_ROCAUC': test_roc_auc,
            'Test_PRAUC': test_pr_auc,
            'Test_PartialPRAUC': test_partial_pr_auc,
            'Test_PrecAtRecall': test_prec_at_recall,
            'Sampling_Ratio': sampling_ratio,
            'Scale_Pos_Weight': scale_pos_weight
        })

# Now check if model_results is empty
if not model_results:
    raise ValueError("No model results available. Please ensure at least one model was trained successfully.")

# Create results DataFrame from model_results (not the business analysis results)
results_df = pd.DataFrame(model_results)

# Debug print to see what columns we have
print("Available columns in results_df:", results_df.columns.tolist())
print("Number of rows:", len(results_df))

# Sort by the best available metric
if 'Test_Metric' in results_df.columns:
    results_df = results_df.sort_values('Test_Metric', ascending=False)
elif 'Test_ROCAUC' in results_df.columns:
    results_df = results_df.sort_values('Test_ROCAUC', ascending=False)
else:
    print("Warning: No metric columns found for sorting")

# Get best model name
if not results_df.empty:
    best_model_name = results_df.iloc[0]['Model']
else:
    # If we only have XGBoost
    best_model_name = 'XGBoost' if 'XGBoost' in predictions else list(predictions.keys())[0]

# Helper function to create metric table
def create_metric_table(model_name):
    """Create a formatted metric table for a specific model"""

    # Calculate all metrics for all sets
    train_roc_auc = roc_auc_score(y_train, predictions[model_name]['train'])
    valid_roc_auc = roc_auc_score(y_valid, predictions[model_name]['valid'])
    test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])

    train_pr_auc = average_precision_score(y_train, predictions[model_name]['train'])
    valid_pr_auc = average_precision_score(y_valid, predictions[model_name]['valid'])
    test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

    train_partial_pr = partial_pr_auc_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['partial_range'])
    valid_partial_pr = partial_pr_auc_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['partial_range'])
    test_partial_pr = partial_pr_auc_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['partial_range'])

    train_prec_recall = precision_at_recall_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['target_recall'])
    valid_prec_recall = precision_at_recall_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['target_recall'])
    test_prec_recall = precision_at_recall_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['target_recall'])

    # Create metric data
    metrics_data = []

    # Determine which metric was used for optimization
    opt_metric_type = OPTIMIZATION_CONFIG['metric']

    # Add optimization metric first (highlighted)
    if opt_metric_type == 'partial_pr_auc':
        range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
        metrics_data.append({
            'Metric': f'★ Partial PR-AUC {range_str}',
            'Train': f"{train_partial_pr:.4f}",
            'Validation': f"{valid_partial_pr:.4f}",
            'Test': f"{test_partial_pr:.4f}"
        })

    # Add other metrics
    metrics_data.append({
        'Metric': 'ROC-AUC',
        'Train': f"{train_roc_auc:.4f}",
        'Validation': f"{valid_roc_auc:.4f}",
        'Test': f"{test_roc_auc:.4f}"
    })

    metrics_data.append({
        'Metric': 'PR-AUC',
        'Train': f"{train_pr_auc:.4f}",
        'Validation': f"{valid_pr_auc:.4f}",
        'Test': f"{test_pr_auc:.4f}"
    })

    if opt_metric_type != 'partial_pr_auc':
        range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
        metrics_data.append({
            'Metric': f'Partial PR-AUC {range_str}',
            'Train': f"{train_partial_pr:.4f}",
            'Validation': f"{valid_partial_pr:.4f}",
            'Test': f"{test_partial_pr:.4f}"
        })

    metrics_data.append({
        'Metric': f"Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
        'Train': f"{train_prec_recall:.4f}",
        'Validation': f"{valid_prec_recall:.4f}",
        'Test': f"{test_prec_recall:.4f}"
    })

    return pd.DataFrame(metrics_data)

# ==============================================================================
# BUILD THE REPORT STRING
# ==============================================================================

report_lines.append("BANKRUPTCY PREDICTION MODEL TRAINING COMPLETED")
report_lines.append("=" * 80)

# Handle time calculation
if 'tracker' in globals() and hasattr(tracker, 'start_time'):
    total_time = tracker.format_time(time.time() - tracker.start_time)
elif 'training_metadata' in globals() and 'training_time' in training_metadata:
    total_time = f"{training_metadata['training_time']/3600:.1f} hours"
else:
    total_time = "N/A"

report_lines.append(f"Total time: {total_time}")
report_lines.append(f"Optimization Metric: {get_metric_display_name(OPTIMIZATION_CONFIG)}")
report_lines.append("=" * 80)
report_lines.append("\nMODEL PERFORMANCE SUMMARY:")
report_lines.append("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        continue

    report_lines.append(f"\n{model_name.upper()} METRICS:")
    report_lines.append("-" * 60)

    # Create and format metric table
    metric_table = create_metric_table(model_name)
    # Use to_string() for clean text formatting
    report_lines.append(metric_table.to_string(index=False))

    # Get sampling info
    model_result = results_df[results_df['Model'] == model_name].iloc[0]
    sampling_ratio = model_result['Sampling_Ratio']
    scale_pos_weight = model_result['Scale_Pos_Weight']

    # Get dataset info if available
    if 'sampled_datasets' in globals() and sampling_ratio in sampled_datasets:
        dataset_info = sampled_datasets[sampling_ratio]
        report_lines.append(f"\nSampling Configuration:")
        report_lines.append(f"  • Sampling ratio: {sampling_ratio} (kept {sampling_ratio*100:.0f}% of healthy companies)")
        report_lines.append(f"  • Scale pos weight: {scale_pos_weight:.2f}")
        report_lines.append(f"  • Training bankruptcy rate: {dataset_info['bankruptcy_rate']:.3%}")
        report_lines.append(f"  • Training bankruptcies: {dataset_info['n_bankrupt']:,}")
    else:
        report_lines.append(f"\nSampling Configuration:")
        report_lines.append(f"  • Sampling ratio: {sampling_ratio}")
        report_lines.append(f"  • Scale pos weight: {scale_pos_weight:.2f}")

report_lines.append(f"\nBest Model: {best_model_name} (based on {get_metric_display_name(OPTIMIZATION_CONFIG)})")

# --- OPTIMIZED HYPERPARAMETERS ---
report_lines.append("\n\nOPTIMIZED HYPERPARAMETERS:")
report_lines.append("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name in best_params:
        report_lines.append(f"\n{model_name}:")
        params = best_params[model_name]
        for param in sorted(params.keys()):
            if param == 'sampling_ratio': continue
            value = params[param]
            if isinstance(value, float):
                report_lines.append(f"  • {param}: {value:.4f}")
            else:
                report_lines.append(f"  • {param}: {value}")

# --- BUSINESS ANALYSIS ---
report_lines.append("\n\nBUSINESS ANALYSIS FOR BEST MODEL:")
report_lines.append("=" * 80)
best_pred = predictions[best_model_name]['test']
total_companies = len(y_test)
total_bankruptcies = int(y_test.sum())
bankruptcy_rate = y_test.mean()

report_lines.append(f"Test Set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

precision, recall, thresholds = precision_recall_curve(y_test, best_pred)

for target_recall in [0.90, 0.95, 0.99]:
    valid_indices = np.where(recall >= target_recall)[0]
    if len(valid_indices) == 0:
        report_lines.append(f"\n⚠️ Cannot achieve {target_recall:.0%} recall")
        continue

    idx = valid_indices[-1]
    actual_recall = recall[idx]
    actual_precision = precision[idx]
    threshold = thresholds[idx] if idx < len(thresholds) else thresholds[-1] - 0.0001
    companies_flagged = np.sum(best_pred >= threshold)
    investigation_rate = companies_flagged / total_companies
    bankruptcies_to_catch = int(total_bankruptcies * actual_recall)
    random_companies = int(total_companies * target_recall)
    improvement = random_companies / companies_flagged if companies_flagged > 0 else 1.0

    report_lines.append(f"\nAt {target_recall:.0%} recall target:")
    report_lines.append(f"  • Actual recall: {actual_recall:.3f} ({bankruptcies_to_catch}/{total_bankruptcies} bankruptcies)")
    report_lines.append(f"  • Precision: {actual_precision:.3%}")
    report_lines.append(f"  • Must investigate: {companies_flagged:,} companies ({investigation_rate:.1%})")
    report_lines.append(f"  • Threshold: {threshold:.4f}")
    report_lines.append(f"  • Improvement over random: {improvement:.2f}x")

# --- MODEL DIAGNOSTIC ---
report_lines.append("\n\nMODEL DIAGNOSTIC:")
report_lines.append("-" * 80)
bankrupt_scores = best_pred[y_test == 1]
healthy_scores = best_pred[y_test == 0]
report_lines.append(f"Bankruptcy scores: mean={bankrupt_scores.mean():.4f}, median={np.median(bankrupt_scores):.4f}")
report_lines.append(f"Healthy scores: mean={healthy_scores.mean():.4f}, median={np.median(healthy_scores):.4f}")
overlap_threshold = np.percentile(healthy_scores, 95)
bankrupt_below_threshold = np.sum(bankrupt_scores < overlap_threshold) / len(bankrupt_scores)
report_lines.append(f"Score overlap: {bankrupt_below_threshold:.1%} of bankruptcies score below 95th percentile of healthy")

# --- ADDITIONAL CLASSIFICATION METRICS ---
report_lines.append("\n\nADDITIONAL CLASSIFICATION METRICS:")
report_lines.append("-" * 80)

# Calculate No Information Rate (NIR)
nir_train = 1 - y_train.mean()  # Majority class proportion
nir_valid = 1 - y_valid.mean()
nir_test = 1 - y_test.mean()

report_lines.append(f"\nNo Information Rate (NIR):")
report_lines.append(f"  • Training set: {nir_train:.3%} (baseline accuracy if always predicting 'healthy')")
report_lines.append(f"  • Validation set: {nir_valid:.3%}")
report_lines.append(f"  • Test set: {nir_test:.3%}")

# For each model, calculate Cohen's Kappa and MCC at different thresholds
for model_name in predictions.keys():
    report_lines.append(f"\n{model_name.upper()} - Advanced Metrics:")

    # Get predictions
    pred_proba_test = predictions[model_name]['test']

    # Calculate metrics at different thresholds
    thresholds_to_test = [0.5, 0.7, 0.8, 0.9]  # Common thresholds

    # Also add the threshold for 90% recall if available
    precision, recall, thresholds = precision_recall_curve(y_test, pred_proba_test)
    valid_indices = np.where(recall >= 0.90)[0]
    if len(valid_indices) > 0:
        idx = valid_indices[-1]
        if idx < len(thresholds):
            optimal_threshold = thresholds[idx]
            if optimal_threshold not in thresholds_to_test and 0 < optimal_threshold < 1:
                thresholds_to_test.append(optimal_threshold)

    # Sort thresholds
    thresholds_to_test = sorted(thresholds_to_test)

    # Table header
    report_lines.append(f"\n  Threshold | Accuracy | Cohen's κ |    MCC   | Precision | Recall")
    report_lines.append("  " + "-" * 65)

    for threshold in thresholds_to_test:
        # Make binary predictions
        y_pred = (pred_proba_test >= threshold).astype(int)

        # Calculate metrics
        accuracy = np.mean(y_pred == y_test)
        kappa = cohen_kappa_score(y_test, y_pred)
        mcc = matthews_corrcoef(y_test, y_pred)

        # Calculate precision and recall for this threshold
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0

        # Format the row
        threshold_str = f"{threshold:.3f}"
        if 'optimal_threshold' in locals() and threshold == optimal_threshold:
            threshold_str += "*"  # Mark the 90% recall threshold

        report_lines.append(f"  {threshold_str:8} | {accuracy:8.3%} | {kappa:8.3f} | {mcc:8.3f} | {precision_val:8.3%} | {recall_val:6.3%}")

    if 'optimal_threshold' in locals():
        report_lines.append("  * Threshold for 90% recall target")

    # Best MCC analysis
    # Find threshold that maximizes MCC
    mcc_scores = []
    test_thresholds = np.linspace(0.01, 0.99, 100)
    for t in test_thresholds:
        y_pred = (pred_proba_test >= t).astype(int)
        if len(np.unique(y_pred)) > 1:  # Ensure we have both classes
            mcc_scores.append(matthews_corrcoef(y_test, y_pred))
        else:
            mcc_scores.append(-1)

    if mcc_scores:
        best_mcc_idx = np.argmax(mcc_scores)
        best_mcc_threshold = test_thresholds[best_mcc_idx]
        best_mcc = mcc_scores[best_mcc_idx]

        # Calculate metrics at best MCC threshold
        y_pred_best_mcc = (pred_proba_test >= best_mcc_threshold).astype(int)
        accuracy_best_mcc = np.mean(y_pred_best_mcc == y_test)
        kappa_best_mcc = cohen_kappa_score(y_test, y_pred_best_mcc)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_best_mcc).ravel()
        precision_best_mcc = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall_best_mcc = tp / (tp + fn) if (tp + fn) > 0 else 0

        report_lines.append(f"\n  Best MCC Performance:")
        report_lines.append(f"    • Threshold: {best_mcc_threshold:.3f}")
        report_lines.append(f"    • MCC: {best_mcc:.3f}")
        report_lines.append(f"    • Cohen's κ: {kappa_best_mcc:.3f}")
        report_lines.append(f"    • Accuracy: {accuracy_best_mcc:.3%} (vs NIR: {nir_test:.3%})")
        report_lines.append(f"    • Precision: {precision_best_mcc:.3%}")
        report_lines.append(f"    • Recall: {recall_best_mcc:.3%}")

# Interpretation guide
report_lines.append("\n\nMETRIC INTERPRETATION GUIDE:")
report_lines.append("  • NIR: No Information Rate - accuracy achieved by always predicting majority class")
report_lines.append("  • Cohen's κ: Agreement corrected for chance (-1 to 1, 0 = no better than chance)")
report_lines.append("  • MCC: Matthews Correlation Coefficient (-1 to 1, 0 = no better than random)")
report_lines.append("  • For imbalanced data: MCC and Cohen's κ are more informative than accuracy")

# --- DISTRIBUTION SHIFT ANALYSIS ---
report_lines.append("\n\nDISTRIBUTION SHIFT ANALYSIS:")
report_lines.append("-" * 80)
report_lines.append(f"Original training bankruptcy rate: {y_train.mean():.3%}")

# Handle different data availability scenarios
if 'sampled_datasets' in globals():
    best_model_result = results_df.iloc[0]
    best_sampling_ratio = best_model_result['Sampling_Ratio']
    best_dataset = sampled_datasets[best_sampling_ratio]
    report_lines.append(f"Sampled training bankruptcy rate: {best_dataset['bankruptcy_rate']:.3%}")
    dist_shift = best_dataset['bankruptcy_rate'] / y_test.mean()
else:
    report_lines.append(f"Sampled training bankruptcy rate: N/A")
    dist_shift = "N/A"

report_lines.append(f"Validation bankruptcy rate: {y_valid.mean():.3%}")
report_lines.append(f"Test bankruptcy rate: {y_test.mean():.3%}")
if isinstance(dist_shift, float):
    report_lines.append(f"Distribution shift (sampled train vs test): {dist_shift:.1f}x")
else:
    report_lines.append(f"Distribution shift (sampled train vs test): {dist_shift}")

# --- OPTIMIZATION SUMMARY ---
if 'tracker' in globals() and hasattr(tracker, 'trial_stats'):
    report_lines.append("\n\nOPTIMIZATION SUMMARY:")
    report_lines.append("-" * 80)
    for model_name, stats in tracker.trial_stats.items():
        completion_rate = stats['completed'] / stats['total_attempted'] * 100 if stats['total_attempted'] > 0 else 0
        report_lines.append(f"{model_name}: {stats['completed']} completed ({completion_rate:.0f}% success rate)")

# ==============================================================================
# FINAL REPORT OUTPUT AND SAVE
# ==============================================================================
# Join all lines into a single string
final_report_text = "\n".join(report_lines)

# Print the final report (without clearing output)
print(final_report_text)

# Save the report to the SAME FOLDER as other outputs
try:
    # Use the same path where you save models
    if 'save_path' in globals():
        report_path = os.path.join(save_path, "model_summary_report.txt")
    else:
        # Fallback to path_MODEL if save_path doesn't exist
        report_path = os.path.join(path_MODEL, "model_summary_report.txt")

    with open(report_path, "w", encoding='utf-8') as f:
        f.write(final_report_text)

    print("\n" + "="*80)
    print(f"✅ Summary report saved to: {report_path}")
    print("="*80)

except Exception as e:
    print(f"\n⚠️ Could not save summary report. Error: {e}")

# Store or update metadata
if 'training_metadata' not in globals():
    training_metadata = {}

training_metadata.update({
    'optimization_config': OPTIMIZATION_CONFIG,
    'best_model': best_model_name,
    'results_df': results_df,
    'test_bankruptcies': total_bankruptcies,
    'test_companies': total_companies,
    'bankruptcy_rate': bankruptcy_rate,
    'predictions': predictions,
    'models': models
})

print(f"\nAll data stored in 'training_metadata' dictionary for analysis.")

In [ ]:
#@title 📊 Dataset Split Visualizations - Modern & Minimal

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.patches as mpatches

# Set modern style
plt.style.use('seaborn-v0_8-whitegrid')

# Define colors - using light cobalt for healthy
colors = {
    'train': '#3498db',      # Modern blue
    'valid': '#f39c12',      # Modern red
    'test': '#2ecc71',       # Modern green
    'bankrupt': '#e74c3c',   # Red for bankrupt
    'healthy': '#80A4ED',    # Light Cobalt Blue
    'text': '#2c3e50'        # Dark blue-gray text
}

# ==============================================================================
# VISUALIZATION 1: Timeline with Bankruptcy Rate
# ==============================================================================
print("📊 Generating Timeline Visualization...")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), height_ratios=[3, 1], sharex=True)
fig.patch.set_facecolor('white')

# Use actual yearly data from df_long
years = [2016, 2017, 2018, 2019, 2020, 2021]
x_positions = np.arange(len(years))

yearly_counts = df_long['Year'].value_counts().reindex(years, fill_value=0).tolist()
yearly_bankrupt_counts = df_long.groupby('Year')['Target'].sum().reindex(years, fill_value=0).tolist()
yearly_bankruptcy_rates = (df_long.groupby('Year')['Target'].mean().reindex(years, fill_value=0) * 100).tolist()
bar_colors = [colors['train']] * 3 + [colors['valid']] * 1 + [colors['test']] * 2

# Top panel: Sample sizes
ax1.bar(x_positions, yearly_counts, width=0.8, color=bar_colors, alpha=0.7,
        edgecolor='white', linewidth=2)

ax1.text(1, -0.07, 'TRAINING', ha='center', va='top', transform=ax1.get_xaxis_transform(),
         fontsize=14, weight='bold', color=colors['train'], clip_on=False)
ax1.text(3, -0.07, 'VALIDATION', ha='center', va='top', transform=ax1.get_xaxis_transform(),
         fontsize=14, weight='bold', color=colors['valid'], clip_on=False)
ax1.text(4.5, -0.07, 'TEST', ha='center', va='top', transform=ax1.get_xaxis_transform(),
         fontsize=14, weight='bold', color=colors['test'], clip_on=False)

for i, count in enumerate(yearly_counts):
    ax1.text(x_positions[i], count + 5000, f'{count:,}',
             ha='center', va='bottom', fontsize=10,
             fontweight='bold', color=bar_colors[i])

ax1.set_ylabel('Number of Companies', fontsize=12, color=colors['text'])
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))
fig.suptitle('Temporal Dataset Split', fontsize=18, weight='bold',
             color=colors['text'])

# Bottom panel: Plots absolute bankruptcy counts
ax2.plot(x_positions, yearly_bankrupt_counts, 'o-', color=colors['bankrupt'],
         linewidth=3, markersize=10, alpha=0.8)
ax2.fill_between(x_positions, yearly_bankrupt_counts, alpha=0.2, color=colors['bankrupt'])

# --- CHANGE: Reverted labels to show absolute counts and increased the spacing ---
for i, count in enumerate(yearly_bankrupt_counts):
    # This offset is a percentage of the total y-axis height to ensure consistent spacing
    offset = max(yearly_bankrupt_counts) * 0.08
    ax2.text(x_positions[i], yearly_bankrupt_counts[i] + offset, f'{count}',
             ha='center', va='bottom', fontsize=9,
             color=colors['bankrupt'], fontweight='bold')

ax2.set_xticks(x_positions)
ax2.set_xticklabels(years, fontsize=12)
ax2.set_ylabel('Number of Bankruptcies', fontsize=12, color=colors['text'])
ax2.set_ylim(0, max(yearly_bankrupt_counts) * 1.5) # Adjusted ylim for text
# Removed percentage formatter

# Remove spines
for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'dataset_split_timeline')
plt.show()

# ==============================================================================
# VISUALIZATION 2: Modern Stacked Bar with Magnified Inset
# (This part remains unchanged)
# ==============================================================================
print("\n📊 Generating Stacked Bar Visualization...")

fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor('white')

# Main plot
ax_main = plt.subplot2grid((3, 3), (0, 0), colspan=3, rowspan=2)

# Data for stacked bar
sets = ['Training\n2016-2018', 'Validation\n2019', 'Test\n2020-2021']
healthy_counts = [len(y_train) - y_train.sum(), len(y_valid) - y_valid.sum(), len(y_test) - y_test.sum()]
bankrupt_counts = [y_train.sum(), y_valid.sum(), y_test.sum()]
total_counts = [len(y_train), len(y_valid), len(y_test)]

x = np.arange(len(sets))
width = 0.6

# Create bars
p1 = ax_main.bar(x, healthy_counts, width, label='Healthy',
                 color=colors['healthy'], alpha=0.8, edgecolor='white', linewidth=2)
p2 = ax_main.bar(x, bankrupt_counts, width, bottom=healthy_counts,
                 label='Bankrupt', color=colors['bankrupt'], alpha=0.9,
                 edgecolor='white', linewidth=2)

# Customize main plot
ax_main.set_ylabel('Number of Companies', fontsize=14, color=colors['text'])
ax_main.set_xticks(x)
ax_main.set_xticklabels(sets, fontsize=12)
ax_main.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))
ax_main.set_title('Dataset Composition', fontsize=20, weight='bold',
                  color=colors['text'], pad=20)

for i, total in enumerate(total_counts):
    ax_main.text(i, total + 5000, f'{total:,}', ha='center', va='bottom',
                 fontsize=11, weight='bold', color=colors['text'])

# Inset for bankruptcy detail
ax_inset = plt.subplot2grid((3, 3), (2, 0), colspan=3)
ax_inset.bar(x, bankrupt_counts, width, color=colors['bankrupt'],
             alpha=0.9, edgecolor='white', linewidth=2)

for i, b in enumerate(bankrupt_counts):
    percentage = b / total_counts[i]
    ax_inset.text(i, b + 5, f'{b}\n({percentage:.3%})', ha='center', va='bottom',
                  fontsize=10, weight='bold', color=colors['text'])

ax_inset.set_xticks(x)
ax_inset.set_xticklabels(sets, fontsize=11)
ax_inset.set_ylabel('Bankruptcies', fontsize=12, color=colors['text'])
ax_inset.set_title('Bankruptcy Detail', fontsize=14,
                   color=colors['text'], pad=10)
# Set a dynamic y-limit for the inset
ax_inset.set_ylim(0, max(bankrupt_counts) * 1.4)


# Remove spines
for ax in [ax_main, ax_inset]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Add legend
ax_main.legend(loc='upper right', frameon=True, fancybox=True,
               shadow=True, fontsize=12)

plt.tight_layout()
save_fig(fig, 'dataset_split_stacked')
plt.show()

print("\n" + "="*80)
print("✅ All visualizations saved successfully!")
print("="*80)

In [ ]:
#@title 🗺️ Geographic Analysis – two maps saved (CartoDB Voyager & Esri GrayCanvas)

# ==============================================================================
# ⚙️ VISUALIZATION CONFIGURATION
# ==============================================================================
BANKRUPT_COLOR, HEALTHY_COLOR   = 'red', 'green'
BANKRUPT_ALPHA, HEALTHY_ALPHA   = 0.30, 0.07
LATITUDE_LINE_COLOR             = '#d62728'
LONGITUDE_LINE_COLOR            = '#d62728'
SHOW_HEALTHY_COMPANIES          = True
SHOW_BANKRUPT_COMPANIES         = True
NUM_H                           = 50000
TOP_N_SPLITS                    = 1000        # Number of top splits to show (increase for more lines)

# ==============================================================================
print("🗺️ GEOGRAPHIC SPLIT ANALYSIS")
print("=" * 80)

import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import numpy as np, contextily as ctx, re, os, random


# ------------------------------------------------------------------------------
# 1️⃣  identify latitude / longitude feature indices
lat_feature = "Registered_office_address_Latitude"
lon_feature = "Registered_office_address_Longitude"
lat_idx = feature_names.index(lat_feature);  lon_idx = feature_names.index(lon_feature)

# ------------------------------------------------------------------------------
# 2️⃣  collect split-gains from the *winning* model
lat_split_gains, lon_split_gains = {}, {}
model = models[best_model_name]

if hasattr(model, "get_booster"):                               # XGBoost
    for tree in model.get_booster().get_dump(with_stats=True):
        for line in tree.split('\n'):
            m = re.search(r"\[f(\d+)<([-\d\.e]+)\].*?,gain=([-\d\.e]+),", line)
            if not m: continue
            f, thr, gain = int(m.group(1)), float(m.group(2)), float(m.group(3))
            d = lat_split_gains if f==lat_idx else lon_split_gains if f==lon_idx else None
            if d is not None: d[thr] = d.get(thr,0)+gain

elif hasattr(model, "booster_"):                                 # LightGBM
    def walk(node):
        if "split_feature" not in node: return
        f, thr, gain = node["split_feature"], float(node["threshold"]), node["split_gain"]
        d = lat_split_gains if f==lat_idx else lon_split_gains if f==lon_idx else None
        if d is not None: d[thr] = d.get(thr,0)+gain
        walk(node["left_child"]); walk(node["right_child"])
    for ti in model.booster_.dump_model()["tree_info"]:
        walk(ti["tree_structure"])
else:
    raise ValueError("Unknown model type for geographic analysis")

print(f"Latitude splits: {len(lat_split_gains)}   Longitude splits: {len(lon_split_gains)}")

# ------------------------------------------------------------------------------
# 3️⃣  data arrays & plotting helpers
lat_vals, lon_vals = X_train[:,lat_idx], X_train[:,lon_idx]
ITALY_LAT_MIN, ITALY_LAT_MAX = 36.5, 47.5
ITALY_LON_MIN, ITALY_LON_MAX =  6.5, 19.0
healthy_idx = np.where(y_train==0)[0]
bankrupt_idx = np.where(y_train==1)[0]

def draw_map(bg_source, file_stub):
    fig, ax = plt.subplots(figsize=(14,14))

    # decision lines -----------------------------------------------------------
    # OPTION 3: Show top N splits by gain
    if lat_split_gains:
        sorted_lat_splits = sorted(lat_split_gains.items(), key=lambda x: x[1], reverse=True)
        top_lat_splits = sorted_lat_splits[:TOP_N_SPLITS]
        if top_lat_splits:
            mx = max(lat_split_gains.values())
            for s, g in top_lat_splits:
                ax.axhline(s, color=LATITUDE_LINE_COLOR,
                         alpha=min(1.0, 0.1+0.9*g/mx), lw=1.5, zorder=2)

    if lon_split_gains:
        sorted_lon_splits = sorted(lon_split_gains.items(), key=lambda x: x[1], reverse=True)
        top_lon_splits = sorted_lon_splits[:TOP_N_SPLITS]
        if top_lon_splits:
            mx = max(lon_split_gains.values())
            for s, g in top_lon_splits:
                ax.axvline(s, color=LONGITUDE_LINE_COLOR,
                         alpha=min(1.0, 0.1+0.9*g/mx), lw=1.5, zorder=2)

    # points -------------------------------------------------------------------
    if SHOW_HEALTHY_COMPANIES:
        samp = random.sample(list(healthy_idx), k=min(NUM_H,len(healthy_idx)))
        ax.scatter(lon_vals[samp], lat_vals[samp], c=HEALTHY_COLOR, alpha=HEALTHY_ALPHA,
                   s=20, label="Healthy (sample)", zorder=3)
    if SHOW_BANKRUPT_COMPANIES:
        ax.scatter(lon_vals[bankrupt_idx], lat_vals[bankrupt_idx], c=BANKRUPT_COLOR,
                   alpha=BANKRUPT_ALPHA, s=40, edgecolors="white", linewidth=0.5,
                   label="Bankrupt (all)", zorder=4)

    # background map -----------------------------------------------------------
    ax.set_xlim(ITALY_LON_MIN, ITALY_LON_MAX);  ax.set_ylim(ITALY_LAT_MIN, ITALY_LAT_MAX)
    try:
        ctx.add_basemap(ax, crs="EPSG:4326", source=bg_source, zorder=1)
    except Exception as e:
        print(f"⚠️  Map background failed: {e}")

    # legend -------------------------------------------------------------------
    legend_elems=[]
    if LATITUDE_LINE_COLOR==LONGITUDE_LINE_COLOR:
        legend_elems.append(mpatches.Patch(color=LATITUDE_LINE_COLOR,alpha=0.6,
                                           label=f"Decision boundaries"))
    else:
        legend_elems.extend([
            mpatches.Patch(color=LONGITUDE_LINE_COLOR,alpha=0.6,label=f"Longitude splits"),
            mpatches.Patch(color=LATITUDE_LINE_COLOR, alpha=0.6,label=f"Latitude splits")])
    legend_elems.extend([
        plt.Line2D([0],[0],marker='o',color='w',label='Healthy (sample)',
                   markerfacecolor=HEALTHY_COLOR,alpha=0.8,markersize=8),
        plt.Line2D([0],[0],marker='o',color='w',label='Bankrupt (all)',
                   markerfacecolor=BANKRUPT_COLOR,alpha=0.8,markersize=8)])
    ax.legend(handles=legend_elems,loc="upper right")

    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title(f"Geographic decision boundaries by importance", fontsize=16, weight="bold")
    save_fig(fig, file_stub)
    plt.show()

# ------------------------------------------------------------------------------
# 4️⃣  produce both maps
print(f"\nShowing top {TOP_N_SPLITS} splits for each dimension (lat/lon)")
draw_map(ctx.providers.CartoDB.Positron,  "geographic_splits_map_Positron")
draw_map(ctx.providers.CartoDB.DarkMatter,"geographic_splits_map_Dark")
draw_map(ctx.providers.CartoDB.Voyager,     "geographic_splits_map_voyager")
draw_map(ctx.providers.Esri.WorldGrayCanvas,"geographic_splits_map_Gray")
draw_map(ctx.providers.Esri.WorldShadedRelief,"geographic_splits_map_shaded")

In [ ]:
#@title 🗺️ Pixelated Bankruptcy Rate Map with Decision Boundaries

# ==============================================================================
# ⚙️ VISUALIZATION CONFIGURATION
# ==============================================================================
GRID_RESOLUTION = 50  # Grid resolution (50x50 cells)
MIN_COMPANIES_PER_CELL = 5  # Minimum companies to calculate rate
BOUNDARY_COLOR_LAT = '#d62728'  # Red for latitude lines
BOUNDARY_COLOR_LON = '#d62728'  # Red for longitude lines

# ==============================================================================
print("🗺️ PIXELATED BANKRUPTCY RATE MAP")
print("=" * 80)

import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1️⃣  identify latitude / longitude feature indices
lat_feature = "Registered_office_address_Latitude"
lon_feature = "Registered_office_address_Longitude"
lat_idx = feature_names.index(lat_feature)
lon_idx = feature_names.index(lon_feature)

# ------------------------------------------------------------------------------
# 2️⃣  collect split-gains from the winning model
lat_split_gains, lon_split_gains = {}, {}
model = models[best_model_name]

if hasattr(model, "get_booster"):  # XGBoost
    for tree in model.get_booster().get_dump(with_stats=True):
        for line in tree.split('\n'):
            m = re.search(r"\[f(\d+)<([-\d\.e]+)\].*?,gain=([-\d\.e]+),", line)
            if not m: continue
            f, thr, gain = int(m.group(1)), float(m.group(2)), float(m.group(3))
            d = lat_split_gains if f==lat_idx else lon_split_gains if f==lon_idx else None
            if d is not None: d[thr] = d.get(thr,0)+gain

print(f"Found {len(lat_split_gains)} latitude splits and {len(lon_split_gains)} longitude splits")

# ------------------------------------------------------------------------------
# 3️⃣  prepare data
lat_vals = X_train[:, lat_idx]
lon_vals = X_train[:, lon_idx]
ITALY_LAT_MIN, ITALY_LAT_MAX = 36.5, 47.5
ITALY_LON_MIN, ITALY_LON_MAX = 6.5, 19.0

# Filter to Italy bounds
mask = (lat_vals >= ITALY_LAT_MIN) & (lat_vals <= ITALY_LAT_MAX) & \
       (lon_vals >= ITALY_LON_MIN) & (lon_vals <= ITALY_LON_MAX)
lat_vals = lat_vals[mask]
lon_vals = lon_vals[mask]
y_train_filtered = y_train[mask]

print(f"Companies in map bounds: {len(y_train_filtered):,} | Bankruptcies: {y_train_filtered.sum():,}")

# ------------------------------------------------------------------------------
# 4️⃣  create pixel grids
def create_pixel_grids(lat_vals, lon_vals, y_vals, grid_res=GRID_RESOLUTION):
    """Create pixel grids for company density and bankruptcy rate"""

    # Create bins
    lon_edges = np.linspace(ITALY_LON_MIN, ITALY_LON_MAX, grid_res + 1)
    lat_edges = np.linspace(ITALY_LAT_MIN, ITALY_LAT_MAX, grid_res + 1)

    # Count total companies per pixel
    H_total, _, _ = np.histogram2d(lon_vals, lat_vals, bins=[lon_edges, lat_edges])

    # Count bankrupt companies per pixel
    bankrupt_mask = y_vals == 1
    H_bankrupt, _, _ = np.histogram2d(
        lon_vals[bankrupt_mask], lat_vals[bankrupt_mask],
        bins=[lon_edges, lat_edges]
    )

    # Calculate bankruptcy rate per pixel (as percentage)
    bankruptcy_rate = np.zeros_like(H_total)
    mask = H_total >= MIN_COMPANIES_PER_CELL
    bankruptcy_rate[mask] = (H_bankrupt[mask] / H_total[mask]) * 100

    # Create masked arrays
    bankruptcy_rate_masked = np.ma.masked_where(H_total < MIN_COMPANIES_PER_CELL, bankruptcy_rate)
    H_total_masked = np.ma.masked_where(H_total == 0, H_total)

    return bankruptcy_rate_masked.T, H_total_masked.T, lat_edges, lon_edges

# Create the grids
bankruptcy_rate, company_density, lat_edges, lon_edges = create_pixel_grids(
    lat_vals, lon_vals, y_train_filtered
)

# ------------------------------------------------------------------------------
# 5️⃣  create pixelated visualization
def draw_pixelated_map(file_stub):
    """Draw pixelated bankruptcy rate map with company density"""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    extent = [ITALY_LON_MIN, ITALY_LON_MAX, ITALY_LAT_MIN, ITALY_LAT_MAX]

    # LEFT PLOT: Company Density (Blue)
    # Use log scale for better visualization
    density_log = np.log10(company_density + 1)

    im1 = ax1.imshow(density_log,
                     extent=extent,
                     origin='lower',
                     cmap='Blues',
                     aspect='auto',
                     interpolation='nearest')  # 'nearest' for pixelated look

    # Add decision boundaries
    if lat_split_gains:
        sorted_lat = sorted(lat_split_gains.items(), key=lambda x: x[1], reverse=True)
        max_gain = max(lat_split_gains.values())
        for lat, gain in sorted_lat:
            if ITALY_LAT_MIN <= lat <= ITALY_LAT_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain)
                ax1.axhline(lat, color=BOUNDARY_COLOR_LAT, alpha=alpha, lw=1.5, zorder=3)

    if lon_split_gains:
        sorted_lon = sorted(lon_split_gains.items(), key=lambda x: x[1], reverse=True)
        max_gain = max(lon_split_gains.values())
        for lon, gain in sorted_lon:
            if ITALY_LON_MIN <= lon <= ITALY_LON_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain)
                ax1.axvline(lon, color=BOUNDARY_COLOR_LON, alpha=alpha, lw=1.5, zorder=3)

    ax1.set_xlim(ITALY_LON_MIN, ITALY_LON_MAX)
    ax1.set_ylim(ITALY_LAT_MIN, ITALY_LAT_MAX)
    ax1.set_xlabel("Longitude", fontsize=12)
    ax1.set_ylabel("Latitude", fontsize=12)
    ax1.set_title("Company Density (log scale)", fontsize=14, weight='bold')
    ax1.grid(True, alpha=0.3, color='white', linewidth=0.5)

    # Colorbar for density
    cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.8)
    cbar1.set_label('log10(Company Count + 1)', fontsize=10)

    # RIGHT PLOT: Bankruptcy Rate (Red gradient)
    # Custom colormap from white to dark red
    colors = ['#FFFFFF', '#FFE4E1', '#FFC0CB', '#FF6B6B', '#DC143C', '#8B0000']
    cmap_red = LinearSegmentedColormap.from_list('bankruptcy', colors)

    im2 = ax2.imshow(bankruptcy_rate,
                     extent=extent,
                     origin='lower',
                     cmap=cmap_red,
                     aspect='auto',
                     interpolation='nearest',  # 'nearest' for pixelated look
                     vmin=0,
                     vmax=np.percentile(bankruptcy_rate.compressed(), 95))  # Cap at 95th percentile

    # Add decision boundaries
    if lat_split_gains:
        for lat, gain in sorted_lat:
            if ITALY_LAT_MIN <= lat <= ITALY_LAT_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain)
                ax2.axhline(lat, color=BOUNDARY_COLOR_LAT, alpha=alpha, lw=1.5, zorder=3)

    if lon_split_gains:
        for lon, gain in sorted_lon:
            if ITALY_LON_MIN <= lon <= ITALY_LON_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain)
                ax2.axvline(lon, color=BOUNDARY_COLOR_LON, alpha=alpha, lw=1.5, zorder=3)

    ax2.set_xlim(ITALY_LON_MIN, ITALY_LON_MAX)
    ax2.set_ylim(ITALY_LAT_MIN, ITALY_LAT_MAX)
    ax2.set_xlabel("Longitude", fontsize=12)
    ax2.set_ylabel("Latitude", fontsize=12)
    ax2.set_title("Bankruptcy Rate (%)", fontsize=14, weight='bold')
    ax2.grid(True, alpha=0.3, color='white', linewidth=0.5)

    # Colorbar for bankruptcy rate
    cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8)
    cbar2.set_label('Bankruptcy Rate (%)', fontsize=10)

    # Overall title
    fig.suptitle(f'Pixelated Geographic Analysis ({GRID_RESOLUTION}×{GRID_RESOLUTION} grid)',
                fontsize=16, weight='bold')


    plt.tight_layout()
    save_fig(fig, file_stub)
    plt.show()

    return fig

# Create single combined pixelated map
def draw_single_pixelated_map(file_stub):
    """Draw single pixelated bankruptcy rate map"""

    fig, ax = plt.subplots(figsize=(14, 14))

    extent = [ITALY_LON_MIN, ITALY_LON_MAX, ITALY_LAT_MIN, ITALY_LAT_MAX]

    # Use white background
    ax.set_facecolor('white')

    # Plot bankruptcy rate pixels
    # Custom colormap from light yellow to dark red
    colors = ['#FFFFFF', '#FFE5E5', '#FFCCCC', '#FF9999', '#FF6666', '#FF3333', '#CC0000', '#990000']
    cmap_rate = LinearSegmentedColormap.from_list('bankruptcy', colors)

    im = ax.imshow(bankruptcy_rate,
                   extent=extent,
                   origin='lower',
                   cmap=cmap_rate,
                   aspect='auto',
                   interpolation='nearest',  # Pixelated look
                   vmin=0,
                   vmax=np.percentile(bankruptcy_rate.compressed(), 95) if bankruptcy_rate.compressed().size > 0 else 1)

    # Add decision boundaries
    if lat_split_gains:
        sorted_lat = sorted(lat_split_gains.items(), key=lambda x: x[1], reverse=True)
        max_gain_lat = max(lat_split_gains.values())
        for lat, gain in sorted_lat:
            if ITALY_LAT_MIN <= lat <= ITALY_LAT_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain_lat)
                ax.axhline(lat, color=BOUNDARY_COLOR_LAT, alpha=alpha, lw=1.5, zorder=3)

    if lon_split_gains:
        sorted_lon = sorted(lon_split_gains.items(), key=lambda x: x[1], reverse=True)
        max_gain_lon = max(lon_split_gains.values())
        for lon, gain in sorted_lon:
            if ITALY_LON_MIN <= lon <= ITALY_LON_MAX:
                alpha = min(1.0, 0.1 + 0.9 * gain / max_gain_lon)
                ax.axvline(lon, color=BOUNDARY_COLOR_LON, alpha=alpha, lw=1.5, zorder=3)

    ax.set_xlim(ITALY_LON_MIN, ITALY_LON_MAX)
    ax.set_ylim(ITALY_LAT_MIN, ITALY_LAT_MAX)
    ax.set_xlabel("Longitude", fontsize=14, weight='bold')
    ax.set_ylabel("Latitude", fontsize=14, weight='bold')
    ax.set_title(f"Pixelated Bankruptcy Rate Map ({GRID_RESOLUTION}×{GRID_RESOLUTION} grid)",
                fontsize=16, weight='bold', pad=20)

    # Add grid
    ax.grid(True, alpha=0.3, color='gray', linewidth=0.5)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label('Bankruptcy Rate (%)', fontsize=12, weight='bold')



    plt.tight_layout()
    save_fig(fig, file_stub)
    plt.show()

    return fig

# ------------------------------------------------------------------------------
# 6️⃣  create visualizations
print("\nGenerating pixelated maps...")

# Create combined view (density + rate)
fig1 = draw_pixelated_map("bankruptcy_pixelated_combined")

# Create single bankruptcy rate map
fig2 = draw_single_pixelated_map("bankruptcy_pixelated_rate")

print("\n" + "="*80)
print("✅ Pixelated map analysis complete!")
print(f"Grid resolution: {GRID_RESOLUTION}×{GRID_RESOLUTION}")
print(f"Each pixel shows the bankruptcy rate for that geographic area")
print("="*80)

In [ ]:
#@title 📊 Sampling Visualization

print("📊 SAMPLING VISUALIZATION")
print("=" * 80)

# Get the best sampling ratio (both models used the same ratio in your results)
best_sampling_ratio = results_df.iloc[0]['Sampling_Ratio']  # 0.3 in your case

# Calculate original numbers
original_bankruptcy_rate = y_train.mean()
original_n_bankrupt = int(y_train.sum())
original_n_healthy = len(y_train) - original_n_bankrupt
original_n_total = len(y_train)

# Calculate sampled numbers based on the best ratio
if best_sampling_ratio == 1.0:
    sampled_n_healthy = original_n_healthy
else:
    sampled_n_healthy = int(original_n_healthy * best_sampling_ratio)

sampled_n_bankrupt = original_n_bankrupt  # Keep all bankruptcies
sampled_n_total = sampled_n_healthy + sampled_n_bankrupt
sampled_bankruptcy_rate = sampled_n_bankrupt / sampled_n_total

print(f"Original: {original_n_total:,} samples ({original_bankruptcy_rate:.2%} bankruptcy rate)")
print(f"After sampling: {sampled_n_total:,} samples ({sampled_bankruptcy_rate:.2%} bankruptcy rate)")
print(f"Sampling ratio used: {best_sampling_ratio} (kept {best_sampling_ratio*100:.0f}% of healthy companies)")

#================================================================================
# MODERN VISUALIZATION
#================================================================================

# Set up modern matte color palette
colors = {
    'healthy': '#7FCDBB',     # Muted mint green
    'bankruptcy': '#FC8D59',  # Muted orange
    'original': '#91BFDB',    # Muted sky blue
    'sampled': '#FEE090'      # Muted light yellow
}
outline = '#555555'

# Create figure with better spacing
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Modern style settings
plt.style.use('seaborn-v0_8-whitegrid')
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E0E0E0')
    ax.spines['bottom'].set_color('#E0E0E0')
    ax.tick_params(colors='#666666')
    ax.grid(axis='y', alpha=0.3, linestyle='--')

# 1. Original Data Distribution - Absolute values
bars1_healthy = axes[0].bar(['Healthy'], [original_n_healthy],
                           color=colors['healthy'], alpha=0.9,
                           edgecolor=outline, linewidth=2)
bars1_bankruptcy = axes[0].bar(['Bankruptcy'], [original_n_bankrupt],
                              color=colors['bankruptcy'], alpha=0.9,
                              edgecolor=outline, linewidth=2)

axes[0].set_title('Original Training Data', fontsize=14, fontweight='600', pad=15, color='#2C3E50')
axes[0].set_ylabel('Number of Companies', fontsize=11, color='#555555')

# Add value labels above bars
axes[0].text(0, original_n_healthy + original_n_healthy * 0.02,
            f'{original_n_healthy:,}',
            ha='center', va='bottom', fontsize=10, color='black', fontweight='600')
axes[0].text(1, original_n_bankrupt + original_n_healthy * 0.02,
            f'{original_n_bankrupt:,}',
            ha='center', va='bottom', fontsize=10, color='black', fontweight='600')

# Set y-limit to show both bars properly
axes[0].set_ylim(0, original_n_healthy * 1.1)

# 2. After Sampling - Same style
bars2_healthy = axes[1].bar(['Healthy'], [sampled_n_healthy],
                           color=colors['healthy'], alpha=0.9,
                           edgecolor=outline, linewidth=2)
bars2_bankruptcy = axes[1].bar(['Bankruptcy'], [sampled_n_bankrupt],
                              color=colors['bankruptcy'], alpha=0.9,
                              edgecolor=outline, linewidth=2)

axes[1].set_title(f'After Sampling ({best_sampling_ratio*100:.0f}% healthy kept)',
                  fontsize=14, fontweight='600', pad=15, color='#2C3E50')
axes[1].set_ylabel('Number of Companies', fontsize=11, color='#555555')

# Add value labels above bars
axes[1].text(0, sampled_n_healthy + max(sampled_n_healthy, sampled_n_bankrupt) * 0.03,
            f'{sampled_n_healthy:,}', ha='center', va='bottom',
            fontsize=10, color='black', fontweight='600')
axes[1].text(1, sampled_n_bankrupt + max(sampled_n_healthy, sampled_n_bankrupt) * 0.03,
            f'{sampled_n_bankrupt:,}', ha='center', va='bottom',
            fontsize=10, color='black', fontweight='600')

# Set reasonable y-limit
axes[1].set_ylim(0, max(sampled_n_healthy, sampled_n_bankrupt) * 1.15)

# 3. Bankruptcy Rate Comparison
x_pos = [0, 1]
rates = [original_bankruptcy_rate * 100, sampled_bankruptcy_rate * 100]
bars3 = axes[2].bar(x_pos, rates,
                    color=[colors['original'], colors['sampled']],
                    edgecolor=outline,
                    linewidth=2,
                    alpha=0.9,
                    width=0.6)

axes[2].set_title('Bankruptcy Rate Change', fontsize=14, fontweight='600', pad=15, color='#2C3E50')
axes[2].set_ylabel('Bankruptcy Rate (%)', fontsize=11, color='#555555')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(['Original', 'Rebalanced'], fontsize=11)
axes[2].set_ylim(0, max(rates) * 1.4)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars3, rates)):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(rates) * 0.03,
                f'{val:.2f}%',
                ha='center', va='bottom', fontsize=11, fontweight='600', color='black')

# Add change factor as simple text
if len(rates) == 2 and rates[0] > 0:
    change_factor = rates[1] / rates[0]
    axes[2].text(0.5, max(rates) * 1.15,
                f'{change_factor:.1f}x increase',
                ha='center', va='center', fontsize=11,
                color='#E74C3C', fontweight='600',
                bbox=dict(boxstyle="round,pad=0.4", facecolor='#FFEAA7',
                         edgecolor='#E74C3C', alpha=0.3, linewidth=1.5))

plt.tight_layout()
save_fig(fig, 'sampling_visualization')
plt.show()

# Additional info
print(f"\nScale pos weight after sampling: {results_df.iloc[0]['Scale_Pos_Weight']:.2f}")

In [ ]:
#@title 📊 ROC Curves

print("📊 ROC CURVES")
print("=" * 80)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot for each model
for model_name in ['XGBoost', 'LightGBM']:
    if model_name in predictions:
        fpr, tpr, _ = roc_curve(y_test, predictions[model_name]['test'])
        auc_score = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=3, color=model_colors[model_name],
                label=f'{model_name} (AUC = {auc_score:.3f})', alpha=0.8)

# Random classifier
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='Random Classifier')

# Styling
ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
ax.set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=12, frameon=True, shadow=True)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

# Add diagonal fill
ax.fill_between([0, 1], [0, 1], alpha=0.1, color='gray')

plt.tight_layout()
save_fig(fig, 'roc_curves')
plt.show()

In [ ]:
#@title 🔬 Parameter Evolution Plots (Optuna)

print("🔬 PARAMETER EVOLUTION PLOTS")
print("=" * 80)

if HAS_OPTUNA_VIS and 'study_xgb' in globals() and 'study_lgb' in globals():
    studies = {'XGBoost': study_xgb, 'LightGBM': study_lgb}

    for model_name, study in studies.items():
        if study is None or len(study.trials) < 2:
            print(f"⚠️ Skipping {model_name} - insufficient trials")
            continue

        print(f"\nProcessing {model_name}...")

        # Parallel Coordinate Plot
        try:
            fig_parallel = optuna_vis.plot_parallel_coordinate(study)
            fig_parallel.update_layout(
                title=f'<b>{model_name}</b> - Parameter Evolution (Parallel Coordinates)',
                height=600,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_parallel, f'param_evolution_parallel_{model_name.lower()}')
            print(f"  ✅ Saved parallel coordinate plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parallel coordinate plot: {str(e)}")

        # Slice Plot
        try:
            fig_slice = optuna_vis.plot_slice(study)
            fig_slice.update_layout(
                title=f'<b>{model_name}</b> - Parameter Importance (Slice Plot)',
                height=800,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_slice, f'param_evolution_slice_{model_name.lower()}')
            print(f"  ✅ Saved slice plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create slice plot: {str(e)}")

        # Optimization History
        try:
            fig_history = optuna_vis.plot_optimization_history(study)
            fig_history.update_layout(
                title=f'<b>{model_name}</b> - Optimization History',
                height=500,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_history, f'optimization_history_{model_name.lower()}')
            print(f"  ✅ Saved optimization history for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create optimization history: {str(e)}")

        # Parameter Importances
        try:
            fig_importance = optuna_vis.plot_param_importances(study)
            fig_importance.update_layout(
                title=f'<b>{model_name}</b> - Hyperparameter Importances',
                height=600,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_importance, f'param_importances_{model_name.lower()}')
            print(f"  ✅ Saved parameter importances for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parameter importances: {str(e)}")

else:
    print("⚠️ Optuna visualization not available or studies not found")

In [ ]:
#@title 🔬 Parameter Evolution Plots - Top perc_num Values Only (Optuna)
perc_num= 99
perc_num_filt=100-perc_num
print("🔬 PARAMETER EVOLUTION PLOTS - TOP perc_num VALUES ONLY")
print("=" * 80)

import optuna
from optuna import visualization as optuna_vis
import numpy as np

def filter_top_trials(study, percentile=perc_num_filt):
    """Filter study to keep only trials above the specified percentile of objective values"""
    # Get completed trials only
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

    if len(completed_trials) < 2:
        return study  # Return original if too few trials

    # Get objective values
    values = [t.value for t in completed_trials]

    # Calculate threshold (percentile)
    threshold = np.percentile(values, percentile)

    # Create new study with same settings
    filtered_study = optuna.create_study(direction=study.direction)

    # Add only trials above threshold
    for trial in completed_trials:
        if trial.value >= threshold:
            # Add trial to filtered study
            filtered_study.add_trial(trial)

    print(f"  Filtered from {len(completed_trials)} to {len(filtered_study.trials)} trials (top {100-percentile}%)")
    print(f"  Threshold value: {threshold:.4f}")

    return filtered_study

if 'study_xgb' in globals() and 'study_lgb' in globals():
    studies = {'XGBoost': study_xgb, 'LightGBM': study_lgb}

    for model_name, study in studies.items():
        if study is None or len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]) < 2:
            print(f"⚠️ Skipping {model_name} - insufficient trials")
            continue

        print(f"\nProcessing {model_name}...")

        # Filter to top perc_num of trials
        filtered_study = filter_top_trials(study, percentile=perc_num_filt)

        # Parallel Coordinate Plot
        try:
            print(f"  Creating parallel coordinate plot...")
            fig_parallel = optuna_vis.plot_parallel_coordinate(filtered_study)
            fig_parallel.update_layout(
                title=f'<b>{model_name}</b> - Parameter Evolution (Top {perc_num}% - Parallel Coordinates)',
                height=600,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_parallel, f'param_evolution_parallel_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved parallel coordinate plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parallel coordinate plot: {str(e)}")

        # Slice Plot
        try:
            print(f"  Creating slice plot...")
            fig_slice = optuna_vis.plot_slice(filtered_study)
            fig_slice.update_layout(
                title=f'<b>{model_name}</b> - Parameter Importance (Top {perc_num}% - Slice Plot)',
                height=800,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_slice, f'param_evolution_slice_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved slice plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create slice plot: {str(e)}")

        # Optimization History - only filtered version
        try:
            print(f"  Creating optimization history...")
            fig_history_filtered = optuna_vis.plot_optimization_history(filtered_study)
            fig_history_filtered.update_layout(
                title=f'<b>{model_name}</b> - Top {perc_num}% Optimization History',
                height=500,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_history_filtered, f'optimization_history_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved optimization history for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create optimization history: {str(e)}")

        # Parameter Importances
        try:
            print(f"  Creating parameter importances...")
            fig_importance = optuna_vis.plot_param_importances(filtered_study)
            fig_importance.update_layout(
                title=f'<b>{model_name}</b> - Hyperparameter Importances (Top {perc_num}%)',
                height=600,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_importance, f'param_importances_top50_{model_name.lower()}')
            print(f"  ✅ Saved parameter importances for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parameter importances: {str(e)}")

else:
    print("⚠️ Optuna studies not found")

print("\n" + "="*80)
print("Visualization complete!")

In [ ]:
#@title 🔬 Parameter Evolution Plots - Top perc_num Values Only (Optuna)
perc_num= 99
perc_num_filt=100-perc_num
print("🔬 PARAMETER EVOLUTION PLOTS - TOP perc_num VALUES ONLY")
print("=" * 80)

import optuna
from optuna import visualization as optuna_vis
import numpy as np

def filter_top_trials(study, percentile=perc_num_filt):
    """Filter study to keep only trials above the specified percentile of objective values"""
    # Get completed trials only
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

    if len(completed_trials) < 2:
        return study  # Return original if too few trials

    # Get objective values
    values = [t.value for t in completed_trials]

    # Calculate threshold (percentile)
    threshold = np.percentile(values, percentile)

    # Create new study with same settings
    filtered_study = optuna.create_study(direction=study.direction)

    # Add only trials above threshold
    for trial in completed_trials:
        if trial.value >= threshold:
            # Add trial to filtered study
            filtered_study.add_trial(trial)

    print(f"  Filtered from {len(completed_trials)} to {len(filtered_study.trials)} trials (top {100-percentile}%)")
    print(f"  Threshold value: {threshold:.4f}")

    return filtered_study

if 'study_xgb' in globals() and 'study_lgb' in globals():
    studies = {'XGBoost': study_xgb, 'LightGBM': study_lgb}

    for model_name, study in studies.items():
        if study is None or len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]) < 2:
            print(f"⚠️ Skipping {model_name} - insufficient trials")
            continue

        print(f"\nProcessing {model_name}...")

        # Filter to top perc_num of trials
        filtered_study = filter_top_trials(study, percentile=perc_num_filt)

        # Parallel Coordinate Plot
        try:
            print(f"  Creating parallel coordinate plot...")
            fig_parallel = optuna_vis.plot_parallel_coordinate(filtered_study)
            fig_parallel.update_layout(
                title=f'<b>{model_name}</b> - Parameter Evolution (Top {perc_num}% - Parallel Coordinates)',
                height=600,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_parallel, f'param_evolution_parallel_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved parallel coordinate plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parallel coordinate plot: {str(e)}")

        # Slice Plot
        try:
            print(f"  Creating slice plot...")
            fig_slice = optuna_vis.plot_slice(filtered_study)
            fig_slice.update_layout(
                title=f'<b>{model_name}</b> - Parameter Importance (Top {perc_num}% - Slice Plot)',
                height=800,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_slice, f'param_evolution_slice_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved slice plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create slice plot: {str(e)}")

        # Contour Plot with optimizations and timeout
        try:
            print(f"  Creating contour plot...")

            # Further reduce trials for contour plot if still too many
            contour_study = filtered_study
            if len(filtered_study.trials) > 100:
                print(f"    Reducing trials from {len(filtered_study.trials)} to top 100 for contour plot...")
                contour_study = filter_top_trials(filtered_study, percentile=90)  # Top 10% of filtered trials

            # Get parameter names to limit contour plot if too many
            param_names = list(contour_study.best_params.keys())

            if len(param_names) > 4:  # Reduced from 6 to 4 for faster computation
                # If too many parameters, show only most important ones
                try:
                    importance = optuna.importance.get_param_importances(contour_study)
                    top_params = list(importance.keys())[:4]  # Top 4 most important
                    print(f"    Too many parameters ({len(param_names)}), showing top 4: {top_params}")
                    fig_contour = optuna_vis.plot_contour(contour_study, params=top_params)
                except:
                    # Fallback: show first 4 parameters
                    top_params = param_names[:4]
                    print(f"    Showing first 4 parameters: {top_params}")
                    fig_contour = optuna_vis.plot_contour(contour_study, params=top_params)
            else:
                fig_contour = optuna_vis.plot_contour(contour_study)

            fig_contour.update_layout(
                title=f'<b>{model_name}</b> - Parameter Interactions (Top {perc_num}% - Contour Plot)',
                height=700,
                width=1200,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_contour, f'param_contour_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved contour plot for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create contour plot: {str(e)}")
            print(f"    Skipping contour plot and continuing...")

        # Optimization History - only filtered version
        try:
            print(f"  Creating optimization history...")
            fig_history_filtered = optuna_vis.plot_optimization_history(filtered_study)
            fig_history_filtered.update_layout(
                title=f'<b>{model_name}</b> - Top {perc_num}% Optimization History',
                height=500,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_history_filtered, f'optimization_history_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved optimization history for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create optimization history: {str(e)}")

        # Parameter Importances
        try:
            print(f"  Creating parameter importances...")
            fig_importance = optuna_vis.plot_param_importances(filtered_study)
            fig_importance.update_layout(
                title=f'<b>{model_name}</b> - Hyperparameter Importances (Top {perc_num}%)',
                height=600,
                width=1000,
                template='plotly_white',
                font=dict(size=12)
            )
            save_plotly_fig(fig_importance, f'param_importances_top{perc_num}_{model_name.lower()}')
            print(f"  ✅ Saved parameter importances for {model_name}")
        except Exception as e:
            print(f"  ⚠️ Could not create parameter importances: {str(e)}")

else:
    print("⚠️ Optuna studies not found")

print("\n" + "="*80)
print("Visualization complete!")

In [ ]:
#@title 📊 Precision-Recall Curves

print("📊 PRECISION-RECALL CURVES")
print("=" * 80)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Calculate baseline (random classifier precision)
baseline = y_test.mean()

# Plot for each model
for model_name in ['XGBoost', 'LightGBM']:
    if model_name in predictions:
        precision, recall, _ = precision_recall_curve(y_test, predictions[model_name]['test'])
        pr_auc = average_precision_score(y_test, predictions[model_name]['test'])
        ax.plot(recall, precision, linewidth=3, color=model_colors[model_name],
                label=f'{model_name} (AUC = {pr_auc:.3f})', alpha=0.8)

# Baseline
ax.axhline(y=baseline, color='red', linestyle='--', linewidth=2,
           label=f'Random Classifier (Precision = {baseline:.3f})')

# Fill area under baseline
ax.fill_between([0, 1], 0, baseline, alpha=0.1, color='red')

# Styling
ax.set_xlabel('Recall', fontsize=14, fontweight='bold')
ax.set_ylabel('Precision', fontsize=14, fontweight='bold')
ax.set_title('Precision-Recall Curves', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', fontsize=12, frameon=True, shadow=True)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
save_fig(fig, 'precision_recall_curves')
plt.show()

In [ ]:
#@title 📊 Confusion Matrices at Business Thresholds

print("📊 CONFUSION MATRICES AT BUSINESS THRESHOLDS")
print("=" * 80)

# Define recall targets
recall_targets = [0.90, 0.95]

for recall_target in recall_targets:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f'Confusion Matrices at {int(recall_target*100)}% Recall Target',
                 fontsize=18, fontweight='bold', y=1.02)

    for idx, model_name in enumerate(['XGBoost', 'LightGBM']):
        if model_name not in predictions:
            continue

        ax = axes[idx]

        # Get predictions
        y_pred_proba = predictions[model_name]['test']

        # Find threshold for target recall
        precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

        # Find the threshold that gives us at least target_recall
        valid_indices = np.where(recall >= recall_target)[0]

        if len(valid_indices) > 0:
            idx_threshold = valid_indices[-1]
            if idx_threshold < len(thresholds):
                optimal_threshold = thresholds[idx_threshold]
            else:
                optimal_threshold = thresholds[-1] - 0.0001

            # Make predictions with this threshold
            y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)

            # Calculate confusion matrix
            cm = confusion_matrix(y_test, y_pred_binary)
            cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
            # Plot confusion matrix
            sns.heatmap(cm_pct, annot=False, fmt='d', cmap='Blues',
                       cbar=False, ax=ax, square=True,
                       xticklabels=['Healthy', 'Failure'],
                       yticklabels=['Healthy', 'Failure'])

            # Add custom annotations with counts and percentages
            for i in range(2):
                for j in range(2):
                    count = cm[i, j]
                    total_row = cm[i, :].sum()
                    percentage = (count / total_row) * 100 if total_row > 0 else 0

                    # Text color based on background
                    text_color = 'white' if cm_pct[i, j] > 50 else 'black'

                    ax.text(j + 0.5, i + 0.5, f'{count:,}\n({percentage:.1f}%)',
                           ha='center', va='center', fontsize=14,
                           fontweight='bold', color=text_color)

            # Calculate metrics
            tn, fp, fn, tp = cm.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            precision_score = tp / (tp + fp) if (tp + fp) > 0 else 0

            # Set title with metrics
            ax.set_title(f'{model_name}\nThreshold: {optimal_threshold:.3f}\n')

            # Styling
            ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
            ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
            ax.tick_params(axis='both', which='major', labelsize=11)

    plt.tight_layout()
    save_fig(fig, f'confusion_matrices_{int(recall_target*100)}pct_recall')
    plt.show()

In [ ]:
#@title 📊 Confusion Matrices at Business Thresholds

print("📊 CONFUSION MATRICES AT BUSINESS THRESHOLDS")
print("=" * 80)

# Define recall targets
recall_targets = [0.90, 0.95]

# --- CHANGE: Loop through each model to create a separate figure for each ---
for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        print(f"⚠️ Skipping {model_name} - predictions not found.")
        continue

    # Create a new figure with two subplots for the current model
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # --- CHANGE: Update the main title to include the model name ---
    fig.suptitle(f'{model_name} - Confusion Matrices at 90% & 95% Recall Targets',
                 fontsize=18, fontweight='bold', y=1.02)

    # Loop through the recall targets to create a subplot for each
    for idx, recall_target in enumerate(recall_targets):
        ax = axes[idx]

        # Get predictions for the current model
        y_pred_proba = predictions[model_name]['test']

        # Find threshold for the current target recall
        precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
        valid_indices = np.where(recall >= recall_target)[0]

        if len(valid_indices) > 0:
            idx_threshold = valid_indices[-1]
            if idx_threshold < len(thresholds):
                optimal_threshold = thresholds[idx_threshold]
            else:
                optimal_threshold = thresholds[-1] - 0.0001

            # Make predictions with this threshold
            y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)

            # Calculate and plot confusion matrix
            cm = confusion_matrix(y_test, y_pred_binary)
            cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

            sns.heatmap(cm_pct, annot=False, fmt='d', cmap='Blues',
                        cbar=False, ax=ax, square=True,
                        xticklabels=['Healthy', 'Failure'],
                        yticklabels=['Healthy', 'Failure'])

            # Add custom annotations with counts and percentages
            for i in range(2):
                for j in range(2):
                    count = cm[i, j]
                    total_row = cm[i, :].sum()
                    percentage = (count / total_row) * 100 if total_row > 0 else 0
                    text_color = 'white' if cm_pct[i, j] > 50 else 'black'

                    ax.text(j + 0.5, i + 0.5, f'{count:,}\n({percentage:.1f}%)',
                            ha='center', va='center', fontsize=14,
                            fontweight='bold', color=text_color)

            # --- CHANGE: Update subplot title for the specific recall target ---
            ax.set_title(f'At {int(recall_target*100)}% Recall Target\nThreshold: {optimal_threshold:.3f}',
                         fontsize=14)

            # Styling
            ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
            ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
            ax.tick_params(axis='both', which='major', labelsize=11)

    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make space for suptitle

    # --- CHANGE: Update filename to be model-specific ---
    save_fig(fig, f'confusion_matrices_{model_name.lower()}')
    plt.show()

In [ ]:
#@title 🔍 Feature Importance - Split Counts

print("🔍 FEATURE IMPORTANCE - SPLIT COUNTS")
print("=" * 80)

# Process each model
for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in models:
        continue

    print(f"\n{model_name}:")

    try:
        model = models[model_name]

        if model_name == 'XGBoost':
            # Get XGBoost booster
            booster = model.get_booster()

            # Get split counts
            importance_dict = booster.get_score(importance_type='weight')

            # Map to feature names
            feature_importance = []
            for i, fname in enumerate(feature_names):
                fid = f'f{i}'
                importance = importance_dict.get(fid, 0)
                feature_importance.append({'feature': fname, 'split_count': importance})

        elif model_name == 'LightGBM':
            # For LightGBM, use the booster to get split counts
            if hasattr(models['LightGBM'], 'booster_'):
                booster = models['LightGBM'].booster_
                importance_dict = booster.feature_importance(importance_type='split')

                feature_importance = []
                for i, (fname, importance) in enumerate(zip(feature_names, importance_dict)):
                    feature_importance.append({'feature': fname, 'split_count': importance})
            else:
                print(f"  ⚠️ Cannot access LightGBM booster")
                continue

        # Create DataFrame and sort
        fi_df = pd.DataFrame(feature_importance).sort_values('split_count', ascending=False)

        # Plot top features
        TOP_N = 8
        top_features = fi_df.head(TOP_N)

        fig, ax = plt.subplots(figsize=(10, 8))

        # Create horizontal bar chart
        y_pos = np.arange(len(top_features))
        ax.barh(y_pos, top_features['split_count'].values,
                color=model_colors[model_name], alpha=0.8, edgecolor='black', linewidth=1)

        # Customize
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_features['feature'].values)
        ax.invert_yaxis()
        ax.set_xlabel('Number of Splits', fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name} - Top {TOP_N} Features by Split Count',
                      fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        # Add value labels
        for i, (idx, row) in enumerate(top_features.iterrows()):
            ax.text(row['split_count'] + max(top_features['split_count'])*0.01, i,
                    f"{int(row['split_count'])}", va='center', fontsize=9)

        plt.tight_layout()
        save_fig(fig, f'feature_importance_splits_{model_name.lower()}')
        plt.show()

        # Print top 10
        print("\nTop 10 features by split count:")
        print(fi_df.head(10).to_string(index=False))

    except Exception as e:
        print(f"  ⚠️ Error processing {model_name}: {str(e)}")

In [ ]:
#@title 🔍 Feature Importance - Gain

print("🔍 FEATURE IMPORTANCE - GAIN")
print("=" * 80)

# Process each model
for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in models:
        continue

    print(f"\n{model_name}:")

    try:
        model = models[model_name]

        # Get feature importances (gain-based)
        importances = model.feature_importances_

        # Create DataFrame
        fi_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importances
        }).sort_values('importance', ascending=False)

        # Normalize importances to sum to 1
        fi_df['importance_normalized'] = fi_df['importance'] / fi_df['importance'].sum()

        # Plot top features
        TOP_N = 8
        top_features = fi_df.head(TOP_N)

        fig, ax = plt.subplots(figsize=(10, 8))

        # Create horizontal bar chart
        y_pos = np.arange(len(top_features))
        ax.barh(y_pos, top_features['importance_normalized'].values,
                color=model_colors[model_name], alpha=0.8, edgecolor='black', linewidth=1)

        # Customize
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_features['feature'].values)
        ax.invert_yaxis()
        ax.set_xlabel('Normalized Importance (Gain)', fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name} - Top {TOP_N} Features by Gain',
                     fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        # Add value labels
        for i, (idx, row) in enumerate(top_features.iterrows()):
            ax.text(row['importance_normalized'] + max(top_features['importance_normalized'])*0.01, i,
                   f"{row['importance_normalized']:.3f}", va='center', fontsize=9)

        # Add cumulative importance line
        plt.tight_layout()
        save_fig(fig, f'feature_importance_gain_{model_name.lower()}')
        plt.show()

        # Print top 10 with cumulative importance
        print("\nTop 10 features by gain:")
        top10 = fi_df.head(10).copy()
        top10['cumulative'] = top10['importance_normalized'].cumsum()
        print(top10[['feature', 'importance_normalized', 'cumulative']].to_string(index=False))
        print(f"\nTop 10 features explain {top10['cumulative'].iloc[-1]:.1%} of total importance")

    except Exception as e:
        print(f"  ⚠️ Error processing {model_name}: {str(e)}")

In [ ]:
#@title 📊 Score Distribution Analysis

print("📊 SCORE DISTRIBUTION ANALYSIS")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        continue

    # Get predictions
    y_pred = predictions[model_name]['test']

    # Separate scores by class
    scores_healthy = y_pred[y_test == 0]
    scores_bankrupt = y_pred[y_test == 1]

    # 1. Score distribution histogram with more bins (narrower bars)
    fig1, ax1 = plt.subplots(figsize=(10, 8))

    # Increase number of bins for narrower bars
    bins = np.linspace(0, 1, 100)  # Changed from 50 to 100 bins

    ax1.hist(scores_healthy, bins=bins, alpha=0.6, label='Healthy',
             color='#2ECC71', density=True, edgecolor='black', linewidth=0.5)
    ax1.hist(scores_bankrupt, bins=bins, alpha=0.7, label='Bankrupt',
             color='#E74C3C', density=True, edgecolor='black', linewidth=0.5)

    # Add vertical lines for means
    ax1.axvline(scores_healthy.mean(), color='#27AE60', linestyle='--',
                linewidth=2, label=f'Healthy mean: {scores_healthy.mean():.3f}')
    ax1.axvline(scores_bankrupt.mean(), color='#C0392B', linestyle='--',
                linewidth=2, label=f'Bankrupt mean: {scores_bankrupt.mean():.3f}')

    ax1.set_xlabel('Predicted Probability', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax1.set_title(f'{model_name} - Score Distribution by Class', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    plt.tight_layout()
    save_fig(fig1, f'score_distribution_histogram_{model_name.lower()}')
    plt.show()

    # 2. Box plot with scatter overlay
    fig2, ax2 = plt.subplots(figsize=(10, 8))

    # Prepare data
    data_to_plot = [scores_healthy, scores_bankrupt]
    positions = [1, 2]

    # Create box plot
    box_plot = ax2.boxplot(data_to_plot,
                          positions=positions,
                          widths=0.6,
                          patch_artist=True,
                          showmeans=False,
                          showfliers=False)  # Hide outliers as we'll show all points

    # Customize box colors
    colors_box = ['#2ECC71', '#E74C3C']
    for patch, color in zip(box_plot['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
        patch.set_edgecolor('#2C3E50')
        patch.set_linewidth(2)

    # Customize whiskers and medians
    for element in ['whiskers', 'caps']:
        plt.setp(box_plot[element], color='#2C3E50', linewidth=1.5)
    plt.setp(box_plot['medians'], color='#2C3E50', linewidth=3)

    # Add scatter points with jitter
    np.random.seed(42)
    jitter_strength = 0.15

    # Sample points if too many (for visibility)
    max_points = 1000
    if len(scores_healthy) > max_points:
        healthy_sample = np.random.choice(scores_healthy, max_points, replace=False)
    else:
        healthy_sample = scores_healthy

    # Plot healthy scores
    x_healthy = np.ones(len(healthy_sample)) + np.random.normal(0, jitter_strength, len(healthy_sample))
    ax2.scatter(x_healthy, healthy_sample, alpha=0.3, color='#27AE60', s=20, edgecolors='none')

    # Plot bankrupt scores (all of them since there are fewer)
    x_bankrupt = 2 * np.ones(len(scores_bankrupt)) + np.random.normal(0, jitter_strength, len(scores_bankrupt))
    ax2.scatter(x_bankrupt, scores_bankrupt, alpha=0.6, color='#C0392B', s=30, edgecolors='black', linewidth=0.5)

    # Customize axes
    ax2.set_xticks(positions)
    ax2.set_xticklabels(['Healthy', 'Bankrupt'], fontsize=14, fontweight='bold')
    ax2.set_ylabel('Predicted Probability', fontsize=14, fontweight='bold')
    ax2.set_title(f'{model_name} - Score Distribution Box Plot', fontsize=16, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    save_fig(fig2, f'score_distribution_boxplot_{model_name.lower()}')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score

def find_threshold_for_recall(y_true, y_pred_proba, target_recall):
    """Find the threshold that achieves the target recall"""
    # Sort predictions in descending order
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    sorted_labels = y_true[sorted_indices]
    sorted_probs = y_pred_proba[sorted_indices]

    # Total positive samples
    total_positives = np.sum(y_true)

    if total_positives == 0:
        return 0.0

    # Find threshold for target recall
    target_tp = int(np.ceil(target_recall * total_positives))

    # Find the threshold where we have exactly target_tp true positives
    if target_tp > total_positives:
        return 0.0

    # Count true positives as we go down the sorted list
    cumulative_tp = np.cumsum(sorted_labels)

    # Find where we reach the target number of true positives
    threshold_idx = np.where(cumulative_tp >= target_tp)[0]

    if len(threshold_idx) == 0:
        return 0.0

    return sorted_probs[threshold_idx[0]]

print("📊 SCORE DISTRIBUTION WITH RECALL THRESHOLDS")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        continue

    # Get predictions
    y_pred = predictions[model_name]['test']

    # Separate scores by class
    scores_healthy = y_pred[y_test == 0]
    scores_bankrupt = y_pred[y_test == 1]

    # Find thresholds for target recalls
    target_recalls = [0.90, 0.95, 0.99]
    thresholds = {}

    for target_recall in target_recalls:
        threshold = find_threshold_for_recall(y_test, y_pred, target_recall)
        thresholds[target_recall] = threshold

        # Verify the actual recall achieved
        y_pred_binary = (y_pred >= threshold).astype(int)
        actual_recall = recall_score(y_test, y_pred_binary)
        print(f"{model_name} - Target {target_recall*100:.0f}% recall: threshold = {threshold:.4f}, actual recall = {actual_recall:.3f}")

    # Create box plot with recall thresholds
    fig, ax = plt.subplots(figsize=(12, 8))

    # Prepare data
    data_to_plot = [scores_healthy, scores_bankrupt]
    positions = [1, 2]

    # Create box plot
    box_plot = ax.boxplot(data_to_plot,
                          positions=positions,
                          widths=0.6,
                          patch_artist=True,
                          showmeans=False,
                          showfliers=False)

    # Customize box colors
    colors_box = ['#2ECC71', '#E74C3C']
    for patch, color in zip(box_plot['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
        patch.set_edgecolor('#2C3E50')
        patch.set_linewidth(2)

    # Customize whiskers and medians
    for element in ['whiskers', 'caps']:
        plt.setp(box_plot[element], color='#2C3E50', linewidth=1.5)
    plt.setp(box_plot['medians'], color='#2C3E50', linewidth=3)

    # Add scatter points with jitter
    np.random.seed(42)
    jitter_strength = 0.15

    # Sample points if too many (for visibility)
    max_points = 20000
    if len(scores_healthy) > max_points:
        healthy_sample = np.random.choice(scores_healthy, max_points, replace=False)
    else:
        healthy_sample = scores_healthy

    # Plot healthy scores
    x_healthy = np.ones(len(healthy_sample)) + np.random.normal(0, jitter_strength, len(healthy_sample))
    ax.scatter(x_healthy, healthy_sample, alpha=0.4, color='#27AE60', s=20, edgecolors='none')

    # Plot bankrupt scores (all of them since there are fewer)
    x_bankrupt = 2 * np.ones(len(scores_bankrupt)) + np.random.normal(0, jitter_strength, len(scores_bankrupt))
    ax.scatter(x_bankrupt, scores_bankrupt, alpha=0.6, color='#C0392B', s=30, edgecolors='black', linewidth=0.5)

    # Add horizontal lines for recall thresholds
    for target_recall, threshold in thresholds.items():
        # Draw horizontal dashed line in gray
        ax.axhline(y=threshold, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)

        # Add simple text label in gray, centered and above the line
        ax.text(0.5, threshold + 0.005, f'{target_recall*100:.0f}% Recall',
                ha='right', va='bottom', fontsize=10, color='gray', fontweight='bold')

    # Customize axes
    ax.set_xticks(positions)
    ax.set_xticklabels(['Healthy', 'Bankrupt'], fontsize=14, fontweight='bold')
    ax.set_ylabel('Predicted Probability', fontsize=14, fontweight='bold')
    ax.set_title(f'{model_name} - Score Distribution with Recall Thresholds', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    save_fig(fig, f'score_distribution_boxplot_with_thresholds_{model_name.lower()}')
    plt.show()

    print(f"✅ Saved boxplot with recall thresholds for {model_name}")
    print("-" * 60)

In [ ]:
#@title 📅 Temporal Performance Analysis

print("📅 TEMPORAL PERFORMANCE ANALYSIS")
print("=" * 80)

# Check if we have temporal data
temporal_analysis_possible = False

# Try to reconstruct temporal information
if 'df_train' in globals() and 'df_valid' in globals() and 'df_test' in globals():
    # We know the years from the data preparation
    train_years_used = train_years if 'train_years' in globals() else [2016, 2017, 2018]
    valid_years_used = valid_years if 'valid_years' in globals() else [2019]
    test_years_used = test_years if 'test_years' in globals() else [2020, 2021]

    # For test set, we need to determine which samples belong to which year
    # This is approximate since we don't have the exact year mapping after transformation
    n_test = len(y_test)

    # Assume roughly equal distribution between test years
    n_per_year = n_test // len(test_years_used)

    # Create approximate year assignments
    year_assignments = []
    for i, year in enumerate(test_years_used):
        if i < len(test_years_used) - 1:
            year_assignments.extend([year] * n_per_year)
        else:
            # Last year gets remaining samples
            year_assignments.extend([year] * (n_test - len(year_assignments)))

    temporal_analysis_possible = True
    print(f"✓ Approximating temporal analysis with test years: {test_years_used}")

if temporal_analysis_possible:
    # Calculate metrics by year
    yearly_metrics = []

    for year in test_years_used:
        year_mask = np.array([y == year for y in year_assignments])

        if year_mask.sum() > 0:
            for model_name in ['XGBoost', 'LightGBM']:
                if model_name in predictions:
                    y_pred_year = predictions[model_name]['test'][year_mask]
                    y_true_year = y_test.iloc[year_mask] if hasattr(y_test, 'iloc') else y_test[year_mask]

                    # Ensure we have both classes
                    if len(np.unique(y_true_year)) > 1:
                        yearly_metrics.append({
                            'Year': year,
                            'Model': model_name,
                            'N_Samples': len(y_true_year),
                            'N_Bankruptcies': int(np.sum(y_true_year)),
                            'Bankruptcy_Rate': np.mean(y_true_year),
                            'ROC_AUC': roc_auc_score(y_true_year, y_pred_year),
                            'PR_AUC': average_precision_score(y_true_year, y_pred_year)
                        })

    if yearly_metrics:
        yearly_df = pd.DataFrame(yearly_metrics)

        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        # 1. ROC-AUC over time
        ax1 = axes[0, 0]
        for model_name in ['XGBoost', 'LightGBM']:
            model_data = yearly_df[yearly_df['Model'] == model_name]
            if not model_data.empty:
                ax1.plot(model_data['Year'], model_data['ROC_AUC'],
                        marker='o', markersize=10, linewidth=3,
                        color=model_colors[model_name], label=model_name, alpha=0.8)

        ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
        ax1.set_ylabel('ROC-AUC', fontsize=12, fontweight='bold')
        ax1.set_title('ROC-AUC Performance Over Time', fontsize=14, fontweight='bold')
        ax1.legend(fontsize=11)
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0.9, 1.0)

        # 2. PR-AUC over time
        ax2 = axes[0, 1]
        for model_name in ['XGBoost', 'LightGBM']:
            model_data = yearly_df[yearly_df['Model'] == model_name]
            if not model_data.empty:
                ax2.plot(model_data['Year'], model_data['PR_AUC'],
                        marker='o', markersize=10, linewidth=3,
                        color=model_colors[model_name], label=model_name, alpha=0.8)

        ax2.set_xlabel('Year', fontsize=12, fontweight='bold')
        ax2.set_ylabel('PR-AUC', fontsize=12, fontweight='bold')
        ax2.set_title('PR-AUC Performance Over Time', fontsize=14, fontweight='bold')
        ax2.legend(fontsize=11)
        ax2.grid(True, alpha=0.3)

        # 3. Bankruptcy rate over time
        ax3 = axes[1, 0]
        year_stats = yearly_df.groupby('Year').agg({
            'N_Samples': 'first',
            'N_Bankruptcies': 'first',
            'Bankruptcy_Rate': 'first'
        }).reset_index()

        bars = ax3.bar(year_stats['Year'], year_stats['Bankruptcy_Rate'] * 100,
                       color='#E74C3C', alpha=0.7, edgecolor='black', linewidth=2)

        # Add value labels
        for bar, rate in zip(bars, year_stats['Bankruptcy_Rate']):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.002,
                    f'{rate:.2%}', ha='center', va='bottom', fontweight='bold')

        ax3.set_xlabel('Year', fontsize=12, fontweight='bold')
        ax3.set_ylabel('Bankruptcy Rate (%)', fontsize=12, fontweight='bold')
        ax3.set_title('Bankruptcy Rate by Year', fontsize=14, fontweight='bold')
        ax3.grid(True, alpha=0.3, axis='y')

        # 4. Performance heatmap
        ax4 = axes[1, 1]
        pivot_table = yearly_df.pivot_table(
            index='Model',
            columns='Year',
            values='ROC_AUC',
            aggfunc='mean'
        )

        sns.heatmap(pivot_table, annot=True, fmt='.3f', cmap='RdYlBu_r',
                   vmin=0.90, vmax=1.0, ax=ax4,
                   cbar_kws={'label': 'ROC-AUC'},
                   linewidths=1, linecolor='white')

        ax4.set_title('ROC-AUC Heatmap by Year and Model', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Year', fontsize=12, fontweight='bold')
        ax4.set_ylabel('Model', fontsize=12, fontweight='bold')
        ax4.set_yticklabels(ax4.get_yticklabels(), rotation=0)

        plt.suptitle('Temporal Performance Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        save_fig(fig, 'temporal_performance_analysis')
        plt.show()

        # Print summary statistics
        print("\nTemporal Performance Summary:")
        print(yearly_df.groupby(['Year', 'Model'])['ROC_AUC'].mean().round(4))
    else:
        print("⚠️ Could not calculate temporal metrics")
else:
    print("⚠️ Temporal analysis not possible - missing required data")
    print("   Note: This analysis requires the original data with year information")

In [ ]:
#@title 📊 Business-Friendly Performance Analysis

print("📊 BUSINESS-FRIENDLY PERFORMANCE ANALYSIS")
print("=" * 80)

# Business parameters
COMPANY_INVESTIGATION_COST = 1000  # € per company
BANKRUPTCY_MISS_COST = 100000     # € cost of missing a bankruptcy

# Get test set statistics
total_companies = len(y_test)
total_bankruptcies = int(y_test.sum())
bankruptcy_rate = y_test.mean()

print(f"Test Set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

# Calculate business metrics for different investigation percentages
investigation_percentages = np.arange(0.5, 30.5, 0.5)  # 0.5% to 30% in 0.5% steps
recall_targets = [0.80, 0.90, 0.95]  # Main recall targets

# Store results for each model
business_results = {}

for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        continue

    y_pred = predictions[model_name]['test']

    # Sort predictions by score (highest first)
    sorted_indices = np.argsort(y_pred)[::-1]
    y_true_sorted = y_test[sorted_indices]

    # Calculate cumulative bankruptcies caught
    cumsum_bankruptcies = np.cumsum(y_true_sorted)

    # Calculate metrics for each percentage
    results = []
    for pct in investigation_percentages:
        n_investigate = int(total_companies * pct / 100)
        if n_investigate == 0 or n_investigate > len(cumsum_bankruptcies):
            continue

        caught = cumsum_bankruptcies[n_investigate-1]
        recall = caught / total_bankruptcies
        precision = caught / n_investigate

        # Calculate costs and ROI
        investigation_cost = n_investigate * COMPANY_INVESTIGATION_COST
        prevented_losses = caught * BANKRUPTCY_MISS_COST
        net_benefit = prevented_losses - investigation_cost
        roi = ((prevented_losses / investigation_cost) - 1) * 100 if investigation_cost > 0 else 0

        results.append({
            'percentage': pct,
            'n_investigate': n_investigate,
            'caught': caught,
            'recall': recall,
            'precision': precision,
            'roi': roi,
            'net_benefit': net_benefit
        })

    business_results[model_name] = pd.DataFrame(results)

# Create visualizations
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.25)

# 1. ROI vs Investigation Effort
ax1 = fig.add_subplot(gs[0, 0])
for model_name, df in business_results.items():
    ax1.plot(df['percentage'], df['roi'],
             label=model_name, color=model_colors[model_name],
             linewidth=3, marker='o', markersize=6, markevery=5)

ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax1.set_xlabel('Companies Investigated (%)', fontsize=12)
ax1.set_ylabel('Return on Investment (%)', fontsize=12)
ax1.set_title('ROI vs Investigation Effort', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 30)

# 2. Recall vs Investigation Effort
ax2 = fig.add_subplot(gs[0, 1])
for model_name, df in business_results.items():
    ax2.plot(df['percentage'], df['recall'] * 100,
             label=model_name, color=model_colors[model_name],
             linewidth=3, marker='o', markersize=6, markevery=5)

# Add horizontal lines for target recalls
for target in recall_targets:
    ax2.axhline(y=target*100, color='gray', linestyle=':', alpha=0.7)
    ax2.text(29, target*100, f'{int(target*100)}%', va='center', fontsize=10)

ax2.set_xlabel('Companies Investigated (%)', fontsize=12)
ax2.set_ylabel('Bankruptcies Caught (%)', fontsize=12)
ax2.set_title('Recall vs Investigation Effort', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 30)
ax2.set_ylim(0, 100)

# 3. Net Benefit Analysis
ax3 = fig.add_subplot(gs[1, 0])
for model_name, df in business_results.items():
    ax3.plot(df['percentage'], df['net_benefit'] / 1e6,
             label=model_name, color=model_colors[model_name],
             linewidth=3, marker='o', markersize=6, markevery=5)

ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax3.set_xlabel('Companies Investigated (%)', fontsize=12)
ax3.set_ylabel('Net Benefit (€ Millions)', fontsize=12)
ax3.set_title('Net Benefit vs Investigation Effort', fontsize=14, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 30)

# 4. Key Metrics Table for Target Recalls
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('tight')
ax4.axis('off')

# Create table data
table_data = []
for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in business_results:
        continue

    df = business_results[model_name]
    for target_recall in recall_targets:
        # Find the minimum investigation % needed for target recall
        mask = df['recall'] >= target_recall
        if mask.any():
            row = df[mask].iloc[0]
            table_data.append([
                model_name,
                f"{int(target_recall*100)}%",
                f"{row['percentage']:.1f}%",
                f"{row['n_investigate']:,}",
                f"{row['roi']:.0f}%",
                f"€{row['net_benefit']/1e6:.1f}M"
            ])

# Create table
if table_data:
    table = ax4.table(cellText=table_data,
                      colLabels=['Model', 'Target Recall', 'Investigate %',
                                'Companies', 'ROI', 'Net Benefit'],
                      cellLoc='center',
                      loc='center',
                      colWidths=[0.15, 0.15, 0.15, 0.15, 0.15, 0.15])

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 2)

    # Style the table
    for i in range(len(table_data) + 1):
        for j in range(6):
            cell = table[(i, j)]
            if i == 0:
                cell.set_facecolor('#34495E')
                cell.set_text_props(weight='bold', color='white')
            else:
                cell.set_facecolor('#ECF0F1' if i % 2 == 0 else 'white')

ax4.set_title('Performance at Target Recall Levels', fontsize=14, fontweight='bold', pad=20)

plt.suptitle('Business Performance Analysis', fontsize=16, fontweight='bold')
save_fig(fig, 'business_performance_analysis')
plt.show()

In [ ]:
#@title 📝 Final Results Summary (with .txt Export)

import os
import pandas as pd
from IPython.display import clear_output
from datetime import datetime
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score

# This list will hold all the lines of our report
report_lines = []

# ==============================================================================
# ENSURE WE HAVE THE REQUIRED DATA
# ==============================================================================

# Force creation of model results (ignore existing 'results' which contains business metrics)
model_results = []

# Check what we have available
if 'predictions' not in globals() or not predictions:
    raise ValueError("No predictions found. Please run the model training first.")

# Create results from whatever models we have
for model_name in predictions.keys():
    if model_name in predictions:
        # Calculate metrics
        test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])
        test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

        # Use default values if optimization config not available
        if 'OPTIMIZATION_CONFIG' in globals():
            test_partial_pr_auc = partial_pr_auc_score(y_test, predictions[model_name]['test'],
                                                       OPTIMIZATION_CONFIG['partial_range'])
            test_prec_at_recall = precision_at_recall_score(y_test, predictions[model_name]['test'],
                                                           OPTIMIZATION_CONFIG['target_recall'])
            _, test_metric_value = eval_metric_func(y_test, predictions[model_name]['test'])
        else:
            test_partial_pr_auc = 0.0
            test_prec_at_recall = 0.0
            test_metric_value = test_roc_auc  # Use ROC-AUC as default

        # Get sampling info from best_params if available
        if 'best_params' in globals() and model_name in best_params:
            sampling_ratio = best_params[model_name].get('sampling_ratio', 0.3)
        else:
            sampling_ratio = 0.3  # Default

        # Calculate scale_pos_weight
        if 'sampled_datasets' in globals() and sampling_ratio in sampled_datasets:
            scale_pos_weight = sampled_datasets[sampling_ratio]['scale_pos_weight']
        else:
            scale_pos_weight = 233.24  # Default from your output

        model_results.append({
            'Model': model_name,
            'Test_Metric': test_metric_value,
            'Test_ROCAUC': test_roc_auc,
            'Test_PRAUC': test_pr_auc,
            'Test_PartialPRAUC': test_partial_pr_auc,
            'Test_PrecAtRecall': test_prec_at_recall,
            'Sampling_Ratio': sampling_ratio,
            'Scale_Pos_Weight': scale_pos_weight
        })

# Now check if model_results is empty
if not model_results:
    raise ValueError("No model results available. Please ensure at least one model was trained successfully.")

# Create results DataFrame from model_results (not the business analysis results)
results_df = pd.DataFrame(model_results)

# Debug print to see what columns we have
print("Available columns in results_df:", results_df.columns.tolist())
print("Number of rows:", len(results_df))

# Sort by the best available metric
if 'Test_Metric' in results_df.columns:
    results_df = results_df.sort_values('Test_Metric', ascending=False)
elif 'Test_ROCAUC' in results_df.columns:
    results_df = results_df.sort_values('Test_ROCAUC', ascending=False)
else:
    print("Warning: No metric columns found for sorting")

# Get best model name
if not results_df.empty:
    best_model_name = results_df.iloc[0]['Model']
else:
    # If we only have XGBoost
    best_model_name = 'XGBoost' if 'XGBoost' in predictions else list(predictions.keys())[0]

# Helper function to create metric table
def create_metric_table(model_name):
    """Create a formatted metric table for a specific model"""

    # Calculate all metrics for all sets
    train_roc_auc = roc_auc_score(y_train, predictions[model_name]['train'])
    valid_roc_auc = roc_auc_score(y_valid, predictions[model_name]['valid'])
    test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])

    train_pr_auc = average_precision_score(y_train, predictions[model_name]['train'])
    valid_pr_auc = average_precision_score(y_valid, predictions[model_name]['valid'])
    test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

    train_partial_pr = partial_pr_auc_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['partial_range'])
    valid_partial_pr = partial_pr_auc_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['partial_range'])
    test_partial_pr = partial_pr_auc_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['partial_range'])

    train_prec_recall = precision_at_recall_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['target_recall'])
    valid_prec_recall = precision_at_recall_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['target_recall'])
    test_prec_recall = precision_at_recall_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['target_recall'])

    # Create metric data
    metrics_data = []

    # Determine which metric was used for optimization
    opt_metric_type = OPTIMIZATION_CONFIG['metric']

    # Add optimization metric first (highlighted)
    if opt_metric_type == 'partial_pr_auc':
        range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
        metrics_data.append({
            'Metric': f'★ Partial PR-AUC {range_str}',
            'Train': f"{train_partial_pr:.4f}",
            'Validation': f"{valid_partial_pr:.4f}",
            'Test': f"{test_partial_pr:.4f}"
        })

    # Add other metrics
    metrics_data.append({
        'Metric': 'ROC-AUC',
        'Train': f"{train_roc_auc:.4f}",
        'Validation': f"{valid_roc_auc:.4f}",
        'Test': f"{test_roc_auc:.4f}"
    })

    metrics_data.append({
        'Metric': 'PR-AUC',
        'Train': f"{train_pr_auc:.4f}",
        'Validation': f"{valid_pr_auc:.4f}",
        'Test': f"{test_pr_auc:.4f}"
    })

    if opt_metric_type != 'partial_pr_auc':
        range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
        metrics_data.append({
            'Metric': f'Partial PR-AUC {range_str}',
            'Train': f"{train_partial_pr:.4f}",
            'Validation': f"{valid_partial_pr:.4f}",
            'Test': f"{test_partial_pr:.4f}"
        })

    metrics_data.append({
        'Metric': f"Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
        'Train': f"{train_prec_recall:.4f}",
        'Validation': f"{valid_prec_recall:.4f}",
        'Test': f"{test_prec_recall:.4f}"
    })

    return pd.DataFrame(metrics_data)

# ==============================================================================
# BUILD THE REPORT STRING
# ==============================================================================

report_lines.append("BANKRUPTCY PREDICTION MODEL TRAINING COMPLETED")
report_lines.append("=" * 80)

# Handle time calculation
if 'tracker' in globals() and hasattr(tracker, 'start_time'):
    total_time = tracker.format_time(time.time() - tracker.start_time)
elif 'training_metadata' in globals() and 'training_time' in training_metadata:
    total_time = f"{training_metadata['training_time']/3600:.1f} hours"
else:
    total_time = "N/A"

report_lines.append(f"Total time: {total_time}")
report_lines.append(f"Optimization Metric: {get_metric_display_name(OPTIMIZATION_CONFIG)}")
report_lines.append("=" * 80)
report_lines.append("\nMODEL PERFORMANCE SUMMARY:")
report_lines.append("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name not in predictions:
        continue

    report_lines.append(f"\n{model_name.upper()} METRICS:")
    report_lines.append("-" * 60)

    # Create and format metric table
    metric_table = create_metric_table(model_name)
    # Use to_string() for clean text formatting
    report_lines.append(metric_table.to_string(index=False))

    # Get sampling info
    model_result = results_df[results_df['Model'] == model_name].iloc[0]
    sampling_ratio = model_result['Sampling_Ratio']
    scale_pos_weight = model_result['Scale_Pos_Weight']

    # Get dataset info if available
    if 'sampled_datasets' in globals() and sampling_ratio in sampled_datasets:
        dataset_info = sampled_datasets[sampling_ratio]
        report_lines.append(f"\nSampling Configuration:")
        report_lines.append(f"  • Sampling ratio: {sampling_ratio} (kept {sampling_ratio*100:.0f}% of healthy companies)")
        report_lines.append(f"  • Scale pos weight: {scale_pos_weight:.2f}")
        report_lines.append(f"  • Training bankruptcy rate: {dataset_info['bankruptcy_rate']:.3%}")
        report_lines.append(f"  • Training bankruptcies: {dataset_info['n_bankrupt']:,}")
    else:
        report_lines.append(f"\nSampling Configuration:")
        report_lines.append(f"  • Sampling ratio: {sampling_ratio}")
        report_lines.append(f"  • Scale pos weight: {scale_pos_weight:.2f}")

report_lines.append(f"\nBest Model: {best_model_name} (based on {get_metric_display_name(OPTIMIZATION_CONFIG)})")

# --- OPTIMIZED HYPERPARAMETERS ---
report_lines.append("\n\nOPTIMIZED HYPERPARAMETERS:")
report_lines.append("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
    if model_name in best_params:
        report_lines.append(f"\n{model_name}:")
        params = best_params[model_name]
        for param in sorted(params.keys()):
            if param == 'sampling_ratio': continue
            value = params[param]
            if isinstance(value, float):
                report_lines.append(f"  • {param}: {value:.4f}")
            else:
                report_lines.append(f"  • {param}: {value}")

# --- BUSINESS ANALYSIS ---
report_lines.append("\n\nBUSINESS ANALYSIS FOR BEST MODEL:")
report_lines.append("=" * 80)
best_pred = predictions[best_model_name]['test']
total_companies = len(y_test)
total_bankruptcies = int(y_test.sum())
bankruptcy_rate = y_test.mean()

report_lines.append(f"Test Set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

precision, recall, thresholds = precision_recall_curve(y_test, best_pred)

for target_recall in [0.90, 0.95, 0.99]:
    valid_indices = np.where(recall >= target_recall)[0]
    if len(valid_indices) == 0:
        report_lines.append(f"\n⚠️ Cannot achieve {target_recall:.0%} recall")
        continue

    idx = valid_indices[-1]
    actual_recall = recall[idx]
    actual_precision = precision[idx]
    threshold = thresholds[idx] if idx < len(thresholds) else thresholds[-1] - 0.0001
    companies_flagged = np.sum(best_pred >= threshold)
    investigation_rate = companies_flagged / total_companies
    bankruptcies_to_catch = int(total_bankruptcies * actual_recall)
    random_companies = int(total_companies * target_recall)
    improvement = random_companies / companies_flagged if companies_flagged > 0 else 1.0

    report_lines.append(f"\nAt {target_recall:.0%} recall target:")
    report_lines.append(f"  • Actual recall: {actual_recall:.3f} ({bankruptcies_to_catch}/{total_bankruptcies} bankruptcies)")
    report_lines.append(f"  • Precision: {actual_precision:.3%}")
    report_lines.append(f"  • Must investigate: {companies_flagged:,} companies ({investigation_rate:.1%})")
    report_lines.append(f"  • Threshold: {threshold:.4f}")
    report_lines.append(f"  • Improvement over random: {improvement:.2f}x")

# --- MODEL DIAGNOSTIC ---
report_lines.append("\n\nMODEL DIAGNOSTIC:")
report_lines.append("-" * 80)
bankrupt_scores = best_pred[y_test == 1]
healthy_scores = best_pred[y_test == 0]
report_lines.append(f"Bankruptcy scores: mean={bankrupt_scores.mean():.4f}, median={np.median(bankrupt_scores):.4f}")
report_lines.append(f"Healthy scores: mean={healthy_scores.mean():.4f}, median={np.median(healthy_scores):.4f}")
overlap_threshold = np.percentile(healthy_scores, 95)
bankrupt_below_threshold = np.sum(bankrupt_scores < overlap_threshold) / len(bankrupt_scores)
report_lines.append(f"Score overlap: {bankrupt_below_threshold:.1%} of bankruptcies score below 95th percentile of healthy")

# --- DISTRIBUTION SHIFT ANALYSIS ---
report_lines.append("\n\nDISTRIBUTION SHIFT ANALYSIS:")
report_lines.append("-" * 80)
report_lines.append(f"Original training bankruptcy rate: {y_train.mean():.3%}")

# Handle different data availability scenarios
if 'sampled_datasets' in globals():
    best_model_result = results_df.iloc[0]
    best_sampling_ratio = best_model_result['Sampling_Ratio']
    best_dataset = sampled_datasets[best_sampling_ratio]
    report_lines.append(f"Sampled training bankruptcy rate: {best_dataset['bankruptcy_rate']:.3%}")
    dist_shift = best_dataset['bankruptcy_rate'] / y_test.mean()
else:
    report_lines.append(f"Sampled training bankruptcy rate: N/A")
    dist_shift = "N/A"

report_lines.append(f"Validation bankruptcy rate: {y_valid.mean():.3%}")
report_lines.append(f"Test bankruptcy rate: {y_test.mean():.3%}")
if isinstance(dist_shift, float):
    report_lines.append(f"Distribution shift (sampled train vs test): {dist_shift:.1f}x")
else:
    report_lines.append(f"Distribution shift (sampled train vs test): {dist_shift}")

# --- OPTIMIZATION SUMMARY ---
if 'tracker' in globals() and hasattr(tracker, 'trial_stats'):
    report_lines.append("\n\nOPTIMIZATION SUMMARY:")
    report_lines.append("-" * 80)
    for model_name, stats in tracker.trial_stats.items():
        completion_rate = stats['completed'] / stats['total_attempted'] * 100 if stats['total_attempted'] > 0 else 0
        report_lines.append(f"{model_name}: {stats['completed']} completed ({completion_rate:.0f}% success rate)")

# ==============================================================================
# FINAL REPORT OUTPUT AND SAVE
# ==============================================================================
# Join all lines into a single string
final_report_text = "\n".join(report_lines)

# Clear previous output and print the final report
clear_output(wait=True)
print(final_report_text)

# Save the report to the SAME FOLDER as other outputs
try:
    # Check if save_path exists, if not create it
    if 'save_path' not in globals():
        # Create save path based on current configuration
        current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
        metric_name = OPTIMIZATION_CONFIG['metric']
        n_trials = MODEL_TRIALS['xgboost'] if 'MODEL_TRIALS' in globals() else 60
        folder_name = f"bankruptcy_model_{metric_name}_{n_trials}trials_{current_time}"
        save_path = os.path.join(path_MODEL, folder_name)
        os.makedirs(save_path, exist_ok=True)

    # Save to the save_path directory
    report_path = os.path.join(save_path, "model_summary_report.txt")
    with open(report_path, "w", encoding='utf-8') as f:
        f.write(final_report_text)

    print("\n" + "="*80)
    print(f"✅ Summary report saved to: {report_path}")
    print("="*80)

except Exception as e:
    print(f"\n⚠️ Could not save summary report. Error: {e}")

# Store or update metadata
if 'training_metadata' not in globals():
    training_metadata = {}

training_metadata.update({
    'optimization_config': OPTIMIZATION_CONFIG,
    'best_model': best_model_name,
    'results_df': results_df,
    'test_bankruptcies': total_bankruptcies,
    'test_companies': total_companies,
    'bankruptcy_rate': bankruptcy_rate,
    'predictions': predictions,
    'models': models
})

print(f"\nAll data stored in 'training_metadata' dictionary for analysis.")

## saving graph data

In [ ]:
#@title 💾 Save All Data for Visualizations

import pickle
import json
import os
from datetime import datetime
import joblib
import pandas as pd
import numpy as np

print("💾 SAVING ALL DATA FOR VISUALIZATIONS")
print("=" * 80)

# Create folder name with important info
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
metric_name = OPTIMIZATION_CONFIG['metric']
n_trials = MODEL_TRIALS['xgboost']
best_model = best_model_name if 'best_model_name' in globals() else 'Unknown'
best_sampling = results_df.iloc[0]['Sampling_Ratio'] if 'results_df' in globals() else 'Unknown'

folder_name = f"bankruptcy_complete_{metric_name}_{n_trials}trials_samp{best_sampling}_{current_time}"
save_path = os.path.join(path_MODEL, folder_name)
os.makedirs(save_path, exist_ok=True)

print(f"Saving to: {save_path}")

# Create a comprehensive data dictionary
save_data = {}

#================================================================================
# CONFIGURATION AND METADATA
#================================================================================
print("\n1. Saving configuration...")

save_data['config'] = {
    'optimization_config': OPTIMIZATION_CONFIG,
    'sampling_ratios': SAMPLING_RATIOS,
    'model_trials': MODEL_TRIALS,
    'optuna_timeout': OPTUNA_TIMEOUT,
    'param_ranges': PARAM_RANGES,
    'random_state': RANDOM_STATE,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'use_gpu': USE_GPU,
    'use_scale_pos_weight': USE_SCALE_POS_WEIGHT,
    'use_pruning': USE_PRUNING,
    'pruning_n_startup_trials': PRUNING_N_STARTUP_TRIALS,
    'pruning_n_warmup_steps': PRUNING_N_WARMUP_STEPS,
    'pruning_interval_steps': PRUNING_INTERVAL_STEPS,
    'bankruptcy_recall_targets': BANKRUPTCY_RECALL_TARGETS,
    'train_years': train_years,
    'valid_years': valid_years,
    'test_years': test_years,
    'creation_time': current_time,
    'best_model_name': best_model_name if 'best_model_name' in globals() else None
}

#================================================================================
# FEATURE NAMES AND DATA
#================================================================================
print("2. Saving feature names and data shapes...")

save_data['features'] = {
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'data_shapes': {
        'X_train': X_train.shape,
        'X_valid': X_valid.shape,
        'X_test': X_test.shape
    }
}

# Save actual data arrays
save_data['data'] = {
    'X_train': X_train,
    'y_train': y_train,
    'X_valid': X_valid,
    'y_valid': y_valid,
    'X_test': X_test,
    'y_test': y_test
}

#================================================================================
# MODELS
#================================================================================
print("3. Saving models...")

# Save models using joblib (handles both XGBoost and LightGBM)
models_dir = os.path.join(save_path, 'models')
os.makedirs(models_dir, exist_ok=True)

for model_name, model in models.items():
    joblib.dump(model, os.path.join(models_dir, f'{model_name.lower()}_model.pkl'))

save_data['model_names'] = list(models.keys())

#================================================================================
# PREDICTIONS
#================================================================================
print("4. Saving predictions...")

save_data['predictions'] = predictions

#================================================================================
# OPTUNA STUDIES
#================================================================================
print("5. Saving Optuna studies...")

if 'study_xgb' in globals() and 'study_lgb' in globals():
    joblib.dump(study_xgb, os.path.join(save_path, 'study_xgboost.pkl'))
    joblib.dump(study_lgb, os.path.join(save_path, 'study_lightgbm.pkl'))
    save_data['has_optuna_studies'] = True
else:
    save_data['has_optuna_studies'] = False

#================================================================================
# RESULTS AND METRICS
#================================================================================
print("6. Saving results and metrics...")

save_data['results'] = {
    'results_df': results_df,
    'best_params': best_params,
    'training_metadata': training_metadata if 'training_metadata' in globals() else None
}

#================================================================================
# SAMPLING INFORMATION
#================================================================================
print("7. Saving sampling information...")

save_data['sampling'] = {
    'sampled_datasets': {
        ratio: {k: v for k, v in dataset.items() if k not in ['X_train', 'y_train']}
        for ratio, dataset in sampled_datasets.items()
    } if 'sampled_datasets' in globals() else None
}

#================================================================================
# BUSINESS METRICS
#================================================================================
print("8. Saving business metrics...")

# Calculate total companies and bankruptcies for business analysis
save_data['business'] = {
    'total_companies': len(y_test),
    'total_bankruptcies': int(y_test.sum()),
    'bankruptcy_rate': y_test.mean()
}

#================================================================================
# EVALUATION FUNCTIONS
#================================================================================
print("9. Saving evaluation function parameters...")

save_data['eval_functions'] = {
    'partial_pr_range': OPTIMIZATION_CONFIG.get('partial_range', [0.9, 0.95]),
    'target_recall': OPTIMIZATION_CONFIG.get('target_recall', 0.90)
}

#================================================================================
# SAVE MAIN PICKLE FILE
#================================================================================
print("\n10. Creating main save file...")

with open(os.path.join(save_path, 'complete_data.pkl'), 'wb') as f:
    pickle.dump(save_data, f)

#================================================================================
# SAVE METADATA JSON
#================================================================================
metadata = {
    'creation_time': current_time,
    'optimization_metric': metric_name,
    'n_trials': n_trials,
    'best_model': best_model,
    'best_sampling_ratio': float(best_sampling) if isinstance(best_sampling, (int, float)) else str(best_sampling),
    'features_count': len(feature_names),
    'train_size': len(y_train),
    'valid_size': len(y_valid),
    'test_size': len(y_test),
    'test_bankruptcy_rate': float(y_test.mean()),
    'path': save_path
}

with open(os.path.join(save_path, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

#================================================================================
# CREATE README
#================================================================================
readme_content = f"""# Bankruptcy Prediction Model - Complete Data

## Overview
- **Created**: {current_time}
- **Optimization Metric**: {metric_name}
- **Trials**: {n_trials}
- **Best Model**: {best_model}
- **Best Sampling Ratio**: {best_sampling}

## Contents
- `complete_data.pkl`: All data needed for visualizations
- `metadata.json`: Quick reference metadata
- `models/`: Trained model files
- `study_*.pkl`: Optuna optimization studies

## Data Shapes
- Training: {X_train.shape}
- Validation: {X_valid.shape}
- Test: {X_test.shape}
- Features: {len(feature_names)}

## To Load
Use the corresponding load cell to restore all data and run visualizations.
"""

with open(os.path.join(save_path, 'README.md'), 'w') as f:
    f.write(readme_content)

print(f"\n✅ All data saved successfully!")
print(f"📁 Location: {save_path}")
print(f"📊 Total size: ~{sum(os.path.getsize(os.path.join(root, file)) for root, _, files in os.walk(save_path) for file in files) / 1024 / 1024:.1f} MB")

## loading graph data

In [ ]:
#@title 📂 Load All Data for Visualizations (CHANGE FOLDER)

import pickle
import json
import os
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, auc, precision_recall_curve

print("📂 LOADING ALL DATA FOR VISUALIZATIONS")
print("=" * 80)

# List available saved folders


print("Available saved models:")
folders = "bankruptcy_complete_roc_auc_700trials_samp1.0_20250707_125843"

# Select folder
folder_idx = 0  # Change this to select different save
load_path = os.path.join(path_MODEL, folders)
print(f"\nLoading from: {load_path}")

#================================================================================
# LOAD MAIN DATA
#================================================================================
print("\n1. Loading main data file...")

with open(os.path.join(load_path, 'complete_data.pkl'), 'rb') as f:
    save_data = pickle.load(f)

#================================================================================
# RESTORE CONFIGURATION
#================================================================================
print("2. Restoring configuration...")

config = save_data['config']
OPTIMIZATION_CONFIG = config['optimization_config']
SAMPLING_RATIOS = config['sampling_ratios']
MODEL_TRIALS = config['model_trials']
OPTUNA_TIMEOUT = config['optuna_timeout']
PARAM_RANGES = config['param_ranges']
RANDOM_STATE = config['random_state']
EARLY_STOPPING_ROUNDS = config['early_stopping_rounds']
USE_GPU = config['use_gpu']
USE_SCALE_POS_WEIGHT = config['use_scale_pos_weight']
USE_PRUNING = config['use_pruning']
PRUNING_N_STARTUP_TRIALS = config['pruning_n_startup_trials']
PRUNING_N_WARMUP_STEPS = config['pruning_n_warmup_steps']
PRUNING_INTERVAL_STEPS = config['pruning_interval_steps']
BANKRUPTCY_RECALL_TARGETS = config['bankruptcy_recall_targets']
train_years = config['train_years']
valid_years = config['valid_years']
test_years = config['test_years']
best_model_name = config['best_model_name']

#================================================================================
# RESTORE FEATURES AND DATA
#================================================================================
print("3. Restoring features and data...")

feature_names = save_data['features']['feature_names']
X_train = save_data['data']['X_train']
y_train = save_data['data']['y_train']
X_valid = save_data['data']['X_valid']
y_valid = save_data['data']['y_valid']
X_test = save_data['data']['X_test']
y_test = save_data['data']['y_test']

print(f"   Data shapes restored:")
print(f"   - Training: {X_train.shape}")
print(f"   - Validation: {X_valid.shape}")
print(f"   - Test: {X_test.shape}")

#================================================================================
# RESTORE MODELS
#================================================================================
print("4. Loading models...")

models = {}
models_dir = os.path.join(load_path, 'models')
for model_name in save_data['model_names']:
    model_path = os.path.join(models_dir, f'{model_name.lower()}_model.pkl')
    models[model_name] = joblib.load(model_path)
    print(f"   ✓ Loaded {model_name}")

#================================================================================
# RESTORE PREDICTIONS
#================================================================================
print("5. Restoring predictions...")

predictions = save_data['predictions']

#================================================================================
# RESTORE OPTUNA STUDIES
#================================================================================
print("6. Loading Optuna studies...")

if save_data.get('has_optuna_studies', False):
    study_xgb = joblib.load(os.path.join(load_path, 'study_xgboost.pkl'))
    study_lgb = joblib.load(os.path.join(load_path, 'study_lightgbm.pkl'))
    print(f"   ✓ Loaded XGBoost study with {len(study_xgb.trials)} trials")
    print(f"   ✓ Loaded LightGBM study with {len(study_lgb.trials)} trials")
else:
    print("   ⚠️ No Optuna studies in this save")

#================================================================================
# RESTORE RESULTS
#================================================================================
print("7. Restoring results...")

results_df = save_data['results']['results_df']
best_params = save_data['results']['best_params']
training_metadata = save_data['results']['training_metadata']

#================================================================================
# RESTORE SAMPLING INFO
#================================================================================
print("8. Restoring sampling information...")

if save_data['sampling']['sampled_datasets']:
    sampled_datasets = save_data['sampling']['sampled_datasets']
    print(f"   ✓ Sampling ratios available: {list(sampled_datasets.keys())}")
else:
    print("   ⚠️ No sampling information available")

#================================================================================
# RESTORE BUSINESS METRICS
#================================================================================
print("9. Restoring business metrics...")

total_companies = save_data['business']['total_companies']
total_bankruptcies = save_data['business']['total_bankruptcies']
bankruptcy_rate = save_data['business']['bankruptcy_rate']

print(f"   Test set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

#================================================================================
# RECREATE EVALUATION FUNCTIONS
#================================================================================
print("10. Recreating evaluation functions...")

def partial_pr_auc_score(y_true, y_pred_proba, recall_range):
    """Calculate partial PR-AUC for a specific recall range"""
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    precision = precision[::-1]
    recall = recall[::-1]
    idx_start = np.searchsorted(recall, recall_range[0])
    idx_end = np.searchsorted(recall, recall_range[1])
    if idx_end <= idx_start:
        return 0.0
    partial_auc = auc(recall[idx_start:idx_end], precision[idx_start:idx_end])
    normalized_auc = partial_auc / (recall_range[1] - recall_range[0])
    return normalized_auc

def precision_at_recall_score(y_true, y_pred_proba, target_recall):
    """Calculate precision at a specific recall level"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    idx = np.argmax(recall >= target_recall)
    if idx == 0 and recall[0] < target_recall:
        return 0.0
    return precision[idx]

def get_metric_display_name(config):
    """Get a display name for the current metric configuration"""
    if config['metric'] == 'partial_pr_auc':
        return f"Partial PR-AUC [{config['partial_range'][0]}-{config['partial_range'][1]}]"
    elif config['metric'] == 'precision_at_recall':
        return f"Precision@{config['target_recall']}Recall"
    else:
        return config['metric'].upper().replace('_', '-')

def create_eval_metric_function(metric_config):
    """Create the appropriate evaluation metric function based on config"""
    metric_type = metric_config['metric']

    if metric_type == 'roc_auc':
        def eval_metric(y_true, y_pred):
            return 'roc_auc', roc_auc_score(y_true, y_pred)
        native_metric = 'auc'
    elif metric_type == 'pr_auc':
        def eval_metric(y_true, y_pred):
            return 'pr_auc', average_precision_score(y_true, y_pred)
        native_metric = 'aucpr'
    elif metric_type == 'partial_pr_auc':
        recall_range = metric_config['partial_range']
        def eval_metric(y_true, y_pred):
            return f'partial_pr_auc_{recall_range[0]}-{recall_range[1]}', partial_pr_auc_score(y_true, y_pred, recall_range)
        native_metric = 'aucpr'
    elif metric_type == 'precision_at_recall':
        target_recall = metric_config['target_recall']
        def eval_metric(y_true, y_pred):
            return f'precision@{target_recall}', precision_at_recall_score(y_true, y_pred, target_recall)
        native_metric = 'aucpr'
    else:
        raise ValueError(f"Unknown metric type: {metric_type}")

    return eval_metric, native_metric

eval_metric_func, native_metric = create_eval_metric_function(OPTIMIZATION_CONFIG)

#================================================================================
# SUMMARY
#================================================================================
print("\n" + "="*80)
print("LOADING COMPLETE!")
print("="*80)
print(f"Loaded data from: {folders[folder_idx]}")
print(f"Best model: {best_model_name}")
print(f"All data required for visualizations is now available.")
print("\nYou can now run all visualization cells!")

# model with auc

In [ ]:
#@title XGBoost & LightGBM Model Training (Optimized with Sampling)

#================================================================================
# CONFIGURATION PARAMETERS (MODIFY HERE)
#================================================================================
# Define temporal splits
train_years = [2016, 2017, 2018]
valid_years = [2019]
test_years = [2020, 2021]

# Sampling Configuration
SAMPLING_RATIOS = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]  # Fraction of healthy companies to keep

# Optimization Configuration
OPTIMIZATION_CONFIG = {
    'metric': 'pr_auc',
        # Options: 'roc_auc', 'pr_auc', 'partial_pr_auc', 'precision_at_recall'
    'partial_range': [0.90, 0.95],  # For partial_pr_auc (recall range)
    'target_recall': 0.90,          # For precision_at_recall
}

# Other Optimization Settings
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 10

# GPU Configuration
USE_GPU = True
GPU_DEVICE_ID = 0

# Class Weight Configuration
USE_SCALE_POS_WEIGHT = True  # Will be calculated for each sampling ratio

# Optuna Configuration
MODEL_TRIALS = {
    'xgboost':  700,    # Number of trials for XGBoost
    'lightgbm': 10     # Number of trials for LightGBM
}
OPTUNA_TIMEOUT = {
    'xgboost':  12000,  # 40 minutes per model
    'lightgbm': 12000   # 40 minutes per model
}

# Pruning Configuration
USE_PRUNING = True
PRUNING_N_STARTUP_TRIALS = 5
PRUNING_N_WARMUP_STEPS = 5
PRUNING_INTERVAL_STEPS = 1

# Hyperparameter Search Ranges
PARAM_RANGES = {
    'xgboost': {
        'max_depth':        (3    , 25  ),
        'learning_rate':    (0.01 , 0.4 ),
        'n_estimators':     (200  , 1500),
        'subsample':        (0.3  , 1   ),
        'colsample_bytree': (0.3  , 1   ),
        'reg_alpha':        (0.01 , 15.0),
        'reg_lambda':       (0.01 , 15.0),
        'min_child_weight': (1    , 100  )
    },
    'lightgbm': {
        'num_leaves':       (2    , 600 ),
        'learning_rate':    (0.005, 0.17),
        'n_estimators':     (30   , 800 ),
        'feature_fraction': (0.3  , 1   ),
        'bagging_fraction': (0.3  , 1   ),
        'bagging_freq':     (20   , 150 ),
        'reg_alpha':        (0.005, 20.0),
        'reg_lambda':       (0.005, 20.0),
        'min_child_weight': (1    , 150 )
    }
}

# Business Metric Configuration
BANKRUPTCY_RECALL_TARGETS = [0.80, 0.90, 0.95, 0.99]  # Recall targets for business metrics

# Trial Logging Configuration
LOG_FAILED_TRIALS = True  # Log details of failed trials

#================================================================================
# IMPORTS AND SETUP
#================================================================================

import subprocess
import gc
import psutil
import os
import time
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    accuracy_score,
    classification_report,
    auc
)
from IPython.display import clear_output
import warnings
import traceback

# Suppress warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# Store failed trials for debugging
failed_trials = {'XGBoost': [], 'LightGBM': []}

#================================================================================
# CUSTOM EVALUATION METRICS
#================================================================================

def partial_pr_auc_score(y_true, y_pred_proba, recall_range):
    """Calculate partial PR-AUC for a specific recall range"""
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

        # Check if we have enough points
        if len(precision) < 2:
            return 0.0

        # Reverse arrays to make recall increasing
        precision = precision[::-1]
        recall = recall[::-1]

        # Find indices for recall range
        idx_start = np.searchsorted(recall, recall_range[0])
        idx_end = np.searchsorted(recall, recall_range[1])

        # Ensure we have at least 2 points
        if idx_end - idx_start < 2:
            # If not enough points in the range, return 0
            return 0.0

        # Calculate partial AUC
        partial_auc = auc(recall[idx_start:idx_end], precision[idx_start:idx_end])

        # Normalize by the width of the recall range
        normalized_auc = partial_auc / (recall_range[1] - recall_range[0])

        return normalized_auc

    except Exception as e:
        # If any error occurs (e.g., all predictions are the same), return 0
        print(f"Warning in partial_pr_auc_score: {e}")
        return 0.0


def precision_at_recall_score(y_true, y_pred_proba, target_recall):
    """
    Calculate precision at the highest-precision threshold whose recall is ≥ target_recall.
    Returns 0.0 if the target recall cannot be reached.
    """
    try:
        # Check if we have any positive samples
        if np.sum(y_true) == 0:
            return 0.0

        # Check if predictions have any variation
        if len(np.unique(y_pred_proba)) < 2:
            # All predictions are the same
            if target_recall == 0:
                return 1.0 if np.all(y_pred_proba < 0.5) else 0.0
            else:
                # If we predict all as negative and need recall > 0, precision is 0
                # If we predict all as positive, precision equals positive rate
                if np.all(y_pred_proba >= 0.5):
                    return np.mean(y_true)
                else:
                    return 0.0

        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

        # Find all indices where the recall is at least the target
        valid = np.where(recall >= target_recall)[0]

        if len(valid) == 0:
            # The target recall level was never reached
            return 0.0

        # Of the valid points, take the one with the highest precision
        idx = valid[-1]
        return precision[idx]

    except Exception as e:
        print(f"Warning in precision_at_recall_score: {e}")
        return 0.0


def create_eval_metric_function(metric_config):
    """Create the appropriate evaluation metric function based on config"""
    metric_type = metric_config['metric']

    if metric_type == 'roc_auc':
        def eval_metric(y_true, y_pred):
            return 'roc_auc', roc_auc_score(y_true, y_pred)
        native_metric = 'auc'

    elif metric_type == 'pr_auc':
        def eval_metric(y_true, y_pred):
            return 'pr_auc', average_precision_score(y_true, y_pred)
        native_metric = 'auc'  # Changed from 'aucpr' to 'auc' for stability

    elif metric_type == 'partial_pr_auc':
        recall_range = metric_config['partial_range']
        def eval_metric(y_true, y_pred):
            return f'partial_pr_auc_{recall_range[0]}-{recall_range[1]}', partial_pr_auc_score(y_true, y_pred, recall_range)
        native_metric = 'auc'  # Changed from 'aucpr' to 'auc' for stability

    elif metric_type == 'precision_at_recall':
        target_recall = metric_config['target_recall']
        def eval_metric(y_true, y_pred):
            return f'precision@{target_recall}', precision_at_recall_score(y_true, y_pred, target_recall)
        native_metric = 'auc'  # Changed from 'aucpr' to 'auc' for stability

    else:
        raise ValueError(f"Unknown metric type: {metric_type}")

    return eval_metric, native_metric

# Create evaluation metric function
eval_metric_func, native_metric = create_eval_metric_function(OPTIMIZATION_CONFIG)

#================================================================================
# GPU DETECTION AND SETUP
#================================================================================

# GPU Detection
GPU_PARAMS = None
if USE_GPU:
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                               '--format=csv,noheader,nounits'],
                               capture_output=True, text=True)
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(', ')
            gpu_name = gpu_info[0]
            gpu_memory = f"{int(gpu_info[1])/1024:.1f} GB"

            print(f"GPU detected: {gpu_name} ({gpu_memory})")

            # Set GPU parameters based on GPU type
            if 'A100' in gpu_name:
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID,
                        'sampling_method': 'gradient_based',
                        'max_bin': 256
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False,
                        'max_bin': 255
                    }
                }
            elif 'T4' in gpu_name:
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID,
                        'sampling_method': 'uniform',
                        'max_bin': 64
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False,
                        'max_bin': 63
                    }
                }
            else:
                # Generic GPU settings
                GPU_PARAMS = {
                    'xgboost': {
                        'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'gpu_id': GPU_DEVICE_ID
                    },
                    'lightgbm': {
                        'device': 'gpu',
                        'gpu_platform_id': 0,
                        'gpu_device_id': GPU_DEVICE_ID,
                        'gpu_use_dp': False
                    }
                }
        else:
            print("GPU not available, using CPU")
            USE_GPU = False
    except:
        print("GPU detection failed, using CPU")
        USE_GPU = False

#================================================================================
# HELPER FUNCTIONS
#================================================================================

def get_memory_usage():
    """Get current memory usage in GB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024 / 1024

def clean_memory():
    """Force garbage collection"""
    gc.collect()

def get_metric_display_name(config):
    """Get a display name for the current metric configuration"""
    if config['metric'] == 'partial_pr_auc':
        return f"Partial PR-AUC [{config['partial_range'][0]}-{config['partial_range'][1]}]"
    elif config['metric'] == 'precision_at_recall':
        return f"Precision@{config['target_recall']}Recall"
    else:
        return config['metric'].upper().replace('_', '-')

#================================================================================
# ENHANCED PROGRESS TRACKING
#================================================================================

class ModelTracker:
    def __init__(self, total_models=2):
        self.total_models = total_models
        self.current_model = 0
        self.results_summary = []
        self.start_time = time.time()
        self.trial_stats = {}
        self.model_start_times = {}
        self.trial_details = {'XGBoost': [], 'LightGBM': []}
        self.metric_name = get_metric_display_name(OPTIMIZATION_CONFIG)
        self.best_sampling_ratios = {}

    def update(self, model_name, status, metrics=None, trial=None, total_trials=None, study=None):
        self.current_time = time.time() - self.start_time

        if model_name not in self.model_start_times:
            self.model_start_times[model_name] = time.time()

        clear_output(wait=True)

        # Header
        print("XGBOOST & LIGHTGBM BANKRUPTCY PREDICTION MODEL TRAINING")
        print("=" * 80)
        print(f"Total Elapsed Time: {self.format_time(self.current_time)}")
        print(f"Optimization Metric: {self.metric_name}")
        print(f"Sampling Ratios: {SAMPLING_RATIOS}")
        print(f"GPU: {'Enabled' if USE_GPU and GPU_PARAMS else 'Disabled'}")
        print(f"Memory Usage: {get_memory_usage():.2f} GB")
        print("=" * 80)

        # Current model status
        model_time = time.time() - self.model_start_times[model_name]
        print(f"\nCurrent Model: {model_name}")
        print(f"Status: {status}")
        print(f"Model Time: {self.format_time(model_time)}")

        # Optuna progress with detailed trial info
        if trial is not None and total_trials is not None:
            actual_trial_num = trial + 1
            progress = actual_trial_num / total_trials
            bar_length = 50
            filled_length = int(bar_length * progress)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)

            trial_info = f"{actual_trial_num}/{total_trials} trials"

            if study is not None:
                completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
                pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
                failed = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

                self.trial_stats[model_name] = {
                    'completed': completed,
                    'pruned': pruned,
                    'failed': failed,
                    'total_attempted': len(study.trials)
                }

                if completed > 0:
                    trial_info += f" ({completed} completed"
                    if pruned > 0:
                        trial_info += f", {pruned} pruned"
                    if failed > 0:
                        trial_info += f", {failed} failed"
                    trial_info += ")"

                    # Show best score with appropriate metric name
                    try:
                        if study.best_trial is not None:
                            trial_info += f" | Best: {study.best_value:.4f}"
                            # Show best sampling ratio if available
                            if 'sampling_ratio' in study.best_params:
                                trial_info += f" (sampling={study.best_params['sampling_ratio']})"
                    except:
                        pass

            print(f"Optimization: [{bar}] {trial_info}")

        # Results summary
        if self.results_summary:
            print("\nCOMPLETED MODELS:")
            print("-" * 80)
            for result in self.results_summary:
                print(f"{result['model']}: {result['metric_name']} = {result['metric_value']} (sampling={result.get('sampling_ratio', 'N/A')})")

    def format_time(self, seconds):
        if seconds < 60:
            return f"{seconds:.1f}s"
        elif seconds < 3600:
            return f"{seconds/60:.1f}m"
        else:
            hours = seconds // 3600
            minutes = (seconds % 3600) // 60
            return f"{hours:.0f}h {minutes:.0f}m"

    def complete_model(self, model_name, train_time, metric_value, sampling_ratio):
        self.current_model += 1
        self.results_summary.append({
            'model': model_name,
            'time': f"{train_time:.1f}s",
            'metric_name': self.metric_name,
            'metric_value': f"{metric_value:.4f}",
            'sampling_ratio': sampling_ratio
        })
        self.best_sampling_ratios[model_name] = sampling_ratio

#================================================================================
# DATA VERIFICATION
#================================================================================

tracker = ModelTracker(total_models=2)
tracker.update("Data Verification", "Checking if df_long exists...")

# Check if df_long exists first
if 'df_long' not in globals():
    raise ValueError("df_long not found! Please ensure the data is loaded before running this cell.")

print(f"\n✓ df_long found with shape: {df_long.shape}")
time.sleep(1)

#================================================================================
# DATA PREPARATION AND TEMPORAL SPLITTING
#================================================================================

tracker.update("Data Preparation", "Preparing temporal splits...")

import re

# Function to sanitize column names
def sanitize_column_names(df):
    """
    Replace special characters in column names that cause issues with XGBoost/LightGBM
    """
    # Characters that cause issues: []<>{},%
    df.columns = df.columns.str.replace('[', '_', regex=False)
    df.columns = df.columns.str.replace(']', '_', regex=False)
    df.columns = df.columns.str.replace('<', '_', regex=False)
    df.columns = df.columns.str.replace('>', '_', regex=False)
    df.columns = df.columns.str.replace('{', '_', regex=False)
    df.columns = df.columns.str.replace('}', '_', regex=False)
    df.columns = df.columns.str.replace(',', '_', regex=False)
    df.columns = df.columns.str.replace('%', 'pct', regex=False)
    df.columns = df.columns.str.replace('/', '_', regex=False)
    df.columns = df.columns.str.replace('\\', '_', regex=False)
    df.columns = df.columns.str.replace(':', '_', regex=False)
    df.columns = df.columns.str.replace('\n', '', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)

    # Remove any remaining special characters
    df.columns = [re.sub(r'[^\w]', '_', col) for col in df.columns]

    # Clean up multiple underscores
    df.columns = [re.sub(r'_+', '_', col) for col in df.columns]

    # Remove leading/trailing underscores
    df.columns = [col.strip('_') for col in df.columns]

    return df

# Check initial data
print(f"Total samples in df_long: {len(df_long):,}")
print(f"Years available: {sorted(df_long['Year'].unique())}")

# Temporal split
df_train = df_long[df_long['Year'].isin(train_years)].copy()
df_valid = df_long[df_long['Year'].isin(valid_years)].copy()
df_test = df_long[df_long['Year'].isin(test_years)].copy()

print(f"\nTemporal splits:")
print(f"Training set ({train_years}): {len(df_train):,} samples")
print(f"Validation set ({valid_years}): {len(df_valid):,} samples")
print(f"Test set ({test_years}): {len(df_test):,} samples")

# Sanitize column names
df_train = sanitize_column_names(df_train)
df_valid = sanitize_column_names(df_valid)
df_test = sanitize_column_names(df_test)
print("\n✓ Column names sanitized for XGBoost/LightGBM compatibility")

# Drop problematic columns
columns_to_drop = []

# Check for postal code columns
postal_columns = [col for col in df_train.columns if 'Postal_code' in col or 'postal_code' in col]
if postal_columns:
    columns_to_drop.extend(postal_columns)

# Drop CCIAA-derived province dummies
province_dummies = [col for col in df_train.columns if col.startswith('Province_')]
if province_dummies:
    print(f"Found {len(province_dummies)} province dummy columns from CCIAA - will be dropped")
    columns_to_drop.extend(province_dummies)

# Drop all identified columns
if columns_to_drop:
    df_train = df_train.drop(columns=columns_to_drop, errors='ignore')
    df_valid = df_valid.drop(columns=columns_to_drop, errors='ignore')
    df_test = df_test.drop(columns=columns_to_drop, errors='ignore')
    print(f"✓ Dropped {len(columns_to_drop)} columns total")

# Define columns to exclude from features
exclude_cols = ['Year', 'Tax_code_number', 'Target']

# Get feature columns
feature_cols = [col for col in df_train.columns if col not in exclude_cols]
feature_names = feature_cols

# Create feature matrices and target vectors for ORIGINAL data (before sampling)
X_train = df_train[feature_cols].values
y_train = df_train['Target'].values

X_valid = df_valid[feature_cols].values
y_valid = df_valid['Target'].values

X_test = df_test[feature_cols].values
y_test = df_test['Target'].values

print(f"\nFeatures: {len(feature_cols)}")
print(f"Original bankruptcy rates:")
print(f"  Training: {y_train.mean():.3%}")
print(f"  Validation: {y_valid.mean():.3%}")
print(f"  Test: {y_test.mean():.3%}")

# Convert to proper data types if needed
X_train = X_train.astype(np.float64)
X_valid = X_valid.astype(np.float64)
X_test = X_test.astype(np.float64)

# Calculate no information rate (baseline)
no_info_rate_train = 1 - y_train.mean()
no_info_rate_valid = 1 - y_valid.mean()
no_info_rate_test = 1 - y_test.mean()

print("\nData preparation completed!")
time.sleep(2)

#================================================================================
# CREATE SAMPLED DATASETS
#================================================================================

print("\n" + "="*80)
print("CREATING SAMPLED DATASETS")
print("="*80)

sampled_datasets = {}

# Get indices of bankrupt and healthy companies
bankrupt_indices = np.where(y_train == 1)[0]
healthy_indices = np.where(y_train == 0)[0]

n_bankrupt = len(bankrupt_indices)
n_healthy = len(healthy_indices)

print(f"\nOriginal training set:")
print(f"Bankrupt companies: {n_bankrupt:,}")
print(f"Healthy companies: {n_healthy:,}")

for ratio in SAMPLING_RATIOS:
    print(f"\nCreating dataset with sampling ratio {ratio}:")

    if ratio == 1.0:
        # No sampling, use original data
        X_train_sampled = X_train
        y_train_sampled = y_train
        sampled_healthy = n_healthy
    else:
        # Sample healthy companies
        n_healthy_to_keep = int(n_healthy * ratio)
        np.random.seed(RANDOM_STATE)
        sampled_healthy_indices = np.random.choice(healthy_indices, n_healthy_to_keep, replace=False)

        # Combine with all bankrupt companies
        all_indices = np.concatenate([bankrupt_indices, sampled_healthy_indices])
        np.random.shuffle(all_indices)

        X_train_sampled = X_train[all_indices]
        y_train_sampled = y_train[all_indices]
        sampled_healthy = n_healthy_to_keep

    # Calculate scale_pos_weight for this sample
    scale_pos_weight = (y_train_sampled == 0).sum() / (y_train_sampled == 1).sum()
    bankruptcy_rate = y_train_sampled.mean()

    # Store in dictionary
    sampled_datasets[ratio] = {
        'X_train': X_train_sampled,
        'y_train': y_train_sampled,
        'scale_pos_weight': scale_pos_weight,
        'n_samples': len(y_train_sampled),
        'n_bankrupt': n_bankrupt,
        'n_healthy': sampled_healthy,
        'bankruptcy_rate': bankruptcy_rate
    }

    print(f"  • Total samples: {len(y_train_sampled):,}")
    print(f"  • Healthy companies kept: {sampled_healthy:,} ({ratio*100:.0f}%)")
    print(f"  • Bankruptcy rate: {bankruptcy_rate:.3%}")
    print(f"  • Scale pos weight: {scale_pos_weight:.2f}")
    print(f"  • Distribution shift vs test: {bankruptcy_rate/y_test.mean():.1f}x")

time.sleep(3)

# Initialize storage
results = []
models = {}
predictions = {}
best_params = {}

#================================================================================
# MODEL 1: XGBOOST
#================================================================================

tracker.update("XGBoost", "Initializing...")

def objective_xgb(trial):
    try:
        # Add sampling ratio as a hyperparameter
        sampling_ratio = trial.suggest_categorical('sampling_ratio', SAMPLING_RATIOS)

        # Get the sampled dataset
        dataset = sampled_datasets[sampling_ratio]
        X_train_sampled = dataset['X_train']
        y_train_sampled = dataset['y_train']
        scale_pos_weight = dataset['scale_pos_weight']

        tracker.update("XGBoost", f"Optimizing hyperparameters (sampling={sampling_ratio})...",
                      trial=trial.number, total_trials=MODEL_TRIALS['xgboost'],
                      study=trial.study)

        params = {
            'max_depth': trial.suggest_int('max_depth', *PARAM_RANGES['xgboost']['max_depth']),
            'learning_rate': trial.suggest_float('learning_rate', *PARAM_RANGES['xgboost']['learning_rate'], log=True),
            'n_estimators': trial.suggest_int('n_estimators', *PARAM_RANGES['xgboost']['n_estimators']),
            'subsample': trial.suggest_float('subsample', *PARAM_RANGES['xgboost']['subsample']),
            'colsample_bytree': trial.suggest_float('colsample_bytree', *PARAM_RANGES['xgboost']['colsample_bytree']),
            'reg_alpha': trial.suggest_float('reg_alpha', *PARAM_RANGES['xgboost']['reg_alpha'], log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', *PARAM_RANGES['xgboost']['reg_lambda'], log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', *PARAM_RANGES['xgboost']['min_child_weight'])
        }

        # Model configuration
        model_params = {
            **params,
            'objective': 'binary:logistic',
            'seed': RANDOM_STATE,
            'eval_metric': native_metric,
            'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
            'verbosity': 0
        }

        # Add GPU parameters
        if USE_GPU and GPU_PARAMS:
            model_params.update(GPU_PARAMS['xgboost'])

        if USE_SCALE_POS_WEIGHT:
            model_params['scale_pos_weight'] = scale_pos_weight

        model = xgb.XGBClassifier(**model_params)

        # Train with sampled data
        model.fit(
            X_train_sampled, y_train_sampled,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        # Evaluate with our custom metric on validation set
        y_pred_proba = model.predict_proba(X_valid)[:, 1]
        _, metric_value = eval_metric_func(y_valid, y_pred_proba)

        return metric_value

    except Exception as e:
        # Log failed trial
        if LOG_FAILED_TRIALS:
            failed_trials['XGBoost'].append({
                'trial': trial.number,
                'params': trial.params,
                'error': str(e),
                'traceback': traceback.format_exc()
            })
        raise

# Create and run study
study_xgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=PRUNING_N_STARTUP_TRIALS,
        n_warmup_steps=PRUNING_N_WARMUP_STEPS,
        interval_steps=PRUNING_INTERVAL_STEPS
    ) if USE_PRUNING else None
)

try:
    study_xgb.optimize(
        objective_xgb,
        n_trials=MODEL_TRIALS['xgboost'],
        timeout=OPTUNA_TIMEOUT['xgboost'],
        n_jobs=1
    )
except Exception as e:
    print(f"\nOptimization stopped: {e}")
    if len(study_xgb.trials) == 0:
        raise ValueError("No trials completed!")

best_params['XGBoost'] = study_xgb.best_params
best_sampling_ratio_xgb = study_xgb.best_params['sampling_ratio']

# Train final model with best parameters
tracker.update("XGBoost", "Training final model...")

# Get the best sampled dataset
best_dataset_xgb = sampled_datasets[best_sampling_ratio_xgb]
X_train_best = best_dataset_xgb['X_train']
y_train_best = best_dataset_xgb['y_train']
scale_pos_weight_best = best_dataset_xgb['scale_pos_weight']

final_xgb_params = {k: v for k, v in study_xgb.best_params.items() if k != 'sampling_ratio'}
final_xgb_params.update({
    'objective': 'binary:logistic',
    'seed': RANDOM_STATE,
    'eval_metric': native_metric,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'verbosity': 0
})

if USE_GPU and GPU_PARAMS:
    final_xgb_params.update(GPU_PARAMS['xgboost'])

if USE_SCALE_POS_WEIGHT:
    final_xgb_params['scale_pos_weight'] = scale_pos_weight_best

xgb_model = xgb.XGBClassifier(**final_xgb_params)

start_time = time.time()
xgb_model.fit(
    X_train_best, y_train_best,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)
train_time = time.time() - start_time

# Predictions on all sets
y_train_pred_xgb = xgb_model.predict_proba(X_train)[:, 1]
y_valid_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
y_test_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Store
models['XGBoost'] = xgb_model
predictions['XGBoost'] = {
    'train': y_train_pred_xgb,
    'valid': y_valid_pred_xgb,
    'test': y_test_pred_xgb
}

# Calculate all metrics
test_roc_auc = roc_auc_score(y_test, y_test_pred_xgb)
test_pr_auc = average_precision_score(y_test, y_test_pred_xgb)
test_partial_pr_auc = partial_pr_auc_score(y_test, y_test_pred_xgb, OPTIMIZATION_CONFIG['partial_range'])
test_prec_at_recall = precision_at_recall_score(y_test, y_test_pred_xgb, OPTIMIZATION_CONFIG['target_recall'])

# Store the optimization metric value
_, test_metric_value = eval_metric_func(y_test, y_test_pred_xgb)

results.append({
    'Model': 'XGBoost',
    'Train_Time': train_time,
    'Test_Metric': test_metric_value,
    'Test_ROCAUC': test_roc_auc,
    'Test_PRAUC': test_pr_auc,
    'Test_PartialPRAUC': test_partial_pr_auc,
    'Test_PrecAtRecall': test_prec_at_recall,
    'Sampling_Ratio': best_sampling_ratio_xgb,
    'Scale_Pos_Weight': scale_pos_weight_best
})

tracker.complete_model("XGBoost", train_time, test_metric_value, best_sampling_ratio_xgb)
clean_memory()
time.sleep(1)

#================================================================================
# MODEL 2: LIGHTGBM
#================================================================================

tracker.update("LightGBM", "Initializing...")

def objective_lgb(trial):
    try:
        # Add sampling ratio as a hyperparameter
        sampling_ratio = trial.suggest_categorical('sampling_ratio', SAMPLING_RATIOS)

        # Get the sampled dataset
        dataset = sampled_datasets[sampling_ratio]
        X_train_sampled = dataset['X_train']
        y_train_sampled = dataset['y_train']
        scale_pos_weight = dataset['scale_pos_weight']

        tracker.update("LightGBM", f"Optimizing hyperparameters (sampling={sampling_ratio})...",
                      trial=trial.number, total_trials=MODEL_TRIALS['lightgbm'],
                      study=trial.study)

        params = {
            'num_leaves': trial.suggest_int('num_leaves', *PARAM_RANGES['lightgbm']['num_leaves']),
            'learning_rate': trial.suggest_float('learning_rate', *PARAM_RANGES['lightgbm']['learning_rate'], log=True),
            'n_estimators': trial.suggest_int('n_estimators', *PARAM_RANGES['lightgbm']['n_estimators']),
            'feature_fraction': trial.suggest_float('feature_fraction', *PARAM_RANGES['lightgbm']['feature_fraction']),
            'bagging_fraction': trial.suggest_float('bagging_fraction', *PARAM_RANGES['lightgbm']['bagging_fraction']),
            'bagging_freq': trial.suggest_int('bagging_freq', *PARAM_RANGES['lightgbm']['bagging_freq']),
            'reg_alpha': trial.suggest_float('reg_alpha', *PARAM_RANGES['lightgbm']['reg_alpha'], log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', *PARAM_RANGES['lightgbm']['reg_lambda'], log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', *PARAM_RANGES['lightgbm']['min_child_weight'])
        }

        # Model configuration
        model_params = {
            **params,
            'objective': 'binary',
            'boosting_type': 'gbdt',
            'random_state': RANDOM_STATE,
            'n_jobs': -1 if not USE_GPU else 1,
            'verbosity': -1
        }

        # Add GPU parameters
        if USE_GPU and GPU_PARAMS:
            model_params.update(GPU_PARAMS['lightgbm'])

        if USE_SCALE_POS_WEIGHT:
            model_params['scale_pos_weight'] = scale_pos_weight

        model = lgb.LGBMClassifier(**model_params)

        # Train with sampled data
        model.fit(
            X_train_sampled, y_train_sampled,
            eval_set=[(X_valid, y_valid)],
            eval_metric='average_precision' if native_metric == 'aucpr' else native_metric,
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
        )

        # Evaluate with our custom metric on validation set
        y_pred_proba = model.predict_proba(X_valid)[:, 1]
        _, metric_value = eval_metric_func(y_valid, y_pred_proba)

        return metric_value

    except Exception as e:
        # Log failed trial
        if LOG_FAILED_TRIALS:
            failed_trials['LightGBM'].append({
                'trial': trial.number,
                'params': trial.params,
                'error': str(e),
                'traceback': traceback.format_exc()
            })
        raise

# Create and run study
study_lgb = optuna.create_study(
   direction='maximize',
   pruner=optuna.pruners.MedianPruner(
       n_startup_trials=PRUNING_N_STARTUP_TRIALS,
       n_warmup_steps=PRUNING_N_WARMUP_STEPS,
       interval_steps=PRUNING_INTERVAL_STEPS
   ) if USE_PRUNING else None
)

try:
   study_lgb.optimize(
       objective_lgb,
       n_trials=MODEL_TRIALS['lightgbm'],
       timeout=OPTUNA_TIMEOUT['lightgbm'],
       n_jobs=1
   )
except Exception as e:
   print(f"\nOptimization stopped: {e}")
   if len(study_lgb.trials) == 0:
       raise ValueError("No trials completed!")

best_params['LightGBM'] = study_lgb.best_params
best_sampling_ratio_lgb = study_lgb.best_params['sampling_ratio']

# Train final model with best parameters
tracker.update("LightGBM", "Training final model...")

# Get the best sampled dataset
best_dataset_lgb = sampled_datasets[best_sampling_ratio_lgb]
X_train_best = best_dataset_lgb['X_train']
y_train_best = best_dataset_lgb['y_train']
scale_pos_weight_best = best_dataset_lgb['scale_pos_weight']

final_lgb_params = {k: v for k, v in study_lgb.best_params.items() if k != 'sampling_ratio'}
final_lgb_params.update({
   'objective': 'binary',
   'boosting_type': 'gbdt',
   'random_state': RANDOM_STATE,
   'n_jobs': -1 if not USE_GPU else 1,
   'verbosity': -1
})

if USE_GPU and GPU_PARAMS:
   final_lgb_params.update(GPU_PARAMS['lightgbm'])

if USE_SCALE_POS_WEIGHT:
   final_lgb_params['scale_pos_weight'] = scale_pos_weight_best

lgb_model = lgb.LGBMClassifier(**final_lgb_params)

start_time = time.time()
lgb_model.fit(
   X_train_best, y_train_best,
   eval_set=[(X_valid, y_valid)],
   eval_metric='average_precision' if native_metric == 'aucpr' else native_metric,
   callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
)
train_time = time.time() - start_time

# Predictions on all sets
y_train_pred_lgb = lgb_model.predict_proba(X_train)[:, 1]
y_valid_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
y_test_pred_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Store
models['LightGBM'] = lgb_model
predictions['LightGBM'] = {
   'train': y_train_pred_lgb,
   'valid': y_valid_pred_lgb,
   'test': y_test_pred_lgb
}

# Calculate all metrics
test_roc_auc = roc_auc_score(y_test, y_test_pred_lgb)
test_pr_auc = average_precision_score(y_test, y_test_pred_lgb)
test_partial_pr_auc = partial_pr_auc_score(y_test, y_test_pred_lgb, OPTIMIZATION_CONFIG['partial_range'])
test_prec_at_recall = precision_at_recall_score(y_test, y_test_pred_lgb, OPTIMIZATION_CONFIG['target_recall'])

# Store the optimization metric value
_, test_metric_value = eval_metric_func(y_test, y_test_pred_lgb)

results.append({
   'Model': 'LightGBM',
   'Train_Time': train_time,
   'Test_Metric': test_metric_value,
   'Test_ROCAUC': test_roc_auc,
   'Test_PRAUC': test_pr_auc,
   'Test_PartialPRAUC': test_partial_pr_auc,
   'Test_PrecAtRecall': test_prec_at_recall,
   'Sampling_Ratio': best_sampling_ratio_lgb,
   'Scale_Pos_Weight': scale_pos_weight_best
})

tracker.complete_model("LightGBM", train_time, test_metric_value, best_sampling_ratio_lgb)
clean_memory()

#================================================================================
# FINAL RESULTS SUMMARY
#================================================================================

clear_output(wait=True)

print("BANKRUPTCY PREDICTION MODEL TRAINING COMPLETED")
print("=" * 80)
print(f"Total time: {tracker.format_time(time.time() - tracker.start_time)}")
print(f"Optimization Metric: {get_metric_display_name(OPTIMIZATION_CONFIG)}")
print("=" * 80)

# Create results DataFrame
results_df = pd.DataFrame(results)

# Sort by test metric
results_df = results_df.sort_values('Test_Metric', ascending=False)

# Determine best model
best_model_name = results_df.iloc[0]['Model']

# Function to create metric table for a model
def create_metric_table(model_name):
   """Create a formatted metric table for a specific model"""

   # Calculate all metrics for all sets
   train_roc_auc = roc_auc_score(y_train, predictions[model_name]['train'])
   valid_roc_auc = roc_auc_score(y_valid, predictions[model_name]['valid'])
   test_roc_auc = roc_auc_score(y_test, predictions[model_name]['test'])

   train_pr_auc = average_precision_score(y_train, predictions[model_name]['train'])
   valid_pr_auc = average_precision_score(y_valid, predictions[model_name]['valid'])
   test_pr_auc = average_precision_score(y_test, predictions[model_name]['test'])

   train_partial_pr = partial_pr_auc_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['partial_range'])
   valid_partial_pr = partial_pr_auc_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['partial_range'])
   test_partial_pr = partial_pr_auc_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['partial_range'])

   train_prec_recall = precision_at_recall_score(y_train, predictions[model_name]['train'], OPTIMIZATION_CONFIG['target_recall'])
   valid_prec_recall = precision_at_recall_score(y_valid, predictions[model_name]['valid'], OPTIMIZATION_CONFIG['target_recall'])
   test_prec_recall = precision_at_recall_score(y_test, predictions[model_name]['test'], OPTIMIZATION_CONFIG['target_recall'])

   # Get optimization metric values
   train_opt_metric_name, train_opt_metric = eval_metric_func(y_train, predictions[model_name]['train'])
   valid_opt_metric_name, valid_opt_metric = eval_metric_func(y_valid, predictions[model_name]['valid'])
   test_opt_metric_name, test_opt_metric = eval_metric_func(y_test, predictions[model_name]['test'])

   # Create metric data
   metrics_data = []

   # Determine which metric was used for optimization
   opt_metric_type = OPTIMIZATION_CONFIG['metric']

   # Add optimization metric first (highlighted)
   if opt_metric_type == 'roc_auc':
       metrics_data.append({
           'Metric': '★ ROC-AUC',
           'Train': f"{train_roc_auc:.4f}",
           'Validation': f"{valid_roc_auc:.4f}",
           'Test': f"{test_roc_auc:.4f}"
       })
   elif opt_metric_type == 'pr_auc':
       metrics_data.append({
           'Metric': '★ PR-AUC',
           'Train': f"{train_pr_auc:.4f}",
           'Validation': f"{valid_pr_auc:.4f}",
           'Test': f"{test_pr_auc:.4f}"
       })
   elif opt_metric_type == 'partial_pr_auc':
       range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
       metrics_data.append({
           'Metric': f'★ Partial PR-AUC {range_str}',
           'Train': f"{train_partial_pr:.4f}",
           'Validation': f"{valid_partial_pr:.4f}",
           'Test': f"{test_partial_pr:.4f}"
       })
   elif opt_metric_type == 'precision_at_recall':
       metrics_data.append({
           'Metric': f"★ Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
           'Train': f"{train_prec_recall:.4f}",
           'Validation': f"{valid_prec_recall:.4f}",
           'Test': f"{test_prec_recall:.4f}"
       })

   # Add other metrics
   if opt_metric_type != 'roc_auc':
       metrics_data.append({
           'Metric': 'ROC-AUC',
           'Train': f"{train_roc_auc:.4f}",
           'Validation': f"{valid_roc_auc:.4f}",
           'Test': f"{test_roc_auc:.4f}"
       })

   if opt_metric_type != 'pr_auc':
       metrics_data.append({
           'Metric': 'PR-AUC',
           'Train': f"{train_pr_auc:.4f}",
           'Validation': f"{valid_pr_auc:.4f}",
           'Test': f"{test_pr_auc:.4f}"
       })

   if opt_metric_type != 'partial_pr_auc':
       range_str = f"[{OPTIMIZATION_CONFIG['partial_range'][0]}-{OPTIMIZATION_CONFIG['partial_range'][1]}]"
       metrics_data.append({
           'Metric': f'Partial PR-AUC {range_str}',
           'Train': f"{train_partial_pr:.4f}",
           'Validation': f"{valid_partial_pr:.4f}",
           'Test': f"{test_partial_pr:.4f}"
       })

   if opt_metric_type != 'precision_at_recall':
       metrics_data.append({
           'Metric': f"Precision@{OPTIMIZATION_CONFIG['target_recall']}Recall",
           'Train': f"{train_prec_recall:.4f}",
           'Validation': f"{valid_prec_recall:.4f}",
           'Test': f"{test_prec_recall:.4f}"
       })

   return pd.DataFrame(metrics_data)

# Display metric tables for each model
print("\nMODEL PERFORMANCE SUMMARY:")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
   if model_name not in predictions:
       continue

   print(f"\n{model_name.upper()} METRICS:")
   print("-" * 60)

   # Create and display metric table
   metric_table = create_metric_table(model_name)

   # Format table with aligned columns
   for _, row in metric_table.iterrows():
       metric_name = row['Metric']
       train_val = row['Train']
       valid_val = row['Validation']
       test_val = row['Test']
       print(f"{metric_name:<30} | Train: {train_val} | Valid: {valid_val} | Test: {test_val}")

   # Get sampling info for this model
   model_result = results_df[results_df['Model'] == model_name].iloc[0]
   sampling_ratio = model_result['Sampling_Ratio']
   scale_pos_weight = model_result['Scale_Pos_Weight']

   # Get dataset info
   dataset_info = sampled_datasets[sampling_ratio]

   print(f"\nSampling Configuration:")
   print(f"  • Sampling ratio: {sampling_ratio} (kept {sampling_ratio*100:.0f}% of healthy companies)")
   print(f"  • Scale pos weight: {scale_pos_weight:.2f}")
   print(f"  • Training bankruptcy rate: {dataset_info['bankruptcy_rate']:.3%}")
   print(f"  • Training bankruptcies: {dataset_info['n_bankrupt']:,}")

print(f"\nBest Model: {best_model_name} (based on {get_metric_display_name(OPTIMIZATION_CONFIG)})")

# Display optimized hyperparameters
print("\nOPTIMIZED HYPERPARAMETERS:")
print("=" * 80)

for model_name in ['XGBoost', 'LightGBM']:
   if model_name in best_params:
       print(f"\n{model_name}:")
       params = best_params[model_name]
       # Sort parameters by name for consistent display
       for param in sorted(params.keys()):
           if param == 'sampling_ratio':
               continue  # Already shown above
           value = params[param]
           if isinstance(value, float):
               print(f"  • {param}: {value:.4f}")
           else:
               print(f"  • {param}: {value}")

# Detailed business metrics for best model
print("\nBUSINESS ANALYSIS FOR BEST MODEL:")
print("=" * 80)
best_pred = predictions[best_model_name]['test']

# Calculate total bankruptcies and companies
total_companies = len(y_test)
total_bankruptcies = int(y_test.sum())
bankruptcy_rate = y_test.mean()

print(f"Test Set: {total_companies:,} companies, {total_bankruptcies} bankruptcies ({bankruptcy_rate:.3%})")

# Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, best_pred)

# Business metrics at different recall levels
for target_recall in [0.90, 0.95, 0.99]:
   # Find where recall drops below target (since recall is decreasing)
   valid_indices = np.where(recall >= target_recall)[0]

   if len(valid_indices) == 0:
       print(f"\n⚠️ Cannot achieve {target_recall:.0%} recall")
       continue

   # Take the last valid index (highest precision for this recall level)
   idx = valid_indices[-1]

   # Get actual values at this point
   actual_recall = recall[idx]
   actual_precision = precision[idx]

   # Handle threshold
   if idx < len(thresholds):
       threshold = thresholds[idx]
   else:
       threshold = thresholds[-1] - 0.0001

   # Calculate business metrics
   companies_flagged = np.sum(best_pred >= threshold)

   # If companies_flagged is too small, recalculate properly
   if companies_flagged < 10:
       sorted_indices = np.argsort(best_pred)[::-1]
       bankruptcies_needed = int(np.ceil(total_bankruptcies * target_recall))
       bankruptcies_found = 0
       companies_flagged = 0

       for i, idx_company in enumerate(sorted_indices):
           companies_flagged = i + 1
           if y_test.iloc[idx_company] == 1:
               bankruptcies_found += 1
           if bankruptcies_found >= bankruptcies_needed:
               break

       actual_precision = bankruptcies_found / companies_flagged
       threshold = best_pred[sorted_indices[companies_flagged-1]]

   investigation_rate = companies_flagged / total_companies
   bankruptcies_to_catch = int(total_bankruptcies * actual_recall)

   # Improvement over random
   random_companies = int(total_companies * target_recall)
   improvement = random_companies / companies_flagged if companies_flagged > 0 else 1.0

   print(f"\nAt {target_recall:.0%} recall target:")
   print(f"  • Actual recall: {actual_recall:.3f} ({bankruptcies_to_catch}/{total_bankruptcies} bankruptcies)")
   print(f"  • Precision: {actual_precision:.3%}")
   print(f"  • Must investigate: {companies_flagged:,} companies ({investigation_rate:.1%})")
   print(f"  • Threshold: {threshold:.4f}")
   print(f"  • Improvement over random: {improvement:.2f}x")

# Model diagnostic
print("\nMODEL DIAGNOSTIC:")
print("-" * 80)

# Score statistics
bankrupt_scores = best_pred[y_test == 1]
healthy_scores = best_pred[y_test == 0]

print(f"Bankruptcy scores: mean={bankrupt_scores.mean():.4f}, median={np.median(bankrupt_scores):.4f}")
print(f"Healthy scores: mean={healthy_scores.mean():.4f}, median={np.median(healthy_scores):.4f}")

# Check score overlap
overlap_threshold = np.percentile(healthy_scores, 95)
bankrupt_below_threshold = np.sum(bankrupt_scores < overlap_threshold) / len(bankrupt_scores)
print(f"Score overlap: {bankrupt_below_threshold:.1%} of bankruptcies score below 95th percentile of healthy")

# Distribution shift analysis
print("\nDISTRIBUTION SHIFT ANALYSIS:")
print("-" * 80)
print(f"Original training bankruptcy rate: {y_train.mean():.3%}")

# Get best model's sampling info
best_model_result = results_df.iloc[0]
best_sampling_ratio = best_model_result['Sampling_Ratio']
best_dataset = sampled_datasets[best_sampling_ratio]

print(f"Sampled training bankruptcy rate: {best_dataset['bankruptcy_rate']:.3%}")
print(f"Validation bankruptcy rate: {y_valid.mean():.3%}")
print(f"Test bankruptcy rate: {y_test.mean():.3%}")
print(f"Distribution shift (sampled train vs test): {best_dataset['bankruptcy_rate'] / y_test.mean():.1f}x")

# Trial statistics
print("\nOPTIMIZATION SUMMARY:")
print("-" * 80)
for model_name, stats in tracker.trial_stats.items():
   completion_rate = stats['completed'] / stats['total_attempted'] * 100 if stats['total_attempted'] > 0 else 0
   print(f"{model_name}: {stats['completed']} completed ({completion_rate:.0f}% success rate)")

print("\n" + "="*80)

# Store metadata
training_metadata = {
   'optimization_config': OPTIMIZATION_CONFIG,
   'sampling_ratios': SAMPLING_RATIOS,
   'best_model': best_model_name,
   'best_sampling_ratios': tracker.best_sampling_ratios,
   'sampled_datasets_info': {ratio: {k: v for k, v in info.items() if k not in ['X_train', 'y_train']}
                             for ratio, info in sampled_datasets.items()},
   'trial_stats': tracker.trial_stats,
   'best_params': best_params,
   'results_df': results_df,
   'training_time': time.time() - tracker.start_time,
   'test_bankruptcies': total_bankruptcies,
   'test_companies': total_companies,
   'bankruptcy_rate': bankruptcy_rate,
   'predictions': predictions,
   'models': models
}

print(f"All data stored in 'training_metadata' dictionary for analysis.")